# Gender / age extension, corpus: blocklist exclusion, dialect x gender, sentiment, gender-first race zoom

One pass over 12 C4.EN.NOBLOCKLIST shards (the reproduction's shard indices, seed 0): for every document, whether
C4's blocklist removed it (and whether only because of a blocklist entry containing the term's own words), which
gender / age / gender x ethnicity terms it mentions, its dialect (Blodgett et al.), and lexicon sentiment of the
paragraphs that mention each term. Then the analysis, then a RoBERTa sentiment classifier on sampled paragraphs.
The URL-year (time) breakdown is computed but **on hold**: C4 has no dated snapshots after April 2019.

**Settings:** Accelerator **None (CPU)** (the classifier then scores at most 150 paragraphs per term; with a GPU it
scores all). Internet **on**. *Save Version -> Save & Run All (Commit)*. About 1.5-2 h. Download `ga_corpus.zip`.

Resumable: a finished shard has a `stats_<shard>.json`; re-running skips it.

In [ ]:
%%bash
mkdir -p /kaggle/working/ga/scripts /kaggle/working/ga/data /kaggle/working/ga/results /kaggle/working/ga/third_party/unqover/templates \
         /kaggle/working/ga/third_party/unqover/word_lists/activities /kaggle/working/ga/third_party/unqover/word_lists/slots \
         /kaggle/working/ga/third_party/unqover/word_lists/fillers /kaggle/working/ga/third_party/unqover/word_lists/nouns/subjects
nproc; free -g | head -2; nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "no GPU"

## Scripts and data (embedded from the repo; these cells only write files)

In [ ]:
%%writefile /kaggle/working/ga/scripts/analyze_probe_stats.py
"""Small statistics helpers shared by the analyses (no heavy imports)."""
import numpy as np


def p_boot(d):
    """Two-sided bootstrap p-value for 'difference = 0'."""
    d = np.asarray(d, float)
    d = d[~np.isnan(d)]
    if not len(d):
        return np.nan
    return min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))


def holm(p):
    """Holm-adjusted p-values."""
    p = np.asarray(p, float)
    adj, running = np.empty_like(p), 0.0
    for rank, i in enumerate(np.argsort(p)):
        running = max(running, min(1.0, (len(p) - rank) * p[i]))
        adj[i] = running
    return adj

In [ ]:
%%writefile /kaggle/working/ga/scripts/scan_corpus.py
"""Extension, corpus: one pass over C4.EN.NOBLOCKLIST shards for gender and age terms.

Per document (fields text / url / timestamp; paragraphs = lines split on "\\n"):
1. Blocklist (Dodge §5): excluded if C4's own regex matches (TFDS v3.0.1, lowercased text, LDNOOBW @ 25e679f, not
   escaped), exactly as reproduction/excluded_data/scripts/scan_noblocklist.py. Kept documents are C4.EN (the
   one-shard pilot matched the released C4.EN for 99.4% of documents).
2. Terms (TERMS below): which gender / age / gender x age / gender x ethnicity terms the document mentions.
3. Mechanical exclusions: LDNOOBW itself contains gender / age / ethnicity words ("girl on", "big black",
   "sultry women", "wrapping men", "milf", ...). A document saying "a girl on the bus" is removed because of the
   phrase that also makes it a "girl" document. For each term we also count exclusions caused by at least one
   blocklist entry that does NOT contain the term's words ("excluded_other"). PMI on that count is the
   non-mechanical part.
4. Dialect (Dodge §5.3, optional): the Blodgett et al. argmax, as in the reproduction (dialect.py), so exclusion can
   be split by dialect x gender ("AAE documents that mention women").
5. URL year: C4's `timestamp` is the crawl date (every document is April 2019; checked on 50,000 documents), so it
   cannot date a page. A year in the URL path (/2014/05/, -2016-) is used as a publication-date proxy, the same idea
   as Dodge §3.2 (which used Internet Archive first-capture dates for a sample). Most URLs carry no year.
6. Sentiment (Dodge A.7) for paragraphs that mention a term, in kept and in excluded documents separately: positive
   and negative token counts under the three reproduction lexicons (SocialSent top 250+250, VADER |score| > 1,
   UnQover). The term words themselves are removed before counting (SocialSent scores "man" as positive: +2.1).

Matching (ours; Dodge's A.7 used case-sensitive fgrep " Jewish "): case-insensitive with word boundaries, because
gender words often start a sentence ("Women ..."). " Jewish " / " Arab " are also counted the paper's way
(case-sensitive, spaces) as a positive control: on these shards their SocialSent gap should match Arm B (7.1).

Counts are summed per (web domain, URL year bucket) so the analysis can bootstrap over domains.

Outputs per shard, in --out_dir:
    doc_counts_<shard>.npz    rows "host<TAB>year" x DOC_COLUMNS
    sent_counts_<shard>.npz   rows "host<TAB>year<TAB>kept" x SENT_COLUMNS
    para_sample_<shard>.jsonl.gz   a random sample of matching paragraphs per term (kept and excluded), for the
                              sentiment classifier (classify_paragraphs.py)
    stats_<shard>.json        written last; marks the shard done

Usage:
    python scripts/scan_corpus.py --n 12 --seed 0 --workers 3 --out_dir results/C1_noblocklist_12_shards
    python scripts/scan_corpus.py --files /path/c4-train.00000-of-01024.json.gz --out_dir ...
Seed 0 with n=12 gives the same shard indices as the reproduction's Arm B (B1) and F5 (S1) runs.
"""
import argparse, gzip, json, os, random, re, sys, time
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from urllib.parse import urlparse

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
REPO = os.path.dirname(os.path.dirname(ROOT))
DATA = os.path.join(ROOT, "data")
BADWORDS = os.path.join(DATA, "ldnoobw_en_25e679f.txt")
LEXICONS = {"SocialSent": "socialsent_250.tsv", "VADER": "vader.tsv", "UnQover": "unqover.tsv"}
MODEL_DIR = os.path.join(DATA, "twitteraae_model")
N_SHARDS = 1024

FEM = ["woman", "women", "girl", "girls", "female", "females", "lady", "ladies"]
MAS = ["man", "men", "boy", "boys", "male", "males", "guy", "guys"]
ETHNICITIES = {"Jewish": ["jewish"], "Arab": ["arab", "arabs"], "Black": ["black"], "White": ["white"],
               "Asian": ["asian"], "Hispanic": ["hispanic"], "Latina/o": ["latina", "latino", "latinas", "latinos"],
               "African American": ["african-american", "african american"], "Indian": ["indian"],
               "European": ["european"], "Chinese": ["chinese"], "Muslim": ["muslim"]}
# Terms are matched on words: the lowercased paragraph is split into words (letters, hyphen-joined) and punctuation
# marks, and each term is a set of word sequences of length 1-3 ("teenage girl" = ("teenage", "girl")). A phrase
# never spans punctuation or a paragraph break. Matching is case-insensitive (ours; A.7's fgrep was case-sensitive).
# name -> (word sequences, axis, gender, age, ethnicity)
def _seqs(*phrases):
    return [tuple(p.split()) for p in phrases]


def _cross(first, second):
    return [tuple(a.split()) + (b,) for a in first for b in second]


TERMS = {
    "women": (_seqs("woman", "women"), "gender", "female", "", ""),
    "men": (_seqs("man", "men"), "gender", "male", "", ""),
    "female": (_seqs("female", "females"), "gender", "female", "", ""),
    "male": (_seqs("male", "males"), "gender", "male", "", ""),
    "girl": (_seqs("girl", "girls"), "gender", "female", "child/teen", ""),
    "boy": (_seqs("boy", "boys"), "gender", "male", "child/teen", ""),
    "lady": (_seqs("lady", "ladies"), "gender", "female", "", ""),
    "gentleman": (_seqs("gentleman", "gentlemen"), "gender", "male", "", ""),
    "mother": (_seqs("mother", "mothers", "mom", "moms", "mum", "mums"), "gender", "female", "", ""),
    "father": (_seqs("father", "fathers", "dad", "dads"), "gender", "male", "", ""),
    "wife": (_seqs("wife", "wives"), "gender", "female", "", ""),
    "husband": (_seqs("husband", "husbands"), "gender", "male", "", ""),
    "daughter": (_seqs("daughter", "daughters"), "gender", "female", "", ""),
    "son": (_seqs("son", "sons"), "gender", "male", "", ""),
    "she/her": (_seqs("she", "her", "hers", "herself"), "pronoun", "female", "", ""),
    "he/him": (_seqs("he", "him", "his", "himself"), "pronoun", "male", "", ""),
    "child": (_seqs("child", "children", "kid", "kids"), "age", "", "child", ""),
    "teen": (_seqs("teen", "teens", "teenager", "teenagers", "teenage", "teenaged", "adolescent", "adolescents"),
             "age", "", "teen", ""),
    "young adult": (_seqs("young adult", "young adults", "young people"), "age", "", "young adult", ""),
    "adult": (_seqs("adult", "adults"), "age", "", "adult", ""),
    "middle-aged": (_seqs("middle-aged", "middle aged"), "age", "", "middle-aged", ""),
    "elderly": (_seqs("elderly", "old people", "older people", "older adults", "older persons", "senior citizen",
                      "senior citizens", "seniors", "pensioner", "pensioners", "retiree", "retirees"),
                "age", "", "elderly", ""),
    "person": (_seqs("person", "people"), "reference", "", "", ""),
    "teen girl": (_cross(["teen", "teenage", "teenaged"], ["girl", "girls"]), "gender_age", "female", "teen", ""),
    "teen boy": (_cross(["teen", "teenage", "teenaged"], ["boy", "boys"]), "gender_age", "male", "teen", ""),
    "little girl": (_cross(["little"], ["girl", "girls"]), "gender_age", "female", "child", ""),
    "little boy": (_cross(["little"], ["boy", "boys"]), "gender_age", "male", "child", ""),
    "young woman": (_cross(["young"], ["woman", "women"]), "gender_age", "female", "young adult", ""),
    "young man": (_cross(["young"], ["man", "men"]), "gender_age", "male", "young adult", ""),
    "middle-aged woman": (_cross(["middle-aged", "middle aged"], ["woman", "women"]), "gender_age", "female",
                          "middle-aged", ""),
    "middle-aged man": (_cross(["middle-aged", "middle aged"], ["man", "men"]), "gender_age", "male", "middle-aged", ""),
    "old woman": (_cross(["old", "older", "elderly"], ["woman", "women"]), "gender_age", "female", "elderly", ""),
    "old man": (_cross(["old", "older", "elderly"], ["man", "men"]), "gender_age", "male", "elderly", ""),
}
for _e, _forms in ETHNICITIES.items():
    TERMS[f"{_e} women"] = (_cross(_forms, FEM), "gender_eth", "female", "", _e)
    TERMS[f"{_e} men"] = (_cross(_forms, MAS), "gender_eth", "male", "", _e)
CASED = {"Jewish (paper)": " Jewish ", "Arab (paper)": " Arab "}   # A.7's fgrep matching, positive control
WORD_TERMS = list(TERMS)
TERMS.update({k: ([], "control", "", "", "") for k in CASED})
TERM_NAMES = list(TERMS)
LOOKUP = defaultdict(set)          # word sequence -> term names
for _t in WORD_TERMS:
    for _seq in TERMS[_t][0]:
        LOOKUP[_seq].add(_t)
MAXLEN = max(len(k) for k in LOOKUP)
FIRST = {k[0] for k in LOOKUP}     # words that can start a term; all other words are skipped quickly
# every word of every term, removed before sentiment counting (SocialSent scores "man" +2.1)
# (tokens() strips punctuation, so "middle-aged" arrives as "middleaged")
TERM_TOKENS = ({w for seq in LOOKUP for x in seq for w in x.split("-")} | {x.replace("-", "") for seq in LOOKUP for x in seq}
               | {"jew", "jews"})
WORD_RE = re.compile(r"[a-z]+(?:-[a-z]+)*|[^\sa-z]")
DIALECTS = ["AAE", "Hispanic", "Other", "WAE", "unassigned"]
YEAR_BUCKETS = ["none", "<=2009", "2010-12", "2013-15", "2016-17", "2018-19"]
YEAR_RE = re.compile(r"[/_.\-](199[5-9]|20[01][0-9])(?=[/_.\-]|$)")

DOC_COLUMNS = (["docs", "excluded"]
               + [f"{t}|{k}" for t in TERM_NAMES for k in ("docs", "excluded", "excluded_other")]
               + [f"dialect:{d}|{g}|{k}" for d in DIALECTS for g in ("all", "female", "male")
                  for k in ("docs", "excluded")])
SENT_COLUMNS = [f"{t}|{x}" for t in TERM_NAMES for x in
                ["paragraphs", "words"] + [f"{l}:{p}" for l in LEXICONS for p in ("pos", "neg")]]


def shard_name(i):
    return f"c4-train.{i:05d}-of-{N_SHARDS:05d}"


def year_bucket(url):
    m = YEAR_RE.search(urlparse(url).path)
    if not m:
        return "none"
    y = int(m.group(1))
    return ("<=2009" if y <= 2009 else "2010-12" if y <= 2012 else "2013-15" if y <= 2015
            else "2016-17" if y <= 2017 else "2018-19")


def badwords():
    words = [w.strip() for w in open(BADWORDS, encoding="utf-8") if w.strip()]
    return words, re.compile(r"(?:\W|^)({})(?:\W|$)".format("|".join(words)))   # TFDS v3.0.1: not escaped


def match_terms(text):
    """Term names mentioned in `text` (word sequences, case-insensitive) plus the paper-style cased matches."""
    w = WORD_RE.findall(text.lower())
    h = set()
    for i, x in enumerate(w):
        if x not in FIRST:
            continue
        for k in range(1, MAXLEN + 1):
            t = LOOKUP.get(tuple(w[i:i + k]))
            if t:
                h |= t
    h.update(t for t, s in CASED.items() if s in text)
    return h


def mechanical_overlap(words):
    """term -> blocklist entries that contain one of the term's words ('girl on' for "girl", 'big black' for
    "Black women"). An exclusion caused only by such entries is partly built into the blocklist."""
    out = {}
    for t, (seqs, *_) in TERMS.items():
        tw = {x for seq in seqs for x in seq} or {CASED[t].strip().lower()}
        out[t] = {e for e in words if tw & set(WORD_RE.findall(e))}
    return out


def load_lexicons():
    lex = {}
    for name, fn in LEXICONS.items():
        rows = [l.rstrip("\n").split("\t") for l in open(os.path.join(DATA, fn), encoding="utf-8")][1:]
        lex[name] = {w: p for w, p, *_ in rows}
    return lex


PUNCT = re.compile(r"[^\w\s]")


def tokens(text):
    return [t for t in (PUNCT.sub("", w).lower() for w in text.split()) if t and t not in TERM_TOKENS]


class Reservoir:
    def __init__(self, cap, rng):
        self.cap, self.rng, self.n, self.items = cap, rng, 0, []

    def add(self, x):
        self.n += 1
        if len(self.items) < self.cap:
            self.items.append(x)
        else:
            j = self.rng.randrange(self.n)
            if j < self.cap:
                self.items[j] = x


def scan_file(path, shard, args):
    words, bad = badwords()
    overlap = mechanical_overlap(words)
    lex = load_lexicons()
    dcol = {c: i for i, c in enumerate(DOC_COLUMNS)}
    scol = {c: i for i, c in enumerate(SENT_COLUMNS)}
    female_terms = {t for t, v in TERMS.items() if v[2] == "female" and v[1] != "pronoun"}
    male_terms = {t for t, v in TERMS.items() if v[2] == "male" and v[1] != "pronoun"}
    model = None
    if args.dialect:
        from dialect import DialectModel
        model = DialectModel(MODEL_DIR)
    rng = random.Random(f"{args.seed}-{shard}")
    samples = {(t, k): Reservoir(args.sample_kept if k else args.sample_excluded, rng)
               for t in TERM_NAMES for k in (1, 0)}
    doc_rows = defaultdict(lambda: np.zeros(len(DOC_COLUMNS), dtype=np.int64))
    sent_rows = defaultdict(lambda: np.zeros(len(SENT_COLUMNS), dtype=np.int64))
    n = n_ex = 0

    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            if args.limit_docs and n >= args.limit_docs:
                break
            d = json.loads(line)
            text, url = d["text"], d["url"]
            n += 1
            low = text.lower()
            matched = {m.strip() for m in bad.findall(low)}
            ex = bool(matched)
            n_ex += ex
            host, yb = urlparse(url).netloc, year_bucket(url)
            row = doc_rows[f"{host}\t{yb}"]
            row[0] += 1
            row[1] += ex
            par_hits = [(p, match_terms(p)) for p in text.split("\n")]
            doc_terms = set().union(*(h for _, h in par_hits))
            for t in doc_terms:
                row[dcol[f"{t}|docs"]] += 1
                if ex:
                    row[dcol[f"{t}|excluded"]] += 1
                    row[dcol[f"{t}|excluded_other"]] += bool(matched - overlap[t])
            if model is not None:
                post = model.fast(model.tokens(text))
                dl = DIALECTS[int(np.argmax(post))] if post is not None else "unassigned"
                for g, present in (("all", True), ("female", bool(doc_terms & female_terms)),
                                   ("male", bool(doc_terms & male_terms))):
                    if present:
                        row[dcol[f"dialect:{dl}|{g}|docs"]] += 1
                        row[dcol[f"dialect:{dl}|{g}|excluded"]] += ex
            if not doc_terms:
                continue
            kept = 0 if ex else 1
            srow = sent_rows[f"{host}\t{yb}\t{kept}"]
            for p, ph in par_hits:
                if not ph:
                    continue
                toks = tokens(p)
                counts = {l: [0, 0] for l in LEXICONS}
                for tok in toks:
                    for l, lx in lex.items():
                        pol = lx.get(tok)
                        if pol == "pos":
                            counts[l][0] += 1
                        elif pol == "neg":
                            counts[l][1] += 1
                nw = len(p.split())
                for t in ph:
                    srow[scol[f"{t}|paragraphs"]] += 1
                    srow[scol[f"{t}|words"]] += nw
                    for l, (cp, cn) in counts.items():
                        srow[scol[f"{t}|{l}:pos"]] += cp
                        srow[scol[f"{t}|{l}:neg"]] += cn
                    samples[(t, kept)].add({"term": t, "kept": kept, "url": url, "year": yb, "text": p[:1500]})

    def save_counts(rows, columns, name):
        keys = list(rows)
        np.savez_compressed(os.path.join(args.out_dir, f"{name}_{shard}.npz"), keys=np.array(keys, dtype=str),
                            counts=np.stack([rows[k] for k in keys]) if keys else np.zeros((0, len(columns)), np.int64),
                            columns=np.array(columns))

    save_counts(doc_rows, DOC_COLUMNS, "doc_counts")
    save_counts(sent_rows, SENT_COLUMNS, "sent_counts")
    tmp = os.path.join(args.out_dir, f"para_sample_{shard}.jsonl.gz.tmp")
    with gzip.open(tmp, "wt", encoding="utf-8") as g:
        for r in samples.values():
            for x in r.items:
                g.write(json.dumps({**x, "shard": shard}, ensure_ascii=False) + "\n")
    os.replace(tmp, tmp[:-4])
    stats = {"shard": shard, "documents": n, "excluded": n_ex, "excluded_rate": n_ex / max(1, n),
             "dialect": bool(model), "terms": TERM_NAMES,
             "mechanical_overlap": {t: sorted(s) for t, s in overlap.items() if s}}
    with open(os.path.join(args.out_dir, f"stats_{shard}.json"), "w") as f:   # written last = shard done
        json.dump(stats, f, indent=1)
    return stats


def process(shard, path, args):
    sys.path.insert(0, HERE)
    sys.path.insert(0, os.path.join(REPO, "reproduction", "excluded_data", "scripts"))
    t0 = time.time()
    downloaded = False
    if path is None:
        from huggingface_hub import hf_hub_download
        path = hf_hub_download("allenai/c4", f"en.noblocklist/{shard}.json.gz", repo_type="dataset",
                               cache_dir=args.cache_dir)
        downloaded = True
    s = scan_file(path, shard, args)
    if downloaded and not args.keep_shards:
        os.remove(os.path.realpath(path))
    return f"{shard}: {s['documents']} docs, {100 * s['excluded_rate']:.2f}% excluded, {time.time() - t0:.0f}s"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--shards", type=int, nargs="*")
    ap.add_argument("--range", type=int, nargs=2, metavar=("START", "END"))
    ap.add_argument("--n", type=int, default=12)
    ap.add_argument("--seed", type=int, default=0, help="shard choice (seed 0 = the reproduction's shard indices)")
    ap.add_argument("--files", nargs="*", help="local en.noblocklist shard files instead of downloading")
    ap.add_argument("--workers", type=int, default=1)
    ap.add_argument("--no_dialect", dest="dialect", action="store_false", help="skip the dialect model (faster)")
    ap.add_argument("--sample_kept", type=int, default=300, help="paragraphs sampled per term from kept docs")
    ap.add_argument("--sample_excluded", type=int, default=100, help="paragraphs sampled per term from excluded docs")
    ap.add_argument("--limit_docs", type=int, default=0, help="only the first N documents per shard (smoke tests)")
    ap.add_argument("--keep_shards", action="store_true")
    ap.add_argument("--cache_dir", default=None)
    args = ap.parse_args()
    os.makedirs(args.out_dir, exist_ok=True)
    sys.path.insert(0, os.path.join(REPO, "reproduction", "excluded_data", "scripts"))
    sys.path.insert(0, HERE)
    if args.dialect:
        from dialect import ensure_files
        ensure_files(MODEL_DIR)                      # download once, before workers start

    if args.files:
        jobs = [(re.search(r"c4-train\.\d{5}-of-\d{5}", os.path.basename(p)).group(0), p) for p in args.files]
    else:
        if args.shards:
            idx = args.shards
        elif args.range:
            idx = list(range(max(0, args.range[0]), min(N_SHARDS, args.range[1])))
        else:
            idx = sorted(random.Random(args.seed).sample(range(N_SHARDS), args.n))
        jobs = [(shard_name(i), None) for i in idx]
    meta_path = os.path.join(args.out_dir, "shards.json")
    meta = json.load(open(meta_path)) if os.path.exists(meta_path) else {"shards": []}
    if meta.get("terms", TERM_NAMES) != TERM_NAMES:
        raise SystemExit(f"{args.out_dir} was scanned with another term list; use another --out_dir")
    meta.update(shards=sorted(set(meta["shards"]) | {s for s, _ in jobs}), config="en.noblocklist", terms=TERM_NAMES)
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=1)

    todo = [(s, p) for s, p in jobs if not os.path.exists(os.path.join(args.out_dir, f"stats_{s}.json"))]
    print(f"{len(jobs)} shards requested, {len(jobs) - len(todo)} done already, {len(todo)} to scan", flush=True)
    t_all = time.time()
    if args.workers <= 1:
        for s, p in todo:
            print(process(s, p, args), flush=True)
    else:
        with ProcessPoolExecutor(args.workers) as ex:
            futs = [ex.submit(process, s, p, args) for s, p in todo]
            for i, fu in enumerate(as_completed(futs), 1):
                print(f"[{i}/{len(todo)}, {(time.time() - t_all) / 60:.0f} min] {fu.result()}", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/ga/scripts/dialect.py
"""Blodgett, Green and O'Connor (2016) demographic dialect model ("TwitterAAE"), ported to Python 3.

Dodge et al. 2021 §5.3 use it to give each document a posterior over four dialect "topics" and assign the argmax.
The model files come from github.com/slanglab/twitteraae at a pinned commit (MIT licence) and are downloaded once.

Two inference routines:
- exact(): a line-by-line port of the repo's predict.py (collapsed variational Bayes, CVB0, sequential token
  updates, alpha=1, 5 passes; None if no token, or fewer than 20% of tokens, are in the vocabulary).
- fast(): the same model and gates, but each pass updates all tokens at once (Jacobi instead of Gauss-Seidel)
  and identical words are updated together. It is about 100x faster, which a free-tier CPU needs for millions of
  documents. scan_noblocklist.py runs both on the first documents of every shard and records how often the
  argmax agrees, so the approximation is measured, not assumed.

Assumptions (the paper does not say): the whole document is scored; tokens are lowercased runs of letters and
digits with internal apostrophes (the repo used a tweet tokenizer, twokenize, on tweets); documents the model
cannot score are reported as "unassigned".
"""
import os, re, urllib.request

import numpy as np

SHA = "db4d6d126ca26b9bb9a70d7ce18bdd5833077523"
URL = f"https://raw.githubusercontent.com/slanglab/twitteraae/{SHA}/model/"
FILES = ["model_count_table.txt", "model_vocab.txt"]
# repo order: African-American, Hispanic, Asian, White. Dodge et al. call the third "other"; the repo README notes
# that only the AA and White topics were validated.
LABELS = ["AAE", "Hispanic", "Other", "WAE"]
TOKEN = re.compile(r"[a-z0-9]+(?:'[a-z]+)*")


def ensure_files(model_dir):
    os.makedirs(model_dir, exist_ok=True)
    for f in FILES:
        p = os.path.join(model_dir, f)
        if not os.path.exists(p):
            urllib.request.urlretrieve(URL + f, p + ".tmp")
            os.replace(p + ".tmp", p)


class DialectModel:
    def __init__(self, model_dir):
        ensure_files(model_dir)
        n_wk = np.loadtxt(os.path.join(model_dir, "model_count_table.txt"))
        self.wordprobs = (n_wk + 1) / n_wk.sum(0)             # as in predict.py
        with open(os.path.join(model_dir, "model_vocab.txt"), encoding="utf-8") as f:
            vocab = [line.split("\t")[-1].strip() for line in f]
        assert len(vocab) == n_wk.shape[0]
        self.w2num = {w: i for i, w in enumerate(vocab)}
        self.K = n_wk.shape[1]

    def tokens(self, text):
        return TOKEN.findall(text.lower())

    def _gate(self, toks, thresh1, thresh2):
        ids = [self.w2num[w] for w in toks if w in self.w2num]
        if len(ids) < thresh1 or len(ids) / max(1, len(toks)) < thresh2:
            return None
        return ids

    def exact(self, toks, alpha=1, numpasses=5, thresh1=1, thresh2=0.2):
        """Port of predict.predict / infer_cvb0 (sequential updates)."""
        ids = self._gate(toks, thresh1, thresh2)
        if ids is None:
            return None
        qs = self.wordprobs[ids].copy()
        qs /= qs.sum(1, keepdims=True)
        lik = qs.copy()
        q_k = qs.sum(0)
        for _ in range(1, numpasses):
            for i in range(len(ids)):
                q_k -= qs[i]
                qs[i] = lik[i] * (q_k + alpha)
                qs[i] /= qs[i].sum()
                q_k += qs[i]
        return q_k / q_k.sum()

    def fast(self, toks, alpha=1, numpasses=5, thresh1=1, thresh2=0.2):
        """Same model and gates; all tokens updated at once per pass, identical words grouped."""
        ids = self._gate(toks, thresh1, thresh2)
        if ids is None:
            return None
        u, c = np.unique(ids, return_counts=True)
        lik = self.wordprobs[u]
        lik = lik / lik.sum(1, keepdims=True)
        qs = lik.copy()
        q_k = c @ qs
        for _ in range(1, numpasses):
            qs = lik * (q_k - qs + alpha)
            qs /= qs.sum(1, keepdims=True)
            q_k = c @ qs
        return q_k / q_k.sum()

In [ ]:
%%writefile /kaggle/working/ga/scripts/analyze_corpus.py
"""Extension, corpus analysis: exclusion and sentiment by gender, age and gender x ethnicity. Reads scan_corpus.py.

1. Exclusion (Dodge §5.2). Per term: share of documents excluded by the blocklist, and document-level
   PMI = ln[P(term, excluded) / (P(term) P(excluded))] (the reproduction's definition, natural log). Also the same
   ratio counting only exclusions caused by a blocklist entry that does not contain the term's own words
   ("non-mechanical"; see scan_corpus.py). Women / men / female / male are Table 6 terms, so their PMIs can be checked
   against Fig. 5 (paper, read off the figure: women 1.73, man 1.65, female 1.82, male 1.90).
2. Dialect x gender (§5.3): removal rate of AAE / Hispanic / WAE documents, overall and among documents that
   mention a female or a male term.
3. Sentiment (A.7). Per term and lexicon: positivity = positive / (positive + negative) sentiment tokens in
   paragraphs mentioning the term, in kept documents (= C4.EN) and separately in excluded ones. Positive control: the
   paper-style " Jewish " / " Arab " gap, which Arm B measured as 7.1 (SocialSent) and 2.0 (VADER) on 12 C4.EN shards.
4. Gaps for matched pairs (women - men, girl - boy, Black women - Black men, ...), for both exclusion and sentiment.
5. Time: the same numbers per URL-year bucket (C4's timestamps are all April 2019; see scan_corpus.py).
6. Classifier (if para_scores.csv.gz exists): positivity = sum p_pos / (sum p_pos + sum p_neg) per term.

Uncertainty: 95% percentile intervals from a bootstrap over web domains (Poisson(1) weights per domain, B=200 by
default), because documents and paragraphs from one site are not independent. A gap's interval comes from the same
bootstrap replicates for both terms.

Usage:
    python scripts/analyze_corpus.py --scan results/C1_noblocklist_12_shards --out_dir results/C1_noblocklist_12_shards/analysis
"""
import argparse, glob, json, os, sys

import numpy as np
import pandas as pd
from scipy import sparse

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

PAIRS = [("women", "men"), ("female", "male"), ("girl", "boy"), ("lady", "gentleman"), ("mother", "father"),
         ("wife", "husband"), ("daughter", "son"), ("she/her", "he/him"), ("teen girl", "teen boy"),
         ("little girl", "little boy"), ("young woman", "young man"), ("middle-aged woman", "middle-aged man"),
         ("old woman", "old man")]
ETH = ["Jewish", "Arab", "Black", "White", "Asian", "Hispanic", "Latina/o", "African American", "Indian",
       "European", "Chinese", "Muslim"]
PAIRS += [(f"{e} women", f"{e} men") for e in ETH]
AGES = ["child", "teen", "young adult", "adult", "middle-aged", "elderly", "person"]
LEXICONS = ["SocialSent", "VADER", "UnQover"]
YEARS = ["<=2009", "2010-12", "2013-15", "2016-17", "2018-19", "none"]
PAPER_FIG5 = {"women": 1.73, "men": 1.65, "female": 1.82, "male": 1.90}
ARM_B = {"SocialSent": 7.1, "VADER": 2.0}
INK, INK2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#fcfcfb"
C_F, C_M = "#e87ba4", "#2a78d6"


def load(scan, name):
    files = sorted(glob.glob(os.path.join(scan, f"{name}_*.npz")))
    done = {os.path.basename(p)[6:-5] for p in glob.glob(os.path.join(scan, "stats_*.json"))}
    files = [p for p in files if p[len(os.path.join(scan, name)) + 1:-4] in done]
    if not files:
        raise SystemExit(f"no finished {name}_*.npz in {scan}")
    keys, mats, columns = [], [], None
    for p in files:
        z = np.load(p)
        columns = list(z["columns"])
        keys += list(z["keys"])
        mats.append(sparse.csr_matrix(z["counts"]))
    X = sparse.vstack(mats).tocsr()
    parts = pd.Series(keys).str.split("\t", expand=True)
    return X, columns, parts, len(files)


class Boot:
    """Totals and domain-bootstrap totals for row subsets of a sparse rows x columns count matrix."""

    def __init__(self, X, host, B, seed):
        self.X, self.B = X, B
        self.codes, uniq = pd.factorize(host)
        rng = np.random.default_rng(seed)
        self.W = rng.poisson(1.0, size=(len(uniq), B)).astype(np.float32)   # hosts x B

    def totals(self, mask=None):
        X = self.X if mask is None else self.X[np.flatnonzero(mask)]
        codes = self.codes if mask is None else self.codes[np.flatnonzero(mask)]
        point = np.asarray(X.sum(0)).ravel().astype(float)
        boot = np.asarray((X.T @ self.W[codes])).T            # B x columns
        return point, boot


def ci(arr):
    return np.nanpercentile(arr, [2.5, 97.5], axis=0)


def exclusion_tables(point, boot, col, terms):
    rows = []
    N, E = point[col["docs"]], point[col["excluded"]]
    bN, bE = boot[:, col["docs"]], boot[:, col["excluded"]]
    with np.errstate(divide="ignore", invalid="ignore"):
        for t in terms:
            d, e, o = (point[col[f"{t}|{k}"]] for k in ("docs", "excluded", "excluded_other"))
            bd, be, bo = (boot[:, col[f"{t}|{k}"]] for k in ("docs", "excluded", "excluded_other"))
            pmi, bpmi = np.log(e * N / (d * E)), np.log(be * bN / (bd * bE))
            pmo, bpmo = np.log(o * N / (d * E)), np.log(bo * bN / (bd * bE))
            rate, brate = 100 * e / d, 100 * be / bd
            rows.append({"term": t, "docs": int(d), "excluded": int(e), "excluded_%": rate,
                         "excluded_%_lo": ci(brate)[0], "excluded_%_hi": ci(brate)[1],
                         "pmi": pmi, "pmi_lo": ci(bpmi)[0], "pmi_hi": ci(bpmi)[1],
                         "pmi_nonmechanical": pmo, "pmi_nonmech_lo": ci(bpmo)[0], "pmi_nonmech_hi": ci(bpmo)[1],
                         "paper_fig5": PAPER_FIG5.get(t)})
    return pd.DataFrame(rows), (bN, bE)


def pair_gaps(point, boot, col, metric):
    """metric(point_or_boot, term) -> value; returns female - male per pair with CI."""
    rows = []
    for f, m in PAIRS:
        try:
            p = metric(point, f) - metric(point, m)
            b = metric(boot, f) - metric(boot, m)
        except KeyError:
            continue
        lo, hi = ci(b)
        rows.append({"female": f, "male": m, "gap": p, "ci_lo": lo, "ci_hi": hi})
    return pd.DataFrame(rows)


def eth_zoom(point, boot, col, value):
    """Gender first, then race: per ethnicity, value(<eth> women) - value(<eth> men), and how far that gap is from
    the gap pooled over ethnicities (all "<eth> women" counts summed vs all "<eth> men"). value(arr, [terms]) sums
    the terms' counts before computing. Bootstrap p-values, Holm-corrected over ethnicities."""
    from analyze_probe_stats import holm, p_boot
    have = [e for e in ETH if f"{e} women" in col and f"{e} men" in col]
    fem, mal = [f"{e} women" for e in have], [f"{e} men" for e in have]
    pooled, bpooled = value(point, fem) - value(point, mal), value(boot, fem) - value(boot, mal)
    rows = []
    for e in have:
        g, bg = value(point, [f"{e} women"]) - value(point, [f"{e} men"]),             value(boot, [f"{e} women"]) - value(boot, [f"{e} men"])
        d = bg - bpooled
        rows.append({"ethnicity": e, "women_minus_men": g, "ci_lo": ci(bg)[0], "ci_hi": ci(bg)[1],
                     "minus_pooled_gap": g - pooled, "diff_lo": ci(d)[0], "diff_hi": ci(d)[1], "p_boot": p_boot(d)})
    t = pd.DataFrame(rows)
    if len(t):
        t["p_holm"] = holm(t["p_boot"].fillna(1))
        t["anomaly"] = t["p_holm"] < 0.05
    return t, pooled


def positivity(arr, col, t, lex):
    pos, neg = arr[..., col[f"{t}|{lex}:pos"]], arr[..., col[f"{t}|{lex}:neg"]]
    with np.errstate(divide="ignore", invalid="ignore"):
        return 100 * pos / (pos + neg)


def sentiment_tables(point, boot, col, terms):
    rows = []
    for t in terms:
        for lex in LEXICONS:
            v, b = positivity(point, col, t, lex), positivity(boot, col, t, lex)
            lo, hi = ci(b)
            rows.append({"term": t, "lexicon": lex, "paragraphs": int(point[col[f"{t}|paragraphs"]]),
                         "sentiment_tokens": int(point[col[f"{t}|{lex}:pos"]] + point[col[f"{t}|{lex}:neg"]]),
                         "positivity": v, "ci_lo": lo, "ci_hi": hi})
    return pd.DataFrame(rows)


def style(ax):
    ax.set_facecolor(SURFACE)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.spines["left"].set_color(GRID)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(colors=INK2, length=0)
    ax.grid(axis="x", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def fig_pairs(df, value, lo, hi, xlabel, title, path, plt, ref=0):
    df = df.iloc[::-1].reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(8, 0.3 * len(df) + 1.5))
    style(ax)
    y = np.arange(len(df))
    v = df[value].to_numpy(float)
    ax.barh(y, v - ref, left=ref, color=[C_F if x >= ref else C_M for x in v], height=0.6, edgecolor=SURFACE)
    ax.errorbar(v, y, xerr=[v - df[lo], df[hi] - v], fmt="none", ecolor=INK2, capsize=3, elinewidth=1.1)
    ax.axvline(ref, color=INK2, linewidth=1)
    ax.set_yticks(y, df["female"] + " vs " + df["male"], fontsize=8.5, color=INK2)
    ax.set_xlabel(xlabel, color=INK2)
    ax.set_title(title, loc="left", fontsize=10.5)
    fig.tight_layout()
    fig.savefig(path, dpi=200, facecolor=SURFACE, bbox_inches="tight")
    plt.close(fig)


def fig_terms(df, value, lo, hi, xlabel, title, path, plt, extra=None):
    df = df.sort_values(value).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(8, 0.26 * len(df) + 1.5))
    style(ax)
    y = np.arange(len(df))
    v = df[value].to_numpy(float)
    ax.errorbar(v, y, xerr=[v - df[lo], df[hi] - v], fmt="o", color=INK, ms=4, ecolor=INK2, capsize=2.5)
    if extra is not None:
        e = df[extra[0]].to_numpy(float)
        ax.scatter(e, y, marker=extra[2], s=28, facecolor=SURFACE, edgecolor=MUTED, zorder=3, label=extra[1])
        ax.legend(frameon=False, fontsize=8, loc="lower right")
    ax.set_yticks(y, df["term"], fontsize=8, color=INK2)
    ax.set_xlabel(xlabel, color=INK2)
    ax.set_title(title, loc="left", fontsize=10.5)
    fig.tight_layout()
    fig.savefig(path, dpi=200, facecolor=SURFACE, bbox_inches="tight")
    plt.close(fig)


def md(df, cols=None):
    d = (df if cols is None else df[cols]).copy()
    for c in d.columns:
        if d[c].dtype.kind == "f":
            d[c] = d[c].map(lambda x: "" if pd.isna(x) else f"{x:.2f}")
    return "| " + " | ".join(d.columns) + " |\n|" + "---|" * len(d.columns) + "\n" + "\n".join(
        "| " + " | ".join(str(x) for x in r) + " |" for r in d.itertuples(index=False)) + "\n"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--scan", required=True)
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--B", type=int, default=200)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--min_docs", type=int, default=30, help="terms with fewer documents are listed but not plotted")
    ap.add_argument("--no_figures", action="store_true")
    args = ap.parse_args()
    os.makedirs(args.out_dir, exist_ok=True)
    plt = None
    if not args.no_figures:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    report = ["# Gender and age in C4: exclusion and sentiment\n"]

    # ---------- exclusion ----------
    X, cols, parts, nshards = load(args.scan, "doc_counts")
    col = {c: i for i, c in enumerate(cols)}
    terms = [c[:-5] for c in cols if c.endswith("|docs") and not c.startswith("dialect:") and c != "docs"]
    host, year = parts[0].to_numpy(), parts[1].to_numpy()
    bt = Boot(X, host, args.B, args.seed)
    point, boot = bt.totals()
    ex, _ = exclusion_tables(point, boot, col, terms)
    ex.round(3).to_csv(os.path.join(args.out_dir, "exclusion_by_term.csv"), index=False)
    report.append(f"{nshards} en.noblocklist shards, {int(point[col['docs']]):,} documents, "
                  f"{100 * point[col['excluded']] / point[col['docs']]:.2f}% excluded by the blocklist.\n")
    report.append("## Exclusion by term (PMI with being excluded; 95% CI over domains)\n")
    report.append(md(ex.sort_values("pmi", ascending=False),
                     ["term", "docs", "excluded_%", "pmi", "pmi_lo", "pmi_hi", "pmi_nonmechanical", "paper_fig5"]))

    def pmi_of(arr, t, kind="excluded"):
        if f"{t}|docs" not in col:
            raise KeyError(t)
        with np.errstate(divide="ignore", invalid="ignore"):
            return np.log(arr[..., col[f"{t}|{kind}"]] * arr[..., col["docs"]]
                          / (arr[..., col[f"{t}|docs"]] * arr[..., col["excluded"]]))
    gp = pair_gaps(point, boot, col, pmi_of)
    gpo = pair_gaps(point, boot, col, lambda a, t: pmi_of(a, t, "excluded_other"))
    gp = gp.merge(gpo.rename(columns={"gap": "gap_nonmechanical", "ci_lo": "nm_lo", "ci_hi": "nm_hi"}),
                  on=["female", "male"])
    gp.round(3).to_csv(os.path.join(args.out_dir, "exclusion_gaps.csv"), index=False)

    def excl_rate(arr, ts):
        with np.errstate(divide="ignore", invalid="ignore"):
            return 100 * sum(arr[..., col[f"{t}|excluded"]] for t in ts) / sum(arr[..., col[f"{t}|docs"]] for t in ts)
    ez, ezp = eth_zoom(point, boot, col, excl_rate)
    ez.round(3).to_csv(os.path.join(args.out_dir, "exclusion_gender_by_ethnicity.csv"), index=False)
    report.append(f"## Exclusion, gender first then race: removal rate of '<eth> women' minus '<eth> men' "
                  f"(pooled gap {ezp:.2f} points)\n")
    report.append(md(ez))
    report.append("## Exclusion: female minus male PMI for matched pairs\n")
    report.append(md(gp))

    # by URL year
    yrows = []
    for yb in YEARS:
        p, b = bt.totals(year == yb)
        e, _ = exclusion_tables(p, b, col, terms)
        g = pair_gaps(p, b, col, pmi_of)
        e["year"], g["year"] = yb, yb
        yrows.append((e, g))
    ey = pd.concat([e for e, _ in yrows])
    gy = pd.concat([g for _, g in yrows])
    ey.round(3).to_csv(os.path.join(args.out_dir, "exclusion_by_year.csv"), index=False)
    gy.round(3).to_csv(os.path.join(args.out_dir, "exclusion_gaps_by_year.csv"), index=False)
    share = {yb: int(bt.totals(year == yb)[0][col["docs"]]) for yb in YEARS}
    report.append(f"URL-year coverage (documents): {share}\n")

    # dialect x gender
    drows = []
    for d in ["AAE", "Hispanic", "Other", "WAE", "unassigned"]:
        for g in ("all", "female", "male"):
            k = f"dialect:{d}|{g}"
            if f"{k}|docs" not in col:
                continue
            n, e = point[col[f"{k}|docs"]], point[col[f"{k}|excluded"]]
            with np.errstate(divide="ignore", invalid="ignore"):
                lo, hi = ci(100 * boot[:, col[f"{k}|excluded"]] / boot[:, col[f"{k}|docs"]])
            drows.append({"dialect": d, "documents mentioning": g, "docs": int(n),
                          "removed_%": 100 * e / n if n else np.nan, "ci_lo": lo, "ci_hi": hi})
    if drows and sum(r["docs"] for r in drows):
        dg = pd.DataFrame(drows)
        dg.round(2).to_csv(os.path.join(args.out_dir, "dialect_by_gender.csv"), index=False)
        report.append("## Blocklist removal by dialect and gender mention (paper overall: AAE 42%, Hispanic 32%, WAE 6.2%)\n")
        report.append(md(dg))

    # ---------- sentiment ----------
    S, scols, sparts, _ = load(args.scan, "sent_counts")
    scol = {c: i for i, c in enumerate(scols)}
    sterms = [c[:-11] for c in scols if c.endswith("|paragraphs")]
    shost, syear, skept = sparts[0].to_numpy(), sparts[1].to_numpy(), sparts[2].to_numpy()
    sb = Boot(S, shost, args.B, args.seed)
    sent_out, gaps_out = [], []
    for label, mask in (("kept (C4.EN)", skept == "1"), ("excluded", skept == "0")):
        p, b = sb.totals(mask)
        t = sentiment_tables(p, b, scol, sterms)
        t.insert(0, "documents", label)
        sent_out.append(t)
        for lex in LEXICONS:
            g = pair_gaps(p, b, scol, lambda a, tt: positivity(a, scol, tt, lex))
            g.insert(0, "lexicon", lex), g.insert(0, "documents", label)
            gaps_out.append(g)
        if label.startswith("kept"):
            for lex in ("SocialSent", "VADER"):
                def pos_sum(arr, ts, lex=lex):
                    pp = sum(arr[..., scol[f"{t}|{lex}:pos"]] for t in ts)
                    nn = sum(arr[..., scol[f"{t}|{lex}:neg"]] for t in ts)
                    with np.errstate(divide="ignore", invalid="ignore"):
                        return 100 * pp / (pp + nn)
                sz, szp = eth_zoom(p, b, scol, pos_sum)
                sz.round(3).to_csv(os.path.join(args.out_dir, f"sentiment_gender_by_ethnicity_{lex}.csv"), index=False)
                report.append(f"## Sentiment ({lex}, C4.EN), gender first then race: positivity of '<eth> women' "
                              f"minus '<eth> men' (pooled gap {szp:.2f} points)\n")
                report.append(md(sz))
            ctrl = {}
            for lex in ("SocialSent", "VADER"):
                v = positivity(p, scol, "Jewish (paper)", lex) - positivity(p, scol, "Arab (paper)", lex)
                bb = positivity(b, scol, "Jewish (paper)", lex) - positivity(b, scol, "Arab (paper)", lex)
                ctrl[lex] = {"ours": round(float(v), 2), "ci": [round(float(x), 2) for x in ci(bb)],
                             "arm_b_c4en_12_shards": ARM_B[lex]}
            report.append(f"## Positive control: Jewish - Arab positivity gap (paper-style matching)\n\n{json.dumps(ctrl)}\n")
    st = pd.concat(sent_out)
    sg = pd.concat(gaps_out)
    st.round(2).to_csv(os.path.join(args.out_dir, "sentiment_by_term.csv"), index=False)
    sg.round(2).to_csv(os.path.join(args.out_dir, "sentiment_gaps.csv"), index=False)
    report.append("## Sentiment: female minus male positivity (points), kept documents\n")
    report.append(md(sg[sg["documents"].str.startswith("kept")]))
    report.append("## Sentiment by age term, kept documents (SocialSent)\n")
    report.append(md(st[(st["documents"].str.startswith("kept")) & (st["lexicon"] == "SocialSent")
                        & st["term"].isin(AGES)], ["term", "paragraphs", "positivity", "ci_lo", "ci_hi"]))

    yrows = []
    for yb in YEARS:
        p, b = sb.totals((syear == yb) & (skept == "1"))
        for lex in ("SocialSent", "VADER"):
            g = pair_gaps(p, b, scol, lambda a, tt: positivity(a, scol, tt, lex))
            g.insert(0, "lexicon", lex), g.insert(0, "year", yb)
            yrows.append(g)
    sgy = pd.concat(yrows)
    sgy.round(2).to_csv(os.path.join(args.out_dir, "sentiment_gaps_by_year.csv"), index=False)

    # ---------- classifier ----------
    ps = os.path.join(args.scan, "para_scores.csv.gz")
    if os.path.exists(ps):
        c = pd.read_csv(ps)
        c["host"] = c["url"].str.extract(r"^\w+://([^/]+)")[0]
        crow = []
        rng = np.random.default_rng(args.seed)
        for (t, k), d in c.groupby(["term", "kept"]):
            h = d.groupby("host")[["p_pos", "p_neg"]].sum()
            W = rng.poisson(1.0, size=(len(h), args.B))
            bp, bn = h["p_pos"].to_numpy() @ W, h["p_neg"].to_numpy() @ W
            with np.errstate(divide="ignore", invalid="ignore"):
                lo, hi = ci(100 * bp / (bp + bn))
            crow.append({"term": t, "kept": k, "paragraphs": len(d),
                         "positivity": 100 * h["p_pos"].sum() / (h["p_pos"].sum() + h["p_neg"].sum()),
                         "ci_lo": lo, "ci_hi": hi, "mean_pos_minus_neg": float((d["p_pos"] - d["p_neg"]).mean())})
        ct = pd.DataFrame(crow)
        ct.round(2).to_csv(os.path.join(args.out_dir, "classifier_by_term.csv"), index=False)
        kept = ct[ct["kept"] == 1].set_index("term")
        cg = [{"female": f, "male": m, "gap": kept.loc[f, "positivity"] - kept.loc[m, "positivity"]}
              for f, m in PAIRS if f in kept.index and m in kept.index]
        report.append("## Classifier (RoBERTa sentiment) positivity, kept documents: female minus male\n")
        report.append(md(pd.DataFrame(cg)))

    # ---------- figures ----------
    if plt is not None:
        big = ex[ex["docs"] >= args.min_docs]
        fig_terms(big, "pmi", "pmi_lo", "pmi_hi", "PMI with being removed by the blocklist (ln; 95% CI over domains)",
                  "Which gender / age mentions the blocklist removes (hollow: without mechanical matches)",
                  os.path.join(args.out_dir, "fig_exclusion_pmi.png"), plt, extra=("pmi_nonmechanical", "non-mechanical", "o"))
        okp = gp.dropna(subset=["gap"])
        fig_pairs(okp, "gap", "ci_lo", "ci_hi", "PMI(female term) - PMI(male term); > 0: female mentions removed more",
                  "Blocklist: female vs male mentions", os.path.join(args.out_dir, "fig_exclusion_gaps.png"), plt)
        k = sg[(sg["documents"].str.startswith("kept")) & (sg["lexicon"] == "SocialSent")].dropna(subset=["gap"])
        fig_pairs(k, "gap", "ci_lo", "ci_hi", "positivity(female) - positivity(male), points; SocialSent, C4.EN",
                  "Sentiment of paragraphs mentioning female vs male terms",
                  os.path.join(args.out_dir, "fig_sentiment_gaps.png"), plt)
        kt = st[(st["documents"].str.startswith("kept")) & (st["lexicon"] == "SocialSent") & (st["paragraphs"] >= args.min_docs)]
        fig_terms(kt, "positivity", "ci_lo", "ci_hi", "SocialSent positivity of paragraphs (%; 95% CI over domains)",
                  "Sentiment of C4.EN paragraphs mentioning each term", os.path.join(args.out_dir, "fig_sentiment_terms.png"), plt)
        fig, ax = plt.subplots(figsize=(7, 4))
        style(ax)
        for (f, m), colr in zip([("women", "men"), ("she/her", "he/him"), ("girl", "boy")], [C_F, "#6250d6", "#1baf7a"]):
            g = sgy[(sgy["female"] == f) & (sgy["lexicon"] == "SocialSent") & (sgy["year"] != "none")]
            x = np.arange(len(g))
            ax.errorbar(x, g["gap"], yerr=[g["gap"] - g["ci_lo"], g["ci_hi"] - g["gap"]], marker="o", color=colr,
                        capsize=3, label=f"{f} - {m}")
            ax.set_xticks(x, g["year"])
        ax.axhline(0, color=INK2, linewidth=1)
        ax.set_ylabel("female - male positivity (points)", color=INK2)
        ax.set_xlabel("year in the page's URL", color=INK2)
        ax.legend(frameon=False, fontsize=8)
        ax.set_title("Sentiment gap by URL year (C4.EN, SocialSent)", loc="left", fontsize=10.5)
        fig.tight_layout()
        fig.savefig(os.path.join(args.out_dir, "fig_sentiment_gap_by_year.png"), dpi=200, facecolor=SURFACE)
        plt.close(fig)

    open(os.path.join(args.out_dir, "summary.md"), "w", encoding="utf-8").write("\n".join(report))
    print("\n".join(report[:3]))
    print(f"wrote {args.out_dir}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/ga/scripts/classify_paragraphs.py
"""Extension, corpus: score the sampled paragraphs with a sentiment classifier (GPU if available).

The reproduction's corpus sentiment (A.7) counts lexicon words. Our scoping note asked whether a trained classifier
changes the picture, so the paragraphs sampled by scan_corpus.py (para_sample_*.jsonl.gz, a random sample per term,
from kept and from excluded documents) are scored with a 3-class sentiment model. Like the lexicon count, this is the
sentiment of the whole paragraph, not of the attitude toward the person mentioned.

Default model: cardiffnlp/twitter-roberta-base-sentiment-latest (RoBERTa-base, negative / neutral / positive).
Inputs are truncated to 256 tokens.

Writes <scan_dir>/para_scores.csv.gz: term, kept, year, url, p_neg, p_neu, p_pos. Resumable: paragraphs already in
the output (by term, url, text hash) are skipped.

Usage:
    python scripts/classify_paragraphs.py --scan results/C1_noblocklist_12_shards
"""
import argparse, csv, glob, gzip, hashlib, json, os, time

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer


def key(r):
    return f"{r['term']}|{r['url']}|{hashlib.md5(r['text'].encode()).hexdigest()[:12]}"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--scan", required=True)
    ap.add_argument("--model", default="cardiffnlp/twitter-roberta-base-sentiment-latest")
    ap.add_argument("--batch_size", type=int, default=64)
    ap.add_argument("--max_len", type=int, default=256)
    ap.add_argument("--limit", type=int, default=0, help="only the first N paragraphs (smoke tests)")
    ap.add_argument("--max_per_term", type=int, default=0,
                    help="at most N paragraphs per (term, kept), drawn at random with --seed (use on CPU)")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    args = ap.parse_args()

    rows = []
    for p in sorted(glob.glob(os.path.join(args.scan, "para_sample_*.jsonl.gz"))):
        with gzip.open(p, "rt", encoding="utf-8") as f:
            rows += [json.loads(l) for l in f]
    if args.max_per_term:
        import random
        rng, groups = random.Random(args.seed), {}
        for r in rows:
            groups.setdefault((r["term"], r["kept"]), []).append(r)
        rows = [x for g in groups.values() for x in (rng.sample(g, args.max_per_term) if len(g) > args.max_per_term else g)]
    if args.limit:
        rows = rows[:args.limit]
    out = os.path.join(args.scan, "para_scores.csv.gz")
    done = set()
    if os.path.exists(out):
        with gzip.open(out, "rt", newline="", encoding="utf-8") as f:
            done = {r["key"] for r in csv.DictReader(f)}
    todo = [r for r in rows if key(r) not in done]
    print(f"{len(rows)} sampled paragraphs, {len(todo)} to score on {args.device}", flush=True)
    if not todo:
        return

    tok = AutoTokenizer.from_pretrained(args.model)
    model = AutoModelForSequenceClassification.from_pretrained(args.model).to(args.device).eval()
    labels = [model.config.id2label[i].lower() for i in range(model.config.num_labels)]
    idx = {l: labels.index(l) for l in ("negative", "neutral", "positive")}
    new = not os.path.exists(out)
    t0 = time.time()
    with gzip.open(out, "at", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["key", "term", "kept", "year", "url", "p_neg", "p_neu", "p_pos"])
        for b in range(0, len(todo), args.batch_size):
            batch = todo[b:b + args.batch_size]
            enc = tok([r["text"] for r in batch], return_tensors="pt", padding=True, truncation=True,
                      max_length=args.max_len).to(args.device)
            with torch.no_grad():
                p = model(**enc).logits.float().softmax(-1).cpu().numpy()
            for r, q in zip(batch, p):
                w.writerow([key(r), r["term"], r["kept"], r["year"], r["url"], round(float(q[idx["negative"]]), 5),
                            round(float(q[idx["neutral"]]), 5), round(float(q[idx["positive"]]), 5)])
            if (b // args.batch_size) % 50 == 0:
                n = b + len(batch)
                print(f"  {n}/{len(todo)}, {n / (time.time() - t0):.0f}/s", flush=True)
    print("done", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/ga/data/ldnoobw_en_25e679f.txt
2g1c
2 girls 1 cup
acrotomophilia
alabama hot pocket
alaskan pipeline
anal
anilingus
anus
apeshit
arsehole
ass
asshole
assmunch
auto erotic
autoerotic
babeland
baby batter
baby juice
ball gag
ball gravy
ball kicking
ball licking
ball sack
ball sucking
bangbros
bareback
barely legal
barenaked
bastard
bastardo
bastinado
bbw
bdsm
beaner
beaners
beaver cleaver
beaver lips
bestiality
big black
big breasts
big knockers
big tits
bimbos
birdlock
bitch
bitches
black cock
blonde action
blonde on blonde action
blowjob
blow job
blow your load
blue waffle
blumpkin
bollocks
bondage
boner
boob
boobs
booty call
brown showers
brunette action
bukkake
bulldyke
bullet vibe
bullshit
bung hole
bunghole
busty
butt
buttcheeks
butthole
camel toe
camgirl
camslut
camwhore
carpet muncher
carpetmuncher
chocolate rosebuds
circlejerk
cleveland steamer
clit
clitoris
clover clamps
clusterfuck
cock
cocks
coprolagnia
coprophilia
cornhole
coon
coons
creampie
cum
cumming
cunnilingus
cunt
darkie
date rape
daterape
deep throat
deepthroat
dendrophilia
dick
dildo
dingleberry
dingleberries
dirty pillows
dirty sanchez
doggie style
doggiestyle
doggy style
doggystyle
dog style
dolcett
domination
dominatrix
dommes
donkey punch
double dong
double penetration
dp action
dry hump
dvda
eat my ass
ecchi
ejaculation
erotic
erotism
escort
eunuch
faggot
fecal
felch
fellatio
feltch
female squirting
femdom
figging
fingerbang
fingering
fisting
foot fetish
footjob
frotting
fuck
fuck buttons
fuckin
fucking
fucktards
fudge packer
fudgepacker
futanari
gang bang
gay sex
genitals
giant cock
girl on
girl on top
girls gone wild
goatcx
goatse
god damn
gokkun
golden shower
goodpoop
goo girl
goregasm
grope
group sex
g-spot
guro
hand job
handjob
hard core
hardcore
hentai
homoerotic
honkey
hooker
hot carl
hot chick
how to kill
how to murder
huge fat
humping
incest
intercourse
jack off
jail bait
jailbait
jelly donut
jerk off
jigaboo
jiggaboo
jiggerboo
jizz
juggs
kike
kinbaku
kinkster
kinky
knobbing
leather restraint
leather straight jacket
lemon party
lolita
lovemaking
make me come
male squirting
masturbate
menage a trois
milf
missionary position
motherfucker
mound of venus
mr hands
muff diver
muffdiving
nambla
nawashi
negro
neonazi
nigga
nigger
nig nog
nimphomania
nipple
nipples
nsfw images
nude
nudity
nympho
nymphomania
octopussy
omorashi
one cup two girls
one guy one jar
orgasm
orgy
paedophile
paki
panties
panty
pedobear
pedophile
pegging
penis
phone sex
piece of shit
pissing
piss pig
pisspig
playboy
pleasure chest
pole smoker
ponyplay
poof
poon
poontang
punany
poop chute
poopchute
porn
porno
pornography
prince albert piercing
pthc
pubes
pussy
queaf
queef
quim
raghead
raging boner
rape
raping
rapist
rectum
reverse cowgirl
rimjob
rimming
rosy palm
rosy palm and her 5 sisters
rusty trombone
sadism
santorum
scat
schlong
scissoring
semen
sex
sexo
sexy
shaved beaver
shaved pussy
shemale
shibari
shit
shitblimp
shitty
shota
shrimping
skeet
slanteye
slut
s&m
smut
snatch
snowballing
sodomize
sodomy
spic
splooge
splooge moose
spooge
spread legs
spunk
strap on
strapon
strappado
strip club
style doggy
suck
sucks
suicide girls
sultry women
swastika
swinger
tainted love
taste my
tea bagging
threesome
throating
tied up
tight white
tit
tits
titties
titty
tongue in a
topless
tosser
towelhead
tranny
tribadism
tub girl
tubgirl
tushy
twat
twink
twinkie
two girls one cup
undressing
upskirt
urethra play
urophilia
vagina
venus mound
vibrator
violet wand
vorarephilia
voyeur
vulva
wank
wetback
wet dream
white power
wrapping men
wrinkled starfish
xx
xxx
yaoi
yellow showers
yiffy
zoophilia
🖕

In [ ]:
%%writefile /kaggle/working/ga/data/socialsent_250.tsv
word	polarity	score
beautiful	pos	2.9491
lovely	pos	2.8998
gorgeous	pos	2.8991
cool	pos	2.7289
thank	pos	2.6888
wonderful	pos	2.6807
sweet	pos	2.6364
great	pos	2.6355
wow	pos	2.6042
thanks	pos	2.5809
merry	pos	2.4667
btw	pos	2.4577
congrats	pos	2.4496
artwork	pos	2.4250
supportive	pos	2.4051
brilliant	pos	2.3753
appreciate	pos	2.3746
kudos	pos	2.3646
badass	pos	2.3575
spectacular	pos	2.3415
neat	pos	2.3385
thx	pos	2.3305
sharing	pos	2.3301
incredible	pos	2.3105
cheers	pos	2.3095
haha	pos	2.2920
appreciated	pos	2.2842
impressive	pos	2.2566
sir	pos	2.2321
soundtrack	pos	2.2222
welcome	pos	2.2191
adorable	pos	2.2147
dope	pos	2.2065
dude	pos	2.1956
excellent	pos	2.1951
helpful	pos	2.1682
epic	pos	2.1578
hey	pos	2.1438
respectful	pos	2.1402
enjoy	pos	2.1304
informative	pos	2.1085
man	pos	2.1064
congratulations	pos	2.1052
birthday	pos	2.0931
props	pos	2.0884
damn	pos	2.0755
classy	pos	2.0629
mate	pos	2.0462
luck	pos	2.0271
ty	pos	2.0265
bro	pos	2.0049
good	pos	1.9770
oh	pos	1.9380
outstanding	pos	1.9240
hugs	pos	1.9200
phenomenal	pos	1.9164
alright	pos	1.9038
hahaha	pos	1.8952
sexy	pos	1.8705
ah	pos	1.8379
polite	pos	1.8154
woah	pos	1.8115
interesting	pos	1.8082
bravo	pos	1.8005
gl	pos	1.7932
hi	pos	1.7821
buddy	pos	1.7779
generous	pos	1.7712
gotcha	pos	1.7599
ahh	pos	1.7542
wish	pos	1.7541
clarification	pos	1.7528
detailed	pos	1.7507
loving	pos	1.7486
celebrate	pos	1.7477
worries	pos	1.7428
looks	pos	1.7348
holidays	pos	1.7305
10/10	pos	1.7300
lol	pos	1.7249
ooh	pos	1.7240
fun	pos	1.7032
xd	pos	1.6976
giveaway	pos	1.6895
holy	pos	1.6820
bud	pos	1.6811
glad	pos	1.6662
freaking	pos	1.6563
ha	pos	1.6526
whoa	pos	1.6384
ya	pos	1.6349
wallpaper	pos	1.6326
amazingly	pos	1.6302
glorious	pos	1.6254
grateful	pos	1.6249
hehe	pos	1.6208
yep	pos	1.6204
enjoyed	pos	1.6202
compliment	pos	1.6141
yay	pos	1.6123
lt	pos	1.6090
yea	pos	1.6062
insight	pos	1.6035
relaxing	pos	1.6025
dang	pos	1.6015
yeah	pos	1.6000
friendly	pos	1.5986
nonetheless	pos	1.5856
ahhh	pos	1.5856
bless	pos	1.5846
hello	pos	1.5787
christmas	pos	1.5773
tho	pos	1.5645
talented	pos	1.5547
pleasant	pos	1.5523
p	pos	1.5504
thats	pos	1.5440
yup	pos	1.5394
nah	pos	1.5388
ok	pos	1.5311
vid	pos	1.5240
genius	pos	1.5163
feedback	pos	1.5142
thanksgiving	pos	1.5127
correction	pos	1.5093
gosh	pos	1.5073
p.s	pos	1.5069
cake	pos	1.5061
tasty	pos	1.4980
hahah	pos	1.4958
guys	pos	1.4956
cute	pos	1.4904
hah	pos	1.4897
job	pos	1.4780
aww	pos	1.4648
chill	pos	1.4620
nailed	pos	1.4549
suggestions	pos	1.4526
art	pos	1.4512
dat	pos	1.4439
pal	pos	1.4342
perfect	pos	1.4195
advice	pos	1.4157
m8	pos	1.4123
dear	pos	1.4053
fellow	pos	1.4002
goodness	pos	1.4001
tips	pos	1.3980
omg	pos	1.3938
huh	pos	1.3881
okay	pos	1.3849
inspiration	pos	1.3781
corrected	pos	1.3662
woo	pos	1.3564
aw	pos	1.3463
tip	pos	1.3422
holiday	pos	1.3405
info	pos	1.3361
unique	pos	1.3345
humble	pos	1.3287
favorite	pos	1.3191
uh	pos	1.3188
indeed	pos	1.3097
lucky	pos	1.3032
input	pos	1.3011
hahahaha	pos	1.3003
ta	pos	1.2901
wishes	pos	1.2899
artist	pos	1.2820
thankful	pos	1.2735
suggestion	pos	1.2707
username	pos	1.2646
heh	pos	1.2497
ohh	pos	1.2467
fuckin	pos	1.2398
vids	pos	1.2349
darn	pos	1.2327
clever	pos	1.2287
nicely	pos	1.2255
calm	pos	1.2211
amen	pos	1.2151
beauty	pos	1.2146
constructive	pos	1.2126
presents	pos	1.2117
hmm	pos	1.2090
sincerely	pos	1.2060
god	pos	1.1915
lmao	pos	1.1901
hope	pos	1.1854
documentary	pos	1.1849
exciting	pos	1.1792
gift	pos	1.1737
nope	pos	1.1732
kidding	pos	1.1695
delicious	pos	1.1664
grammar	pos	1.1649
wedding	pos	1.1515
reply	pos	1.1500
pls	pos	1.1482
gifts	pos	1.1435
hilarious	pos	1.1369
edit	pos	1.1351
creative	pos	1.1320
wut	pos	1.1285
clearing	pos	1.1277
favourite	pos	1.1269
yes	pos	1.1244
wat	pos	1.1219
solid	pos	1.1207
apologies	pos	1.1170
spelling	pos	1.1122
smooth	pos	1.1117
plz	pos	1.1114
unbelievable	pos	1.1106
artists	pos	1.1095
anniversary	pos	1.1014
u	pos	1.1007
puppy	pos	1.0964
series	pos	1.0959
lord	pos	1.0885
parks	pos	1.0863
album	pos	1.0819
um	pos	1.0814
truly	pos	1.0798
mr	pos	1.0774
promising	pos	1.0772
easter	pos	1.0746
ps	pos	1.0745
enjoying	pos	1.0739
o_o	pos	1.0735
fancy	pos	1.0721
warm	pos	1.0717
movie	pos	1.0656
underrated	pos	1.0597
soo	pos	1.0567
forgot	pos	1.0550
ur	pos	1.0542
recommendations	pos	1.0541
recommendation	pos	1.0525
brave	pos	1.0495
anyways	pos	1.0493
bye	pos	1.0448
dem	pos	1.0411
music	pos	1.0352
accurate	pos	1.0350
o	pos	1.0336
hug	pos	1.0306
reminds	pos	1.0239
compelling	pos	1.0218
damnit	pos	1.0204
shitty	neg	-3.8013
hating	neg	-3.5363
crappy	neg	-3.4425
disgusting	neg	-3.2327
worse	neg	-3.0114
bad	neg	-2.9648
unplayable	neg	-2.9608
poor	neg	-2.9500
dislike	neg	-2.7634
hatred	neg	-2.6279
pathetic	neg	-2.5961
garbage	neg	-2.5807
gays	neg	-2.5627
miserable	neg	-2.5159
nightmare	neg	-2.4581
buggy	neg	-2.4547
stupid	neg	-2.4389
smells	neg	-2.3656
gross	neg	-2.3564
annoys	neg	-2.3559
bashing	neg	-2.3340
painful	neg	-2.3286
tragic	neg	-2.3278
questionable	neg	-2.2969
bland	neg	-2.2869
complained	neg	-2.2571
pisses	neg	-2.2451
sucked	neg	-2.1799
haters	neg	-2.1796
blame	neg	-2.1681
complain	neg	-2.1366
cruel	neg	-2.1288
tragedy	neg	-2.1187
headache	neg	-2.1009
suck	neg	-2.0915
infection	neg	-2.0563
inconsistent	neg	-1.9970
unacceptable	neg	-1.9492
disaster	neg	-1.9491
annoying	neg	-1.9318
unfortunate	neg	-1.9127
mess	neg	-1.9041
fanboy	neg	-1.8984
criticize	neg	-1.8970
bitter	neg	-1.8808
depressed	neg	-1.8776
stressed	neg	-1.8751
suffered	neg	-1.8672
blamed	neg	-1.8540
lonely	neg	-1.8486
idiotic	neg	-1.8446
piss	neg	-1.8331
insecure	neg	-1.8139
suffer	neg	-1.8112
mediocre	neg	-1.7857
bothers	neg	-1.7844
dirty	neg	-1.7816
hypocritical	neg	-1.7750
embarrassing	neg	-1.7747
bullshit	neg	-1.7722
blaming	neg	-1.7659
chemicals	neg	-1.7640
sucks	neg	-1.7606
retarded	neg	-1.7506
horribly	neg	-1.7437
ignorant	neg	-1.7277
disrespectful	neg	-1.7198
suffering	neg	-1.7153
symptoms	neg	-1.7108
incompetent	neg	-1.7053
selfish	neg	-1.6989
obnoxious	neg	-1.6809
angry	neg	-1.6772
dumb	neg	-1.6700
bitching	neg	-1.6687
racist	neg	-1.6620
messy	neg	-1.6522
complaining	neg	-1.6517
dull	neg	-1.6413
worthless	neg	-1.6363
toxic	neg	-1.6276
childish	neg	-1.6270
utter	neg	-1.6158
minorities	neg	-1.6107
bugged	neg	-1.6060
itt	neg	-1.5901
assholes	neg	-1.5884
smell	neg	-1.5858
disappointing	neg	-1.5843
immature	neg	-1.5748
severe	neg	-1.5692
sexist	neg	-1.5670
torture	neg	-1.5604
lazy	neg	-1.5596
whine	neg	-1.5583
fault	neg	-1.5581
pain	neg	-1.5527
ugly	neg	-1.5520
ugh	neg	-1.5515
crap	neg	-1.5416
thinks	neg	-1.5375
trash	neg	-1.5309
sore	neg	-1.5302
whining	neg	-1.5269
ridiculous	neg	-1.5265
useless	neg	-1.5166
arrogant	neg	-1.5092
butthurt	neg	-1.5088
scum	neg	-1.5082
greedy	neg	-1.4964
absolute	neg	-1.4898
scenario	neg	-1.4865
rid	neg	-1.4793
circlejerk	neg	-1.4646
awkward	neg	-1.4627
ruined	neg	-1.4578
unfair	neg	-1.4496
idiots	neg	-1.4489
oppression	neg	-1.4449
petty	neg	-1.4417
excuse	neg	-1.4367
burnt	neg	-1.4358
nazis	neg	-1.4226
screwed	neg	-1.4210
messed	neg	-1.4165
excuses	neg	-1.4159
shocking	neg	-1.4077
fucked	neg	-1.4011
pointless	neg	-1.3975
irrational	neg	-1.3973
racism	neg	-1.3972
saying	neg	-1.3925
smug	neg	-1.3853
insulting	neg	-1.3848
diagnosed	neg	-1.3842
overpriced	neg	-1.3818
lame	neg	-1.3796
uncomfortable	neg	-1.3728
discrimination	neg	-1.3707
anger	neg	-1.3704
anxiety	neg	-1.3660
diseases	neg	-1.3645
tired	neg	-1.3640
shady	neg	-1.3531
shitting	neg	-1.3493
likes	neg	-1.3461
rude	neg	-1.3441
outdated	neg	-1.3411
hurts	neg	-1.3281
cares	neg	-1.3273
jerk	neg	-1.3266
stomach	neg	-1.3238
cringe	neg	-1.3182
illness	neg	-1.3140
harmful	neg	-1.3104
depression	neg	-1.3041
condescending	neg	-1.3010
disease	neg	-1.2926
nonsense	neg	-1.2847
poop	neg	-1.2810
sweat	neg	-1.2713
stressful	neg	-1.2687
bizarre	neg	-1.2676
entitled	neg	-1.2605
guilty	neg	-1.2605
silly	neg	-1.2574
dae	neg	-1.2567
boring	neg	-1.2564
knows	neg	-1.2520
calling	neg	-1.2518
pee	neg	-1.2480
believes	neg	-1.2478
naive	neg	-1.2456
depressing	neg	-1.2417
brutal	neg	-1.2372
feminists	neg	-1.2344
delusional	neg	-1.2339
experiencing	neg	-1.2335
asshole	neg	-1.2277
morally	neg	-1.2248
caused	neg	-1.2207
bs	neg	-1.2207
rushed	neg	-1.2160
prone	neg	-1.2126
guilt	neg	-1.2122
idiot	neg	-1.2040
frustrating	neg	-1.2028
prick	neg	-1.2016
shallow	neg	-1.2000
flawed	neg	-1.1981
hurt	neg	-1.1977
disorder	neg	-1.1963
secretly	neg	-1.1953
anti	neg	-1.1870
freaked	neg	-1.1786
fear	neg	-1.1778
confused	neg	-1.1771
ashamed	neg	-1.1768
bleeding	neg	-1.1746
upset	neg	-1.1733
annoyed	neg	-1.1707
hitler	neg	-1.1704
violent	neg	-1.1699
unnecessary	neg	-1.1678
defending	neg	-1.1623
mentality	neg	-1.1581
pile	neg	-1.1572
harsh	neg	-1.1564
unreasonable	neg	-1.1562
sees	neg	-1.1555
bash	neg	-1.1555
cry	neg	-1.1515
inappropriate	neg	-1.1485
rage	neg	-1.1483
aids	neg	-1.1447
frustrated	neg	-1.1421
embarrassed	neg	-1.1349
struggled	neg	-1.1345
pissed	neg	-1.1338
douche	neg	-1.1309
offended	neg	-1.1291
weak	neg	-1.1287
causes	neg	-1.1248
desperate	neg	-1.1246
everywhere	neg	-1.1244
associate	neg	-1.1232
lying	neg	-1.1230
filthy	neg	-1.1214
arguing	neg	-1.1209
disappointment	neg	-1.1206
except	neg	-1.1173
terrified	neg	-1.1152
blatant	neg	-1.1135
virus	neg	-1.1133
corrupt	neg	-1.1092
latency	neg	-1.1064
pity	neg	-1.0993
disturbing	neg	-1.0952
injury	neg	-1.0947
morons	neg	-1.0864
salty	neg	-1.0788
spit	neg	-1.0783
terror	neg	-1.0765
crimes	neg	-1.0742
ruins	neg	-1.0722
causing	neg	-1.0710
blatantly	neg	-1.0627
due	neg	-1.0579
moron	neg	-1.0550
broken	neg	-1.0548

In [ ]:
%%writefile /kaggle/working/ga/data/vader.tsv
word	polarity	score
$:	neg	-1.5000
%-)	neg	-1.5000
( '}{' )	pos	1.6000
('-:	pos	2.2000
(':	pos	2.3000
((-:	pos	2.1000
(*	pos	1.1000
(-*	pos	1.3000
(-:	pos	1.6000
(-:0	pos	2.8000
(-:o	pos	1.5000
(-:O	pos	1.5000
(-:|>*	pos	1.9000
(-;	pos	1.3000
(-;|	pos	2.1000
(8	pos	2.6000
(:	pos	2.2000
(:0	pos	2.4000
(:o	pos	2.5000
(:O	pos	2.5000
(;	pos	1.1000
(=	pos	2.2000
(?:	pos	2.1000
(^:	pos	1.5000
(^;	pos	1.5000
(^;0	pos	2.0000
(^;o	pos	1.9000
(o:	pos	1.6000
)':	neg	-2.0000
)-':	neg	-2.1000
)-:	neg	-2.1000
)-:<	neg	-2.2000
)-:{	neg	-2.1000
):	neg	-1.8000
):<	neg	-1.9000
):{	neg	-2.3000
);<	neg	-2.6000
*-:	pos	2.1000
*-;	pos	2.4000
*:	pos	1.9000
*<|:-)	pos	1.6000
*\0/*	pos	2.3000
*^:	pos	1.6000
,-:	pos	1.2000
---'-;-{@	pos	2.3000
--<--<@	pos	2.2000
.-:	neg	-1.2000
..###-:	neg	-1.7000
..###:	neg	-1.9000
/-:	neg	-1.3000
/:	neg	-1.3000
/:<	neg	-1.4000
/o:	neg	-1.4000
0-|	neg	-1.2000
0:)	pos	1.9000
0:-)	pos	1.4000
0:-3	pos	1.5000
0:03	pos	1.9000
0;^)	pos	1.6000
10q	pos	2.1000
1337	pos	2.1000
143	pos	3.2000
1432	pos	2.6000
14aa41	pos	2.4000
182	neg	-2.9000
187	neg	-3.1000
2g2b4g	pos	2.8000
2qt	pos	2.1000
3:(	neg	-2.2000
3:-(	neg	-2.3000
3:-)	neg	-1.4000
4col	neg	-2.2000
4q	neg	-3.1000
5fs	pos	1.5000
8)	pos	1.9000
8-d	pos	1.7000
86	neg	-1.6000
8d	pos	2.9000
:###..	neg	-2.4000
:'(	neg	-2.2000
:')	pos	2.3000
:'-(	neg	-2.4000
:'-)	pos	2.7000
:(	neg	-1.9000
:)	pos	2.0000
:*	pos	2.5000
:-###..	neg	-2.5000
:-(	neg	-1.5000
:-)	pos	1.3000
:-))	pos	2.8000
:-*	pos	1.7000
:-,	pos	1.1000
:-/	neg	-1.2000
:-<	neg	-1.5000
:-d	pos	2.3000
:-D	pos	2.3000
:-p	pos	1.2000
:-[	neg	-1.6000
:-c	neg	-1.3000
:-p	pos	1.5000
:-||	neg	-2.5000
:/	neg	-1.4000
:3	pos	2.3000
:<	neg	-2.1000
:>	pos	2.1000
:?)	pos	1.3000
:?c	neg	-1.6000
:@	neg	-2.5000
:d	pos	2.3000
:D	pos	2.3000
:l	neg	-1.7000
:s	neg	-1.2000
:[	neg	-2.0000
:\	neg	-1.3000
:]	pos	2.2000
:^)	pos	2.1000
:^*	pos	2.6000
:^/	neg	-1.2000
:c	neg	-2.1000
:c)	pos	2.0000
:o)	pos	2.1000
:o/	neg	-1.4000
:o\	neg	-1.1000
:P	pos	1.4000
:{	neg	-1.9000
:}	pos	2.1000
:Þ	pos	1.1000
;-*	pos	2.2000
;^)	pos	1.4000
</3	neg	-3.0000
<3	pos	1.9000
<:	pos	2.1000
<:-|	neg	-1.4000
=)	pos	2.2000
=-3	pos	2.0000
=-d	pos	2.4000
=-D	pos	2.4000
=/	neg	-1.4000
=3	pos	2.1000
=d	pos	2.3000
=D	pos	2.3000
=l	neg	-1.2000
=\	neg	-1.2000
=]	pos	1.6000
=p	pos	1.3000
>-:	neg	-2.0000
>.<	neg	-1.3000
>:	neg	-2.1000
>:(	neg	-2.7000
>:-(	neg	-2.7000
>:/	neg	-1.6000
>:o	neg	-1.2000
>:[	neg	-2.1000
>:\	neg	-1.7000
>;(	neg	-2.9000
>_>^	pos	2.1000
@:	neg	-2.1000
@>-->--	pos	2.1000
@}-;-'---	pos	2.2000
aas	pos	2.5000
aayf	pos	2.7000
afu	neg	-2.9000
alol	pos	2.8000
ambw	pos	2.9000
aml	pos	3.4000
atab	neg	-1.9000
awol	neg	-1.3000
ayor	neg	-1.2000
bfd	neg	-2.7000
bfe	neg	-2.6000
bff	pos	2.9000
bl	pos	2.3000
bsod	neg	-2.2000
btd	neg	-2.1000
b^d	pos	2.6000
cwot	neg	-2.3000
d-':	neg	-2.5000
d8	neg	-3.2000
d:	neg	-2.9000
d:<	neg	-3.2000
d;	neg	-2.9000
d=	neg	-3.0000
doa	neg	-2.3000
dx	neg	-3.0000
ez	pos	1.5000
fav	pos	2.4000
fcol	neg	-1.8000
ff	pos	1.8000
ffs	neg	-2.8000
fkm	neg	-2.4000
foaf	pos	1.8000
ftw	pos	2.0000
fu	neg	-3.7000
fubar	neg	-3.0000
fwb	pos	2.5000
g1	pos	1.4000
gg	pos	1.2000
gga	pos	1.7000
gj	pos	2.0000
gl	pos	1.3000
gla	pos	2.5000
gn	pos	1.2000
gr8	pos	2.7000
gt	pos	1.1000
h&k	pos	2.3000
hagd	pos	2.2000
hagn	pos	2.2000
hago	pos	1.2000
hak	pos	1.9000
hand	pos	2.2000
heart	pos	3.2000
hearts	pos	3.3000
hho1/2k	pos	1.4000
hhoj	pos	2.0000
hugz	pos	2.0000
hi5	pos	1.9000
ilu	pos	3.4000
iluaaf	pos	2.7000
ily	pos	3.4000
ily2	pos	2.6000
iyq	pos	2.3000
j/j	pos	2.0000
j/k	pos	1.6000
j/p	pos	1.4000
j4f	pos	1.4000
j4g	pos	1.7000
jw	pos	1.6000
jealz	neg	-1.2000
k4y	pos	2.3000
kfy	pos	2.3000
kia	neg	-3.2000
kk	pos	1.5000
kmuf	pos	2.2000
l	pos	2.0000
l&r	pos	2.2000
laoj	pos	1.3000
lmao	pos	2.0000
lmbao	pos	1.8000
lmfao	pos	2.5000
lmso	pos	2.7000
lol	pos	2.9000
lolz	pos	2.7000
lts	pos	1.6000
ly	pos	2.6000
ly4e	pos	2.7000
lya	pos	3.3000
lyb	pos	3.0000
lyl	pos	3.1000
lylab	pos	2.7000
lylas	pos	2.6000
lylb	pos	1.6000
m8	pos	1.4000
mia	neg	-1.2000
mml	pos	2.0000
mofo	neg	-2.4000
muah	pos	2.8000
mwah	pos	2.5000
n1	pos	1.9000
nbd	pos	1.3000
nfc	neg	-2.7000
nfw	neg	-2.4000
nh	pos	2.2000
nimy	neg	-1.4000
nitl	neg	-1.5000
nme	neg	-2.1000
np	pos	1.4000
ntmu	pos	1.4000
o-|	neg	-1.1000
o:)	pos	1.5000
o:-)	pos	2.0000
o:-3	pos	2.2000
o:3	pos	2.3000
o;^)	pos	1.6000
ok	pos	1.6000
pita	neg	-2.4000
po	neg	-2.6000
ptl	pos	2.6000
pu	neg	-1.1000
qq	neg	-2.2000
qt	pos	1.8000
r&r	pos	2.4000
rofl	pos	2.7000
roflmao	pos	2.5000
rotfl	pos	2.6000
rotflmao	pos	2.8000
rotflmfao	pos	2.5000
rotflol	pos	3.0000
rotgl	pos	2.9000
rotglmao	pos	1.8000
s:	neg	-1.1000
sapfu	neg	-1.1000
sete	pos	2.8000
sfete	pos	2.7000
sgtm	pos	2.4000
slaw	pos	2.1000
smh	neg	-1.3000
snafu	neg	-2.5000
sob	neg	-2.8000
swak	pos	2.3000
tgif	pos	2.3000
thks	pos	1.4000
thx	pos	1.5000
tia	pos	2.3000
tnx	pos	1.1000
true	pos	1.8000
tx	pos	1.5000
txs	pos	1.1000
ty	pos	1.6000
tyvm	pos	2.5000
urw	pos	1.9000
vbg	pos	2.1000
vbs	pos	3.1000
vip	pos	2.3000
vwd	pos	2.6000
vwp	pos	2.1000
wd	pos	2.7000
wtf	neg	-2.8000
wtg	pos	2.1000
wth	neg	-2.4000
x-d	pos	2.7000
x-p	pos	1.8000
xd	pos	2.7000
xlnt	pos	3.0000
xoxo	pos	3.0000
xoxozzz	pos	2.3000
xp	pos	1.2000
xqzt	pos	1.6000
yolo	pos	1.1000
yvw	pos	1.6000
yw	pos	1.8000
ywia	pos	2.5000
zzz	neg	-1.2000
[:	pos	1.3000
[=	pos	1.7000
\:<	neg	-1.7000
\=	neg	-1.1000
\^:	neg	-1.3000
\o/	pos	2.2000
\o:	neg	-1.2000
]-:	neg	-2.1000
]:	neg	-1.6000
]:<	neg	-2.5000
^<_<	pos	1.4000
^urs	neg	-2.8000
abandon	neg	-1.9000
abandoned	neg	-2.0000
abandoner	neg	-1.9000
abandoners	neg	-1.9000
abandoning	neg	-1.6000
abandonment	neg	-2.4000
abandonments	neg	-1.7000
abandons	neg	-1.3000
abducted	neg	-2.3000
abduction	neg	-2.8000
abductions	neg	-2.0000
abhor	neg	-2.0000
abhorred	neg	-2.4000
abhorrent	neg	-3.1000
abhors	neg	-2.9000
ability	pos	1.3000
absentee	neg	-1.1000
absolve	pos	1.2000
absolved	pos	1.5000
absolves	pos	1.3000
absolving	pos	1.6000
abuse	neg	-3.2000
abused	neg	-2.3000
abuser	neg	-2.6000
abusers	neg	-2.6000
abuses	neg	-2.6000
abusing	neg	-2.0000
abusive	neg	-3.2000
abusively	neg	-2.8000
abusiveness	neg	-2.5000
abusivenesses	neg	-3.0000
accept	pos	1.6000
acceptabilities	pos	1.6000
acceptability	pos	1.1000
acceptable	pos	1.3000
acceptableness	pos	1.3000
acceptably	pos	1.5000
acceptance	pos	2.0000
acceptances	pos	1.7000
acceptant	pos	1.6000
acceptation	pos	1.3000
accepted	pos	1.1000
accepting	pos	1.6000
accepts	pos	1.3000
accident	neg	-2.1000
accidentally	neg	-1.4000
accidents	neg	-1.3000
accomplish	pos	1.8000
accomplished	pos	1.9000
accomplishes	pos	1.7000
accusations	neg	-1.3000
accused	neg	-1.2000
accuses	neg	-1.4000
ache	neg	-1.6000
ached	neg	-1.6000
achievable	pos	1.3000
aching	neg	-2.2000
acquitting	pos	1.3000
acrimonious	neg	-1.7000
active	pos	1.7000
actively	pos	1.3000
actives	pos	1.1000
admirability	pos	2.4000
admirable	pos	2.6000
admirableness	pos	2.2000
admirably	pos	2.5000
admiral	pos	1.3000
admirals	pos	1.5000
admiralties	pos	1.6000
admiralty	pos	1.2000
admiration	pos	2.5000
admirations	pos	1.6000
admire	pos	2.1000
admired	pos	2.3000
admirer	pos	1.8000
admirers	pos	1.7000
admires	pos	1.5000
admiring	pos	1.6000
admiringly	pos	2.3000
admits	pos	1.2000
admonished	neg	-1.9000
adorability	pos	2.2000
adorable	pos	2.2000
adorableness	pos	2.5000
adorably	pos	2.1000
adoration	pos	2.9000
adorations	pos	2.2000
adore	pos	2.6000
adored	pos	1.8000
adorer	pos	1.7000
adorers	pos	2.1000
adores	pos	1.6000
adoring	pos	2.6000
adoringly	pos	2.4000
adorner	pos	1.3000
adornment	pos	1.3000
advantaged	pos	1.4000
advantageous	pos	1.5000
advantageously	pos	1.9000
advantageousness	pos	1.6000
advantages	pos	1.5000
advantaging	pos	1.6000
adventure	pos	1.3000
adventured	pos	1.3000
adventurer	pos	1.2000
adventures	pos	1.4000
adventuresome	pos	1.7000
adventuresomeness	pos	1.3000
adventuresses	pos	1.4000
adventuring	pos	2.3000
adventurism	pos	1.5000
adventurist	pos	1.4000
adventuristic	pos	1.7000
adventurists	pos	1.2000
adventurous	pos	1.4000
adventurously	pos	1.3000
adventurousness	pos	1.8000
adversarial	neg	-1.5000
adversative	neg	-1.2000
adverse	neg	-1.5000
adversities	neg	-1.5000
adversity	neg	-1.8000
affection	pos	2.4000
affectional	pos	1.9000
affectionally	pos	1.5000
affectionate	pos	1.9000
affectionately	pos	2.2000
affectioned	pos	1.8000
affectionless	neg	-2.0000
affections	pos	1.5000
afflicted	neg	-1.5000
aggravate	neg	-2.5000
aggravated	neg	-1.9000
aggravates	neg	-1.9000
aggravating	neg	-1.2000
aggress	neg	-1.3000
aggressed	neg	-1.4000
aggression	neg	-1.2000
aggressions	neg	-1.3000
aggressively	neg	-1.3000
aggressiveness	neg	-1.8000
aggressivities	neg	-1.4000
aghast	neg	-1.9000
agitate	neg	-1.7000
agitated	neg	-2.0000
agitatedly	neg	-1.6000
agitates	neg	-1.4000
agitating	neg	-1.8000
agitational	neg	-1.2000
agitations	neg	-1.3000
agitative	neg	-1.3000
agitator	neg	-1.4000
agitators	neg	-2.1000
agog	pos	1.9000
agonise	neg	-2.1000
agonised	neg	-2.3000
agonises	neg	-2.4000
agonising	neg	-1.5000
agonize	neg	-2.3000
agonized	neg	-2.2000
agonizes	neg	-2.3000
agonizing	neg	-2.7000
agonizingly	neg	-2.3000
agony	neg	-1.8000
agree	pos	1.5000
agreeability	pos	1.9000
agreeable	pos	1.8000
agreeableness	pos	1.8000
agreeablenesses	pos	1.3000
agreeably	pos	1.6000
agreed	pos	1.1000
agreeing	pos	1.4000
agreement	pos	2.2000
agreements	pos	1.1000
alarm	neg	-1.4000
alarmed	neg	-1.4000
alarmingly	neg	-2.6000
alarmists	neg	-1.1000
alarms	neg	-1.1000
alas	neg	-1.1000
alert	pos	1.2000
alienation	neg	-1.1000
alive	pos	1.6000
allergic	neg	-1.2000
amaze	pos	2.5000
amazed	pos	2.2000
amazedly	pos	2.1000
amazement	pos	2.5000
amazements	pos	2.2000
amazes	pos	2.2000
amazing	pos	2.8000
ambitious	pos	2.1000
amor	pos	3.0000
amoral	neg	-1.6000
amoralities	neg	-1.2000
amorality	neg	-1.5000
amorino	pos	1.2000
amorist	pos	1.6000
amoroso	pos	2.3000
amorous	pos	1.8000
amorously	pos	2.3000
amorousness	pos	2.0000
amort	neg	-2.1000
amuse	pos	1.7000
amused	pos	1.8000
amusedly	pos	2.2000
amusement	pos	1.5000
amusements	pos	1.5000
amuser	pos	1.1000
amusers	pos	1.3000
amuses	pos	1.7000
amusing	pos	1.6000
amusingness	pos	1.8000
amusive	pos	1.7000
anger	neg	-2.7000
angered	neg	-2.3000
angering	neg	-2.2000
angerly	neg	-1.9000
angers	neg	-2.3000
angrier	neg	-2.3000
angriest	neg	-3.1000
angrily	neg	-1.8000
angriness	neg	-1.7000
angry	neg	-2.3000
anguish	neg	-2.9000
anguished	neg	-1.8000
anguishes	neg	-2.1000
anguishing	neg	-2.7000
animosity	neg	-1.9000
annoy	neg	-1.9000
annoyance	neg	-1.3000
annoyances	neg	-1.8000
annoyed	neg	-1.6000
annoyer	neg	-2.2000
annoyers	neg	-1.5000
annoying	neg	-1.7000
annoys	neg	-1.8000
antagonism	neg	-1.9000
antagonisms	neg	-1.2000
antagonist	neg	-1.9000
antagonistic	neg	-1.7000
antagonistically	neg	-2.2000
antagonists	neg	-1.7000
antagonize	neg	-2.0000
antagonized	neg	-1.4000
antagonizing	neg	-2.7000
anti	neg	-1.3000
aok	pos	2.0000
apathetic	neg	-1.2000
apathy	neg	-1.2000
apocalyptic	neg	-3.4000
apologise	pos	1.6000
apologized	pos	1.3000
apologizes	pos	1.5000
appall	neg	-2.4000
appalled	neg	-2.0000
appalling	neg	-1.5000
appallingly	neg	-2.0000
appalls	neg	-1.9000
appease	pos	1.1000
applaud	pos	2.0000
applauded	pos	1.5000
applauding	pos	2.1000
applauds	pos	1.4000
applause	pos	1.8000
appreciate	pos	1.7000
appreciated	pos	2.3000
appreciates	pos	2.3000
appreciating	pos	1.9000
appreciation	pos	2.3000
appreciations	pos	1.7000
appreciative	pos	2.6000
appreciatively	pos	1.8000
appreciativeness	pos	1.6000
appreciator	pos	2.6000
appreciators	pos	1.5000
appreciatory	pos	1.7000
apprehensible	pos	1.1000
apprehension	neg	-2.1000
approval	pos	2.1000
approved	pos	1.8000
approves	pos	1.7000
ardent	pos	2.1000
argue	neg	-1.4000
argued	neg	-1.5000
arguer	neg	-1.6000
arguers	neg	-1.4000
argues	neg	-1.6000
arguing	neg	-2.0000
argument	neg	-1.5000
argumentative	neg	-1.5000
argumentatively	neg	-1.8000
argumentive	neg	-1.5000
arguments	neg	-1.7000
arrest	neg	-1.4000
arrested	neg	-2.1000
arrests	neg	-1.9000
arrogance	neg	-2.4000
arrogances	neg	-1.9000
arrogant	neg	-2.2000
arrogantly	neg	-1.8000
ashamed	neg	-2.1000
ashamedly	neg	-1.7000
ass	neg	-2.5000
assassination	neg	-2.9000
assassinations	neg	-2.7000
assault	neg	-2.8000
assaulted	neg	-2.4000
assaulting	neg	-2.3000
assaultive	neg	-2.8000
assaults	neg	-2.5000
asset	pos	1.5000
assfucking	neg	-2.5000
assholes	neg	-2.8000
assurance	pos	1.4000
assurances	pos	1.4000
assure	pos	1.4000
assured	pos	1.5000
assuredly	pos	1.6000
assuredness	pos	1.4000
assurers	pos	1.1000
assures	pos	1.3000
assurgent	pos	1.3000
assuring	pos	1.6000
astonished	pos	1.6000
astound	pos	1.7000
astounded	pos	1.8000
astounding	pos	1.8000
astoundingly	pos	2.1000
astounds	pos	2.1000
attachment	pos	1.2000
attachments	pos	1.1000
attack	neg	-2.1000
attacked	neg	-2.0000
attacker	neg	-2.7000
attackers	neg	-2.7000
attacking	neg	-2.0000
attacks	neg	-1.9000
attract	pos	1.5000
attractant	pos	1.3000
attractants	pos	1.4000
attracted	pos	1.8000
attracting	pos	2.1000
attraction	pos	2.0000
attractions	pos	1.8000
attractive	pos	1.9000
attractively	pos	2.2000
attractiveness	pos	1.8000
attractivenesses	pos	2.1000
attractor	pos	1.2000
attractors	pos	1.2000
attracts	pos	1.7000
aversion	neg	-1.9000
aversions	neg	-1.1000
aversive	neg	-1.6000
avid	pos	1.2000
avoid	neg	-1.2000
avoidance	neg	-1.7000
avoidances	neg	-1.1000
avoided	neg	-1.4000
avoider	neg	-1.8000
avoiders	neg	-1.4000
avoiding	neg	-1.4000
award	pos	2.5000
awardable	pos	2.4000
awarded	pos	1.7000
awardee	pos	1.8000
awardees	pos	1.2000
awarders	pos	1.3000
awarding	pos	1.9000
awards	pos	2.0000
awesome	pos	3.1000
awful	neg	-2.0000
awkwardly	neg	-1.3000
axed	neg	-1.3000
bad	neg	-2.5000
badass	pos	1.4000
badly	neg	-2.1000
bamboozle	neg	-1.5000
bamboozled	neg	-1.5000
bamboozles	neg	-1.5000
ban	neg	-2.6000
banish	neg	-1.9000
bankrupt	neg	-2.6000
bankster	neg	-2.1000
banned	neg	-2.0000
bastard	neg	-2.5000
bastardies	neg	-1.8000
bastardise	neg	-2.1000
bastardised	neg	-2.3000
bastardises	neg	-2.3000
bastardising	neg	-2.6000
bastardization	neg	-2.4000
bastardizations	neg	-2.1000
bastardize	neg	-2.4000
bastardized	neg	-2.0000
bastardizes	neg	-1.8000
bastardizing	neg	-2.3000
bastardly	neg	-2.7000
bastards	neg	-3.0000
bastardy	neg	-2.7000
battle	neg	-1.6000
battled	neg	-1.2000
battlefield	neg	-1.6000
battlefront	neg	-1.2000
battleground	neg	-1.7000
battles	neg	-1.6000
battling	neg	-1.1000
beaten	neg	-1.8000
beatific	pos	1.8000
beating	neg	-2.0000
beaut	pos	1.6000
beauteous	pos	2.5000
beauteously	pos	2.6000
beauteousness	pos	2.7000
beautician	pos	1.2000
beauties	pos	2.4000
beautification	pos	1.9000
beautifications	pos	2.4000
beautified	pos	2.1000
beautifier	pos	1.7000
beautifiers	pos	1.7000
beautifies	pos	1.8000
beautiful	pos	2.9000
beautifuler	pos	2.1000
beautifulest	pos	2.6000
beautifully	pos	2.7000
beautifulness	pos	2.6000
beautify	pos	2.3000
beautifying	pos	2.3000
beauts	pos	1.7000
beauty	pos	2.8000
belittle	neg	-1.9000
belittled	neg	-2.0000
beloved	pos	2.3000
benefic	pos	1.4000
beneficed	pos	1.1000
beneficence	pos	2.8000
beneficences	pos	1.5000
beneficent	pos	2.3000
beneficently	pos	2.2000
benefices	pos	1.1000
beneficial	pos	1.9000
beneficially	pos	2.4000
beneficialness	pos	1.7000
beneficiaries	pos	1.8000
beneficiary	pos	2.1000
benefit	pos	2.0000
benefits	pos	1.6000
benefitted	pos	1.7000
benefitting	pos	1.9000
benevolence	pos	1.7000
benevolences	pos	1.9000
benevolent	pos	2.7000
benevolently	pos	1.4000
benevolentness	pos	1.2000
benign	pos	1.3000
benignant	pos	2.2000
benignantly	pos	1.1000
benignity	pos	1.3000
bereave	neg	-2.1000
bereaved	neg	-2.1000
bereaves	neg	-1.9000
bereaving	neg	-1.3000
best	pos	3.2000
betray	neg	-3.2000
betrayal	neg	-2.8000
betrayed	neg	-3.0000
betraying	neg	-2.5000
betrays	neg	-2.5000
better	pos	1.9000
biased	neg	-1.1000
bitch	neg	-2.8000
bitched	neg	-2.6000
bitcheries	neg	-2.3000
bitchery	neg	-2.7000
bitches	neg	-2.9000
bitchier	neg	-2.0000
bitchiest	neg	-3.0000
bitchily	neg	-2.6000
bitchiness	neg	-2.6000
bitching	neg	-1.1000
bitchy	neg	-2.3000
bitter	neg	-1.8000
bittered	neg	-1.8000
bitterer	neg	-1.9000
bitterest	neg	-2.3000
bittering	neg	-1.2000
bitterish	neg	-1.6000
bitterly	neg	-2.0000
bitterness	neg	-1.7000
bizarre	neg	-1.3000
blamable	neg	-1.8000
blamably	neg	-1.8000
blame	neg	-1.4000
blamed	neg	-2.1000
blameful	neg	-1.7000
blamefully	neg	-1.6000
blamer	neg	-2.1000
blamers	neg	-2.0000
blames	neg	-1.7000
blameworthiness	neg	-1.6000
blameworthy	neg	-2.3000
blaming	neg	-2.2000
bless	pos	1.8000
blessed	pos	2.9000
blesseder	pos	2.0000
blessedest	pos	2.8000
blessedly	pos	1.7000
blessedness	pos	1.6000
blesser	pos	2.6000
blessers	pos	1.9000
blesses	pos	2.6000
blessing	pos	2.2000
blessings	pos	2.5000
blind	neg	-1.7000
bliss	pos	2.7000
blissful	pos	2.9000
blithe	pos	1.2000
block	neg	-1.9000
blockbuster	pos	2.9000
blocked	neg	-1.1000
blocking	neg	-1.6000
bloody	neg	-1.9000
bold	pos	1.6000
bolder	pos	1.2000
boldest	pos	1.6000
boldly	pos	1.5000
boldness	pos	1.5000
bolds	pos	1.3000
bomb	neg	-2.2000
bonus	pos	2.5000
bonuses	pos	2.6000
boost	pos	1.7000
boosted	pos	1.5000
boosting	pos	1.4000
boosts	pos	1.3000
bored	neg	-1.1000
boredom	neg	-1.3000
boredoms	neg	-1.1000
borers	neg	-1.2000
bores	neg	-1.3000
boresome	neg	-1.3000
boring	neg	-1.3000
bother	neg	-1.4000
botheration	neg	-1.7000
botherations	neg	-1.3000
bothered	neg	-1.3000
bothering	neg	-1.6000
bothersome	neg	-1.3000
boycott	neg	-1.3000
boycotted	neg	-1.7000
boycotting	neg	-1.7000
boycotts	neg	-1.4000
brainwashing	neg	-1.5000
brave	pos	2.4000
braved	pos	1.9000
bravely	pos	2.3000
braver	pos	2.4000
braveries	pos	2.0000
bravery	pos	2.2000
braves	pos	1.9000
bravest	pos	2.3000
breathtaking	pos	2.0000
bright	pos	1.9000
brighten	pos	1.9000
brightened	pos	2.1000
brightening	pos	2.5000
brightens	pos	1.5000
brighter	pos	1.6000
brightest	pos	3.0000
brightly	pos	1.5000
brightness	pos	1.6000
brightnesses	pos	1.4000
brightwork	pos	1.1000
brilliance	pos	2.9000
brilliances	pos	2.9000
brilliancies	pos	2.3000
brilliancy	pos	2.6000
brilliant	pos	2.8000
brilliantines	pos	2.0000
brilliantly	pos	3.0000
brilliants	pos	1.9000
broke	neg	-1.8000
broken	neg	-2.1000
brutal	neg	-3.1000
brutalise	neg	-2.7000
brutalised	neg	-2.9000
brutalises	neg	-3.2000
brutalising	neg	-2.8000
brutalities	neg	-2.6000
brutality	neg	-3.0000
brutalization	neg	-2.1000
brutalizations	neg	-2.3000
brutalize	neg	-2.9000
brutalized	neg	-2.4000
brutalizes	neg	-3.2000
brutalizing	neg	-3.4000
brutally	neg	-3.0000
bullied	neg	-3.1000
bullshit	neg	-2.8000
bully	neg	-2.2000
bullying	neg	-2.9000
bummer	neg	-1.6000
burden	neg	-1.9000
burdened	neg	-1.7000
burdener	neg	-1.3000
burdeners	neg	-1.7000
burdening	neg	-1.4000
burdens	neg	-1.5000
burdensome	neg	-1.8000
bwahahah	pos	2.5000
calm	pos	1.3000
calmative	pos	1.1000
calmed	pos	1.6000
calmer	pos	1.5000
calmest	pos	1.6000
calming	pos	1.7000
calmly	pos	1.3000
calmness	pos	1.7000
calmnesses	pos	1.6000
calms	pos	1.3000
can't stand	neg	-2.0000
cancer	neg	-3.4000
capable	pos	1.6000
captivated	pos	1.6000
care	pos	2.2000
cared	pos	1.8000
carefree	pos	1.7000
carefulness	pos	2.0000
careless	neg	-1.5000
carelessness	neg	-1.4000
carelessnesses	neg	-1.6000
cares	pos	2.0000
caring	pos	2.2000
casualty	neg	-2.4000
catastrophe	neg	-3.4000
catastrophic	neg	-2.2000
celebrate	pos	2.7000
celebrated	pos	2.7000
celebrates	pos	2.7000
celebrating	pos	2.7000
censor	neg	-2.0000
censors	neg	-1.2000
certain	pos	1.1000
certainly	pos	1.4000
chagrin	neg	-1.9000
chagrined	neg	-1.4000
champ	pos	2.1000
champagne	pos	1.2000
champion	pos	2.9000
championed	pos	1.2000
championing	pos	1.8000
champions	pos	2.4000
championship	pos	1.9000
championships	pos	2.2000
champs	pos	1.8000
chaos	neg	-2.7000
chaotic	neg	-2.2000
charges	neg	-1.1000
charitable	pos	1.7000
charitableness	pos	1.9000
charitablenesses	pos	1.6000
charitably	pos	1.4000
charities	pos	2.2000
charity	pos	1.8000
charm	pos	1.7000
charmed	pos	2.0000
charmer	pos	1.9000
charmers	pos	2.1000
charming	pos	2.8000
charminger	pos	1.5000
charmingest	pos	2.4000
charmingly	pos	2.2000
charmless	neg	-1.8000
charms	pos	1.9000
chastise	neg	-2.5000
chastised	neg	-2.2000
chastises	neg	-1.7000
chastising	neg	-1.7000
cheat	neg	-2.0000
cheated	neg	-2.3000
cheater	neg	-2.5000
cheaters	neg	-1.9000
cheating	neg	-2.6000
cheats	neg	-1.8000
cheer	pos	2.3000
cheered	pos	2.3000
cheerer	pos	1.7000
cheerers	pos	1.8000
cheerful	pos	2.5000
cheerfuller	pos	1.9000
cheerfullest	pos	3.2000
cheerfully	pos	2.1000
cheerfulness	pos	2.1000
cheerier	pos	2.6000
cheeriest	pos	2.2000
cheerily	pos	2.5000
cheeriness	pos	2.5000
cheering	pos	2.3000
cheerio	pos	1.2000
cheerlead	pos	1.7000
cheerleaders	pos	1.2000
cheerleading	pos	1.2000
cheerleads	pos	1.2000
cheerled	pos	1.5000
cheerless	neg	-1.7000
cheerlessness	neg	-1.7000
cheerly	pos	2.4000
cheers	pos	2.1000
cheery	pos	2.6000
cherish	pos	1.6000
cherishable	pos	2.0000
cherished	pos	2.3000
cherisher	pos	2.2000
cherishers	pos	1.9000
cherishes	pos	2.2000
cherishing	pos	2.0000
chic	pos	1.1000
childish	neg	-1.2000
choke	neg	-2.5000
choked	neg	-2.1000
chokes	neg	-2.0000
choking	neg	-2.0000
chuckle	pos	1.7000
chuckled	pos	1.2000
chucklehead	neg	-1.9000
chuckleheaded	neg	-1.3000
chuckleheads	neg	-1.1000
chucklers	pos	1.2000
chuckles	pos	1.1000
chucklesome	pos	1.1000
chuckling	pos	1.4000
chucklingly	pos	1.2000
clarity	pos	1.7000
classy	pos	1.9000
clean	pos	1.7000
clear	pos	1.6000
clearly	pos	1.7000
clever	pos	2.0000
cleverer	pos	2.0000
cleverest	pos	2.6000
cleverly	pos	2.3000
cleverness	pos	2.3000
clevernesses	pos	1.4000
clueless	neg	-1.5000
cocksucker	neg	-3.1000
cocksuckers	neg	-2.6000
coerced	neg	-1.5000
collapse	neg	-2.2000
collapsed	neg	-1.1000
collapses	neg	-1.2000
collapsing	neg	-1.2000
collides	neg	-1.1000
collision	neg	-1.5000
collisions	neg	-1.1000
colluding	neg	-1.2000
combat	neg	-1.4000
comedian	pos	1.6000
comedians	pos	1.2000
comedic	pos	1.7000
comedically	pos	2.1000
comediennes	pos	1.6000
comedies	pos	1.7000
comedy	pos	1.5000
comfort	pos	1.5000
comfortable	pos	2.3000
comfortableness	pos	1.3000
comfortably	pos	1.8000
comforted	pos	1.8000
comforter	pos	1.9000
comforters	pos	1.2000
comforting	pos	1.7000
comfortingly	pos	1.7000
comfortless	neg	-1.8000
comforts	pos	2.1000
commend	pos	1.9000
commended	pos	1.9000
commit	pos	1.2000
commitment	pos	1.6000
committed	pos	1.1000
compassion	pos	2.0000
compassionate	pos	2.2000
compassionated	pos	1.6000
compassionately	pos	1.7000
compassionates	pos	1.6000
compassionating	pos	1.6000
compassionless	neg	-2.6000
competent	pos	1.3000
complain	neg	-1.5000
complainants	neg	-1.1000
complained	neg	-1.7000
complainer	neg	-1.8000
complainers	neg	-1.3000
complainingly	neg	-1.7000
complains	neg	-1.6000
complaint	neg	-1.2000
complaints	neg	-1.7000
compliment	pos	2.1000
complimentarily	pos	1.7000
complimentary	pos	1.9000
complimented	pos	1.8000
complimenting	pos	2.3000
compliments	pos	1.7000
conciliated	pos	1.1000
conciliates	pos	1.1000
conciliating	pos	1.3000
condemn	neg	-1.6000
condemnation	neg	-2.8000
condemned	neg	-1.9000
condemns	neg	-2.3000
confidence	pos	2.3000
confident	pos	2.2000
confidently	pos	2.1000
conflict	neg	-1.3000
conflicting	neg	-1.7000
conflictive	neg	-1.8000
conflicts	neg	-1.6000
confrontation	neg	-1.3000
confrontational	neg	-1.6000
confrontationists	neg	-1.2000
confrontations	neg	-1.5000
confronters	neg	-1.3000
confused	neg	-1.3000
confusedness	neg	-1.5000
confuses	neg	-1.3000
confusingly	neg	-1.4000
confusion	neg	-1.2000
confusional	neg	-1.2000
congrats	pos	2.4000
congratulate	pos	2.2000
congratulation	pos	2.9000
congratulations	pos	2.9000
considerate	pos	1.9000
consolable	pos	1.1000
conspiracy	neg	-2.4000
contagion	neg	-2.0000
contagions	neg	-1.5000
contagious	neg	-1.4000
contempt	neg	-2.8000
contemptibilities	neg	-2.0000
contemptible	neg	-1.6000
contemptibleness	neg	-1.9000
contemptibly	neg	-1.4000
contemptuous	neg	-2.2000
contemptuously	neg	-2.4000
contemptuousness	neg	-1.1000
contented	pos	1.4000
contentedly	pos	1.9000
contentedness	pos	1.4000
contentious	neg	-1.2000
contentment	pos	1.5000
contradict	neg	-1.3000
contradicted	neg	-1.3000
contradicting	neg	-1.3000
contradictions	neg	-1.3000
contradictious	neg	-1.9000
contradictoriness	neg	-1.4000
contradictors	neg	-1.6000
contradictory	neg	-1.4000
contradicts	neg	-1.4000
controversially	neg	-1.1000
convinced	pos	1.7000
convincing	pos	1.7000
convincingly	pos	1.6000
convivial	pos	1.2000
cool	pos	1.3000
cornered	neg	-1.1000
corpse	neg	-2.7000
courage	pos	2.2000
courageous	pos	2.4000
courageously	pos	2.3000
courageousness	pos	2.1000
courteous	pos	2.3000
courtesy	pos	1.5000
cover-up	neg	-1.2000
coward	neg	-2.0000
cowardly	neg	-1.6000
coziness	pos	1.5000
crap	neg	-1.6000
crappy	neg	-2.6000
crash	neg	-1.7000
crazily	neg	-1.5000
craziness	neg	-1.6000
crazy	neg	-1.4000
create	pos	1.1000
creates	pos	1.1000
creating	pos	1.2000
creation	pos	1.1000
creationisms	pos	1.1000
creations	pos	1.6000
creative	pos	1.9000
creatively	pos	1.5000
creativeness	pos	1.8000
creativities	pos	1.7000
creativity	pos	1.6000
credit	pos	1.6000
creditabilities	pos	1.4000
creditability	pos	1.9000
creditable	pos	1.8000
creditableness	pos	1.2000
creditably	pos	1.7000
credited	pos	1.5000
credits	pos	1.5000
creditworthiness	pos	1.9000
creditworthy	pos	2.4000
crestfallen	neg	-2.5000
cried	neg	-1.6000
cries	neg	-1.7000
crime	neg	-2.5000
criminal	neg	-2.4000
criminals	neg	-2.7000
crisis	neg	-3.1000
critic	neg	-1.1000
critical	neg	-1.3000
criticise	neg	-1.9000
criticised	neg	-1.8000
criticises	neg	-1.3000
criticising	neg	-1.7000
criticism	neg	-1.9000
criticize	neg	-1.6000
criticized	neg	-1.5000
criticizer	neg	-1.5000
criticizers	neg	-1.6000
criticizes	neg	-1.4000
criticizing	neg	-1.5000
critics	neg	-1.2000
crude	neg	-2.7000
crudely	neg	-1.2000
crudeness	neg	-2.0000
crudenesses	neg	-2.0000
cruder	neg	-2.0000
crudes	neg	-1.1000
crudest	neg	-2.4000
cruel	neg	-2.8000
crueler	neg	-2.3000
cruelest	neg	-2.6000
crueller	neg	-2.4000
cruellest	neg	-2.9000
cruelly	neg	-2.8000
cruelness	neg	-2.9000
cruelties	neg	-2.3000
cruelty	neg	-2.9000
crushed	neg	-1.8000
crushes	neg	-1.9000
crushing	neg	-1.5000
cry	neg	-2.1000
crying	neg	-2.1000
cunt	neg	-2.2000
cunts	neg	-2.9000
curious	pos	1.3000
curse	neg	-2.5000
cut	neg	-1.1000
cute	pos	2.0000
cutely	pos	1.3000
cuteness	pos	2.3000
cutenesses	pos	1.9000
cuter	pos	2.3000
cutes	pos	1.8000
cutesier	pos	1.5000
cutesiest	pos	2.2000
cutest	pos	2.8000
cutesy	pos	2.1000
cutey	pos	2.1000
cuteys	pos	1.5000
cutie	pos	1.5000
cutiepie	pos	2.0000
cuties	pos	2.2000
cuts	neg	-1.2000
cynic	neg	-1.4000
cynical	neg	-1.6000
cynically	neg	-1.3000
cynicism	neg	-1.7000
cynicisms	neg	-1.7000
d-:	pos	1.6000
d:	pos	1.2000
d=	pos	1.5000
damage	neg	-2.2000
damaged	neg	-1.9000
damager	neg	-1.9000
damagers	neg	-2.0000
damages	neg	-1.9000
damaging	neg	-2.3000
damagingly	neg	-2.0000
damn	neg	-1.7000
damnable	neg	-1.7000
damnableness	neg	-1.8000
damnably	neg	-1.7000
damnation	neg	-2.6000
damnations	neg	-1.4000
damnatory	neg	-2.6000
damned	neg	-1.6000
damnified	neg	-2.8000
damnifies	neg	-1.8000
damnify	neg	-2.2000
damnifying	neg	-2.4000
damning	neg	-1.4000
damningly	neg	-2.0000
damnit	neg	-2.4000
damns	neg	-2.2000
danger	neg	-2.4000
dangered	neg	-2.4000
dangering	neg	-2.5000
dangerous	neg	-2.1000
dangerously	neg	-2.0000
dangerousness	neg	-2.0000
dangers	neg	-2.2000
daring	pos	1.5000
daringly	pos	2.1000
daringness	pos	1.4000
darkest	neg	-2.2000
darling	pos	2.8000
darlingly	pos	1.6000
darlingness	pos	2.3000
darlings	pos	2.2000
dauntless	pos	2.3000
dead	neg	-3.3000
deadlock	neg	-1.4000
deafening	neg	-1.2000
dear	pos	1.6000
dearer	pos	1.9000
dearest	pos	2.6000
dearie	pos	2.2000
dearly	pos	1.8000
dearness	pos	2.0000
dears	pos	1.9000
dearth	neg	-2.3000
deary	pos	1.9000
death	neg	-2.9000
debt	neg	-1.5000
decay	neg	-1.7000
decayed	neg	-1.6000
decayer	neg	-1.6000
decayers	neg	-1.6000
decaying	neg	-1.7000
decays	neg	-1.7000
deceit	neg	-2.0000
deceitful	neg	-1.9000
deceive	neg	-1.7000
deceived	neg	-1.9000
deceives	neg	-1.6000
deceiving	neg	-1.4000
deception	neg	-1.9000
dedicated	pos	2.0000
defeat	neg	-2.0000
defeated	neg	-2.1000
defeater	neg	-1.4000
defeating	neg	-1.6000
defeatism	neg	-1.3000
defeatist	neg	-1.7000
defeatists	neg	-2.1000
defeats	neg	-1.3000
defeature	neg	-1.9000
defeatures	neg	-1.5000
defect	neg	-1.4000
defected	neg	-1.7000
defecting	neg	-1.8000
defection	neg	-1.4000
defections	neg	-1.5000
defective	neg	-1.9000
defectively	neg	-2.1000
defectiveness	neg	-1.8000
defectives	neg	-1.8000
defector	neg	-1.9000
defectors	neg	-1.3000
defects	neg	-1.7000
defenseless	neg	-1.4000
defenselessly	neg	-1.1000
defenselessness	neg	-1.3000
defer	neg	-1.2000
deficit	neg	-1.7000
definite	pos	1.1000
definitely	pos	1.7000
degradation	neg	-2.4000
degradations	neg	-1.5000
degradative	neg	-2.0000
degrade	neg	-1.9000
degraded	neg	-1.8000
degrader	neg	-2.0000
degraders	neg	-2.0000
degrades	neg	-2.1000
degrading	neg	-2.8000
degradingly	neg	-2.7000
dehumanize	neg	-1.8000
dehumanized	neg	-1.9000
dehumanizes	neg	-1.5000
dehumanizing	neg	-2.4000
deject	neg	-2.2000
dejected	neg	-2.2000
dejecting	neg	-2.3000
dejects	neg	-2.0000
delay	neg	-1.3000
delectable	pos	2.9000
delectables	pos	1.4000
delectably	pos	2.8000
delicious	pos	2.7000
deliciously	pos	1.9000
deliciousness	pos	1.8000
delight	pos	2.9000
delighted	pos	2.3000
delightedly	pos	2.4000
delightedness	pos	2.1000
delighter	pos	2.0000
delighters	pos	2.6000
delightful	pos	2.8000
delightfully	pos	2.7000
delightfulness	pos	2.1000
delighting	pos	1.6000
delights	pos	2.0000
delightsome	pos	2.3000
demoralized	neg	-1.6000
denied	neg	-1.9000
denier	neg	-1.5000
deniers	neg	-1.1000
denies	neg	-1.8000
denounce	neg	-1.4000
denounces	neg	-1.9000
deny	neg	-1.4000
denying	neg	-1.4000
depress	neg	-2.2000
depressant	neg	-1.6000
depressants	neg	-1.6000
depressed	neg	-2.3000
depresses	neg	-2.2000
depressible	neg	-1.7000
depressing	neg	-1.6000
depressingly	neg	-2.3000
depression	neg	-2.7000
depressions	neg	-2.2000
depressive	neg	-1.6000
depressively	neg	-2.1000
depressives	neg	-1.5000
depressor	neg	-1.8000
depressors	neg	-1.7000
deprival	neg	-2.1000
deprivals	neg	-1.2000
deprivation	neg	-1.8000
deprivations	neg	-1.8000
deprive	neg	-2.1000
deprived	neg	-2.1000
depriver	neg	-1.6000
deprivers	neg	-1.4000
deprives	neg	-1.7000
depriving	neg	-2.0000
derail	neg	-1.2000
derailed	neg	-1.4000
derails	neg	-1.3000
deride	neg	-1.1000
deriding	neg	-1.5000
derision	neg	-1.2000
desirable	pos	1.3000
desire	pos	1.7000
desired	pos	1.1000
desirous	pos	1.3000
despair	neg	-1.3000
despaired	neg	-2.7000
despairer	neg	-1.3000
despairers	neg	-1.3000
despairing	neg	-2.3000
despairingly	neg	-2.2000
despairs	neg	-2.7000
desperate	neg	-1.3000
desperately	neg	-1.6000
desperateness	neg	-1.5000
desperation	neg	-2.0000
desperations	neg	-2.2000
despise	neg	-1.4000
despised	neg	-1.7000
despisement	neg	-2.4000
despisements	neg	-2.5000
despiser	neg	-1.8000
despisers	neg	-1.6000
despises	neg	-2.0000
despising	neg	-2.7000
despondent	neg	-2.1000
destroy	neg	-2.5000
destroyed	neg	-2.2000
destroyer	neg	-2.0000
destroyers	neg	-2.3000
destroying	neg	-2.6000
destroys	neg	-2.6000
destruct	neg	-2.4000
destructed	neg	-1.9000
destructibility	neg	-1.8000
destructible	neg	-1.5000
destructing	neg	-2.5000
destruction	neg	-2.7000
destructionist	neg	-2.6000
destructionists	neg	-2.1000
destructions	neg	-2.3000
destructive	neg	-3.0000
destructively	neg	-2.4000
destructiveness	neg	-2.4000
destructivity	neg	-2.2000
destructs	neg	-2.4000
detain	neg	-1.8000
detained	neg	-1.7000
detention	neg	-1.5000
determinately	pos	1.2000
determinateness	pos	1.1000
determination	pos	1.7000
determinative	pos	1.1000
determinator	pos	1.1000
determined	pos	1.4000
devastate	neg	-3.1000
devastated	neg	-3.0000
devastates	neg	-2.8000
devastating	neg	-3.3000
devastatingly	neg	-2.4000
devastation	neg	-1.8000
devastations	neg	-1.9000
devastative	neg	-3.2000
devastator	neg	-2.8000
devastators	neg	-2.9000
devil	neg	-3.4000
deviled	neg	-1.6000
deviling	neg	-2.2000
devilish	neg	-2.1000
devilishly	neg	-1.6000
devilishness	neg	-2.3000
devilkin	neg	-2.4000
devilled	neg	-2.3000
devilling	neg	-1.8000
devilment	neg	-1.9000
devilments	neg	-1.1000
devilries	neg	-1.6000
devilry	neg	-2.8000
devils	neg	-2.7000
deviltries	neg	-1.5000
deviltry	neg	-2.8000
devote	pos	1.4000
devoted	pos	1.7000
devotedly	pos	1.6000
devotedness	pos	2.0000
devotee	pos	1.6000
devotement	pos	1.5000
devotements	pos	1.1000
devotes	pos	1.6000
devoting	pos	2.1000
devotion	pos	2.0000
devotional	pos	1.2000
devotionally	pos	2.2000
devotionals	pos	1.2000
devotions	pos	1.8000
diamond	pos	1.4000
dick	neg	-2.3000
dickhead	neg	-3.1000
die	neg	-2.9000
died	neg	-2.6000
difficult	neg	-1.5000
difficulties	neg	-1.2000
difficultly	neg	-1.7000
difficulty	neg	-1.4000
dignified	pos	2.2000
dignifies	pos	2.0000
dignify	pos	1.8000
dignifying	pos	2.1000
dignitary	pos	1.9000
dignities	pos	1.4000
dignity	pos	1.7000
dipshit	neg	-2.1000
dire	neg	-2.0000
direful	neg	-3.1000
dirt	neg	-1.4000
dirtier	neg	-1.4000
dirtiest	neg	-2.4000
dirty	neg	-1.9000
disabling	neg	-2.1000
disadvantage	neg	-1.8000
disadvantaged	neg	-1.7000
disadvantageous	neg	-1.8000
disadvantageously	neg	-2.1000
disadvantageousness	neg	-1.6000
disadvantages	neg	-1.7000
disagree	neg	-1.6000
disagreeable	neg	-1.7000
disagreeableness	neg	-1.7000
disagreeablenesses	neg	-1.9000
disagreeably	neg	-1.5000
disagreed	neg	-1.3000
disagreeing	neg	-1.4000
disagreement	neg	-1.5000
disagreements	neg	-1.8000
disagrees	neg	-1.3000
disappears	neg	-1.4000
disappoint	neg	-1.7000
disappointed	neg	-2.1000
disappointedly	neg	-1.7000
disappointing	neg	-2.2000
disappointingly	neg	-1.9000
disappointment	neg	-2.3000
disappointments	neg	-2.0000
disappoints	neg	-1.6000
disaster	neg	-3.1000
disasters	neg	-2.6000
disastrous	neg	-2.9000
disbelieve	neg	-1.2000
discarded	neg	-1.4000
discomfort	neg	-1.8000
discomfortable	neg	-1.6000
discomforted	neg	-1.6000
discomforting	neg	-1.6000
discomforts	neg	-1.3000
disconsolate	neg	-2.3000
disconsolation	neg	-1.7000
discontented	neg	-1.8000
discord	neg	-1.7000
discourage	neg	-1.8000
discourageable	neg	-1.2000
discouraged	neg	-1.7000
discouragement	neg	-2.0000
discouragements	neg	-1.8000
discourager	neg	-1.7000
discouragers	neg	-1.9000
discourages	neg	-1.9000
discouraging	neg	-1.9000
discouragingly	neg	-1.8000
discredited	neg	-1.9000
disdain	neg	-2.1000
disgrace	neg	-2.2000
disgraced	neg	-2.0000
disguised	neg	-1.1000
disguising	neg	-1.3000
disgust	neg	-2.9000
disgusted	neg	-2.4000
disgustedly	neg	-3.0000
disgustful	neg	-2.6000
disgusting	neg	-2.4000
disgustingly	neg	-2.9000
disgusts	neg	-2.1000
dishearten	neg	-2.0000
disheartened	neg	-2.2000
disheartening	neg	-1.8000
dishearteningly	neg	-2.0000
disheartenment	neg	-2.3000
disheartenments	neg	-2.2000
disheartens	neg	-2.2000
dishonest	neg	-2.7000
disillusioned	neg	-1.9000
disillusioning	neg	-1.3000
disillusionment	neg	-1.7000
disillusionments	neg	-1.5000
disillusions	neg	-1.6000
disinclined	neg	-1.1000
disjointed	neg	-1.3000
dislike	neg	-1.6000
disliked	neg	-1.7000
dislikes	neg	-1.7000
disliking	neg	-1.3000
dismal	neg	-3.0000
dismay	neg	-1.8000
dismayed	neg	-1.9000
dismaying	neg	-2.2000
dismayingly	neg	-1.9000
dismays	neg	-1.8000
disorder	neg	-1.7000
disorganized	neg	-1.2000
disoriented	neg	-1.5000
disparage	neg	-2.0000
disparaged	neg	-1.4000
disparages	neg	-1.6000
disparaging	neg	-2.2000
displeased	neg	-1.9000
dispute	neg	-1.7000
disputed	neg	-1.4000
disputes	neg	-1.1000
disputing	neg	-1.7000
disqualified	neg	-1.8000
disquiet	neg	-1.3000
disregard	neg	-1.1000
disregarded	neg	-1.6000
disregards	neg	-1.4000
disrespect	neg	-1.8000
disrespected	neg	-2.0000
disruption	neg	-1.5000
disruptions	neg	-1.4000
disruptive	neg	-1.3000
dissatisfaction	neg	-2.2000
dissatisfactions	neg	-1.9000
dissatisfactory	neg	-2.0000
dissatisfied	neg	-1.6000
dissatisfies	neg	-1.8000
dissatisfy	neg	-2.2000
dissatisfying	neg	-2.4000
distort	neg	-1.3000
distorted	neg	-1.7000
distorting	neg	-1.1000
distorts	neg	-1.4000
distract	neg	-1.2000
distractable	neg	-1.3000
distracted	neg	-1.4000
distractibility	neg	-1.3000
distractible	neg	-1.5000
distracting	neg	-1.2000
distractingly	neg	-1.4000
distraction	neg	-1.6000
distractive	neg	-1.6000
distracts	neg	-1.3000
distraught	neg	-2.6000
distress	neg	-2.4000
distressed	neg	-1.8000
distresses	neg	-1.6000
distressful	neg	-2.2000
distressfully	neg	-1.7000
distressfulness	neg	-2.4000
distressing	neg	-1.7000
distressingly	neg	-2.2000
distrust	neg	-1.8000
distrusted	neg	-2.4000
distrustful	neg	-2.1000
distrustfully	neg	-1.8000
distrustfulness	neg	-1.6000
distrusting	neg	-2.1000
distrusts	neg	-1.3000
disturb	neg	-1.7000
disturbance	neg	-1.6000
disturbances	neg	-1.4000
disturbed	neg	-1.6000
disturber	neg	-1.4000
disturbers	neg	-2.1000
disturbing	neg	-2.3000
disturbingly	neg	-2.3000
disturbs	neg	-1.9000
divination	pos	1.7000
divinations	pos	1.1000
divinatory	pos	1.6000
divine	pos	2.6000
divinely	pos	2.9000
diviners	pos	1.2000
divinest	pos	2.7000
divinities	pos	1.8000
divinity	pos	2.7000
divinize	pos	2.3000
dolorous	neg	-2.2000
dominating	neg	-1.2000
doom	neg	-1.7000
doomed	neg	-3.2000
doomful	neg	-2.1000
dooming	neg	-2.8000
dooms	neg	-1.1000
doomsayers	neg	-1.7000
doomsaying	neg	-1.5000
doomsayings	neg	-1.5000
doomsday	neg	-2.8000
doomsdayer	neg	-2.2000
doomsdays	neg	-2.4000
doomster	neg	-2.2000
doomsters	neg	-1.6000
doomy	neg	-1.1000
dork	neg	-1.4000
dorkier	neg	-1.1000
dorkiest	neg	-1.2000
dorky	neg	-1.1000
doubt	neg	-1.5000
doubtable	neg	-1.5000
doubted	neg	-1.1000
doubter	neg	-1.6000
doubters	neg	-1.3000
doubtful	neg	-1.4000
doubtfully	neg	-1.2000
doubtfulness	neg	-1.2000
doubting	neg	-1.4000
doubtingly	neg	-1.4000
doubtlessly	pos	1.2000
doubts	neg	-1.2000
douche	neg	-1.5000
douchebag	neg	-3.0000
downcast	neg	-1.8000
downhearted	neg	-2.3000
drained	neg	-1.5000
dread	neg	-2.0000
dreaded	neg	-2.7000
dreadful	neg	-1.9000
dreadfully	neg	-2.7000
dreadfulness	neg	-3.2000
dreadfuls	neg	-2.4000
dreading	neg	-2.4000
dreads	neg	-1.4000
dreams	pos	1.7000
dreary	neg	-1.4000
drop	neg	-1.1000
drown	neg	-2.7000
drowned	neg	-2.9000
drowns	neg	-2.2000
drunk	neg	-1.4000
dubious	neg	-1.5000
dull	neg	-1.7000
dullard	neg	-1.6000
dullards	neg	-1.8000
dulled	neg	-1.5000
duller	neg	-1.7000
dullest	neg	-1.7000
dulling	neg	-1.1000
dullish	neg	-1.1000
dullness	neg	-1.4000
dullnesses	neg	-1.9000
dullsville	neg	-2.4000
dully	neg	-1.1000
dumb	neg	-2.3000
dumbass	neg	-2.6000
dumbed	neg	-1.4000
dumber	neg	-1.5000
dumbest	neg	-2.3000
dumbfounded	neg	-1.6000
dumbhead	neg	-2.6000
dumbheads	neg	-1.9000
dumbly	neg	-1.3000
dumbness	neg	-1.9000
dumbs	neg	-1.5000
dump	neg	-1.6000
dumped	neg	-1.7000
dumper	neg	-1.2000
dumpier	neg	-1.4000
dumpiest	neg	-1.6000
dumpiness	neg	-1.2000
dumping	neg	-1.3000
dumpings	neg	-1.1000
dumpish	neg	-1.8000
dumps	neg	-1.7000
dumpy	neg	-1.7000
dupe	neg	-1.5000
duped	neg	-1.8000
dynamic	pos	1.6000
dynamical	pos	1.2000
dynamically	pos	1.5000
dynamics	pos	1.1000
dynamism	pos	1.6000
dynamisms	pos	1.2000
dynamist	pos	1.4000
dynamistic	pos	1.5000
dynamiter	neg	-1.2000
dysfunction	neg	-1.8000
eager	pos	1.5000
eagerly	pos	1.6000
eagerness	pos	1.7000
eagers	pos	1.6000
earnest	pos	2.3000
ease	pos	1.5000
eased	pos	1.2000
easeful	pos	1.5000
easefully	pos	1.4000
easement	pos	1.6000
eases	pos	1.3000
easier	pos	1.8000
easiest	pos	1.8000
easily	pos	1.4000
easiness	pos	1.6000
easy	pos	1.9000
easygoing	pos	1.3000
easygoingness	pos	1.5000
ecstacy	pos	3.3000
ecstasies	pos	2.3000
ecstasy	pos	2.9000
ecstatic	pos	2.3000
ecstatically	pos	2.8000
ecstatics	pos	2.9000
eerie	neg	-1.5000
effective	pos	2.1000
effectively	pos	1.9000
efficiencies	pos	1.6000
efficiency	pos	1.5000
efficient	pos	1.8000
efficiently	pos	1.7000
effin	neg	-2.3000
egotism	neg	-1.4000
egotist	neg	-2.3000
egotistic	neg	-1.4000
egotistically	neg	-1.8000
egotists	neg	-1.7000
elated	pos	3.2000
elation	pos	1.5000
elegance	pos	2.1000
elegances	pos	1.8000
elegancies	pos	1.6000
elegancy	pos	2.1000
elegant	pos	2.1000
elegantly	pos	1.9000
embarrass	neg	-1.2000
embarrassable	neg	-1.6000
embarrassed	neg	-1.5000
embarrassedly	neg	-1.1000
embarrasses	neg	-1.7000
embarrassing	neg	-1.6000
embarrassingly	neg	-1.7000
embarrassment	neg	-1.9000
embarrassments	neg	-1.7000
embrace	pos	1.3000
emergency	neg	-1.6000
empathetic	pos	1.7000
emptiest	neg	-1.8000
emptiness	neg	-1.9000
emptinesses	neg	-1.5000
enchanted	pos	1.6000
encourage	pos	2.3000
encouraged	pos	1.5000
encouragement	pos	1.8000
encouragements	pos	2.1000
encourager	pos	1.5000
encouragers	pos	1.5000
encourages	pos	1.9000
encouraging	pos	2.4000
encouragingly	pos	2.0000
endorse	pos	1.3000
endorsement	pos	1.3000
endorses	pos	1.4000
enemies	neg	-2.2000
enemy	neg	-2.5000
energetic	pos	1.9000
energetically	pos	1.8000
energise	pos	2.2000
energised	pos	2.1000
energises	pos	2.2000
energising	pos	1.9000
energization	pos	1.6000
energizations	pos	1.5000
energize	pos	2.1000
energized	pos	2.3000
energizer	pos	2.1000
energizers	pos	1.7000
energizes	pos	2.1000
energizing	pos	2.0000
energy	pos	1.1000
engage	pos	1.4000
engaged	pos	1.7000
engagement	pos	2.0000
engager	pos	1.1000
engaging	pos	1.4000
engagingly	pos	1.5000
enjoy	pos	2.2000
enjoyable	pos	1.9000
enjoyableness	pos	1.9000
enjoyably	pos	1.8000
enjoyed	pos	2.3000
enjoyer	pos	2.2000
enjoyers	pos	2.2000
enjoying	pos	2.4000
enjoyment	pos	2.6000
enjoyments	pos	2.0000
enjoys	pos	2.3000
enlighten	pos	2.3000
enlightened	pos	2.2000
enlightening	pos	2.3000
enlightens	pos	1.7000
ennui	neg	-1.2000
enrage	neg	-2.6000
enraged	neg	-1.7000
enrages	neg	-1.8000
enraging	neg	-2.8000
enrapture	pos	3.0000
enslave	neg	-3.1000
enslaved	neg	-1.7000
enslaves	neg	-1.6000
ensure	pos	1.6000
ensuring	pos	1.1000
enterprising	pos	2.3000
entertain	pos	1.3000
entertained	pos	1.7000
entertainer	pos	1.6000
entertaining	pos	1.9000
entertainingly	pos	1.9000
entertainment	pos	1.8000
entertainments	pos	2.3000
entertains	pos	2.4000
enthuse	pos	1.6000
enthused	pos	2.0000
enthuses	pos	1.7000
enthusiasm	pos	1.9000
enthusiasms	pos	2.0000
enthusiast	pos	1.5000
enthusiastic	pos	2.2000
enthusiastically	pos	2.6000
enthusiasts	pos	1.4000
enthusing	pos	1.9000
entitled	pos	1.1000
envied	neg	-1.1000
enviers	neg	-1.1000
envious	neg	-1.1000
envy	neg	-1.1000
envyingly	neg	-1.3000
erroneous	neg	-1.8000
error	neg	-1.7000
errors	neg	-1.4000
esteemed	pos	1.9000
ethical	pos	2.3000
euphoria	pos	3.3000
euphoric	pos	3.2000
eviction	neg	-2.0000
evil	neg	-3.4000
evildoer	neg	-3.1000
evildoers	neg	-2.4000
evildoing	neg	-3.1000
evildoings	neg	-2.5000
eviler	neg	-2.1000
evilest	neg	-2.5000
eviller	neg	-2.9000
evillest	neg	-3.3000
evilly	neg	-3.4000
evilness	neg	-3.1000
evils	neg	-2.7000
exasperated	neg	-1.8000
excel	pos	2.0000
excelled	pos	2.2000
excellence	pos	3.1000
excellences	pos	2.5000
excellencies	pos	2.4000
excellency	pos	2.5000
excellent	pos	2.7000
excellently	pos	3.1000
excelling	pos	2.5000
excels	pos	2.5000
excitabilities	pos	1.5000
excitability	pos	1.2000
excitable	pos	1.5000
excitant	pos	1.8000
excitants	pos	1.2000
excitation	pos	1.8000
excitations	pos	1.8000
excitatory	pos	1.1000
excite	pos	2.1000
excited	pos	1.4000
excitedly	pos	2.3000
excitement	pos	2.2000
excitements	pos	1.9000
exciter	pos	1.9000
exciters	pos	1.4000
excites	pos	2.1000
exciting	pos	2.2000
excitingly	pos	1.9000
excluded	neg	-1.4000
exclusion	neg	-1.2000
excruciate	neg	-2.7000
excruciated	neg	-1.3000
excruciating	neg	-3.3000
excruciatingly	neg	-2.9000
excruciation	neg	-3.4000
excruciations	neg	-1.9000
exhaust	neg	-1.2000
exhausted	neg	-1.5000
exhauster	neg	-1.3000
exhausters	neg	-1.3000
exhausting	neg	-1.5000
exhaustion	neg	-1.5000
exhaustions	neg	-1.1000
exhaustiveness	neg	-1.1000
exhausts	neg	-1.1000
exhilarated	pos	3.0000
exhilarates	pos	2.8000
exhilarating	pos	1.7000
exonerate	pos	1.8000
exonerated	pos	1.8000
exonerates	pos	1.6000
expand	pos	1.3000
expel	neg	-1.9000
expelling	neg	-1.6000
expels	neg	-1.6000
exploited	neg	-2.0000
exploiting	neg	-1.9000
exploits	neg	-1.4000
exposing	neg	-1.1000
exuberant	pos	2.8000
exultant	pos	3.0000
exultantly	pos	1.4000
fab	pos	2.0000
fabulous	pos	2.4000
fabulousness	pos	2.8000
fag	neg	-2.1000
faggot	neg	-3.4000
faggots	neg	-3.2000
fail	neg	-2.5000
failed	neg	-2.3000
failing	neg	-2.3000
failingly	neg	-1.4000
failings	neg	-2.2000
fails	neg	-1.8000
failure	neg	-2.3000
failures	neg	-2.0000
fair	pos	1.3000
faith	pos	1.8000
faithed	pos	1.3000
faithful	pos	1.9000
faithfully	pos	1.8000
faithfulness	pos	1.9000
faithlessness	neg	-1.8000
faiths	pos	1.8000
fake	neg	-2.1000
fakes	neg	-1.8000
faking	neg	-1.8000
fallen	neg	-1.5000
falsified	neg	-1.6000
falsify	neg	-2.0000
fame	pos	1.9000
fan	pos	1.3000
fantastic	pos	2.6000
fantastical	pos	2.0000
fantasticalities	pos	2.1000
fantasticality	pos	1.7000
fantasticalness	pos	1.3000
fantasticate	pos	1.5000
farce	neg	-1.7000
fascinate	pos	2.4000
fascinated	pos	2.1000
fascinates	pos	2.0000
fascination	pos	2.2000
fascinating	pos	2.5000
fascist	neg	-2.6000
fatal	neg	-2.5000
fatalisms	neg	-1.7000
fatalists	neg	-1.2000
fatalities	neg	-2.9000
fatality	neg	-3.5000
fatally	neg	-3.2000
fatigued	neg	-1.4000
fatigues	neg	-1.3000
fatiguing	neg	-1.2000
fatiguingly	neg	-1.5000
fault	neg	-1.7000
faulted	neg	-1.4000
faultfinders	neg	-1.5000
faultfinding	neg	-2.1000
faultier	neg	-2.1000
faultiest	neg	-2.1000
faultily	neg	-2.0000
faultiness	neg	-1.5000
faulting	neg	-1.4000
faultless	pos	2.0000
faultlessly	pos	2.0000
faultlessness	pos	1.1000
faults	neg	-2.1000
faulty	neg	-1.3000
fav	pos	2.0000
fave	pos	1.9000
favor	pos	1.7000
favorable	pos	2.1000
favorableness	pos	2.2000
favorably	pos	1.6000
favored	pos	1.8000
favorer	pos	1.3000
favorers	pos	1.4000
favoring	pos	1.8000
favorite	pos	2.0000
favorited	pos	1.7000
favorites	pos	1.8000
favour	pos	1.9000
favoured	pos	1.8000
favourer	pos	1.6000
favourers	pos	1.6000
favouring	pos	1.3000
favours	pos	1.8000
fear	neg	-2.2000
feared	neg	-2.2000
fearful	neg	-2.2000
fearfuller	neg	-2.2000
fearfullest	neg	-2.5000
fearfully	neg	-2.2000
fearfulness	neg	-1.8000
fearing	neg	-2.7000
fearless	pos	1.9000
fearlessly	pos	1.1000
fearlessness	pos	1.1000
fears	neg	-1.8000
fearsome	neg	-1.7000
fed up	neg	-1.8000
feeble	neg	-1.2000
felonies	neg	-2.5000
felony	neg	-2.5000
ferociously	neg	-1.1000
fervent	pos	1.1000
festival	pos	2.2000
festivalgoer	pos	1.3000
festivalgoers	pos	1.2000
festivals	pos	1.5000
festive	pos	2.0000
festively	pos	2.2000
festiveness	pos	2.4000
festivities	pos	2.1000
festivity	pos	2.2000
feud	neg	-1.4000
feudalistic	neg	-1.1000
feuded	neg	-2.2000
feuding	neg	-1.6000
feudist	neg	-1.1000
feuds	neg	-1.4000
fiasco	neg	-2.3000
fidgety	neg	-1.4000
fiery	neg	-1.4000
fiesta	pos	2.1000
fiestas	pos	1.5000
fight	neg	-1.6000
fighting	neg	-1.5000
fightings	neg	-1.9000
fights	neg	-1.7000
fire	neg	-1.4000
fired	neg	-2.6000
firing	neg	-1.4000
fit	pos	1.5000
fitness	pos	1.1000
flattered	pos	1.6000
flatteries	pos	1.2000
flattering	pos	1.3000
flawed	neg	-2.1000
flawless	pos	2.3000
flexibility	pos	1.4000
flexibly	pos	1.3000
flirtation	pos	1.7000
flop	neg	-1.4000
flops	neg	-1.4000
flu	neg	-1.6000
flunk	neg	-1.3000
flunked	neg	-2.1000
flunker	neg	-1.9000
flunkers	neg	-1.6000
flunkey	neg	-1.8000
flunkies	neg	-1.4000
flunking	neg	-1.5000
flunks	neg	-1.8000
flunky	neg	-1.8000
focused	pos	1.6000
foe	neg	-1.9000
foeman	neg	-1.8000
foes	neg	-2.0000
foetid	neg	-2.3000
foetor	neg	-3.0000
foetors	neg	-2.1000
fond	pos	1.9000
fondly	pos	1.9000
fondness	pos	2.5000
fool	neg	-1.9000
fooled	neg	-1.6000
fooleries	neg	-1.8000
foolery	neg	-1.8000
foolhardier	neg	-1.5000
foolhardiest	neg	-1.3000
foolhardiness	neg	-1.6000
foolhardy	neg	-1.4000
fooling	neg	-1.7000
foolish	neg	-1.1000
foolisher	neg	-1.7000
foolishest	neg	-1.4000
foolishly	neg	-1.8000
foolishness	neg	-1.8000
foolishnesses	neg	-2.0000
foolproof	pos	1.6000
fools	neg	-2.2000
forbid	neg	-1.3000
forbiddance	neg	-1.4000
forbidden	neg	-1.8000
forbidder	neg	-1.6000
forbidders	neg	-1.5000
forbidding	neg	-1.9000
forbiddingly	neg	-1.9000
forbids	neg	-1.3000
forced	neg	-2.0000
foreclosures	neg	-2.4000
forgave	pos	1.4000
forgetful	neg	-1.1000
forgivable	pos	1.7000
forgivably	pos	1.6000
forgive	pos	1.1000
forgiven	pos	1.6000
forgiveness	pos	1.1000
forgiver	pos	1.7000
forgivers	pos	1.2000
forgives	pos	1.7000
forgiving	pos	1.9000
forgivingly	pos	1.4000
forgivingness	pos	1.8000
fortunate	pos	1.9000
fought	neg	-1.3000
foughten	neg	-1.9000
frantic	neg	-1.9000
frantically	neg	-1.4000
fraud	neg	-2.8000
frauds	neg	-2.3000
fraudster	neg	-2.5000
fraudsters	neg	-2.4000
fraudulence	neg	-2.3000
fraudulent	neg	-2.2000
freak	neg	-1.9000
freaked	neg	-1.2000
freakier	neg	-1.3000
freakiest	neg	-1.6000
freakiness	neg	-1.4000
freaking	neg	-1.8000
freakish	neg	-2.1000
freakishness	neg	-1.4000
freakout	neg	-1.8000
freakouts	neg	-1.5000
freaky	neg	-1.5000
free	pos	2.3000
freebee	pos	1.3000
freebees	pos	1.3000
freebie	pos	1.8000
freebies	pos	1.8000
freebooter	neg	-1.7000
freeborn	pos	1.2000
freed	pos	1.7000
freedman	pos	1.1000
freedom	pos	3.2000
freedoms	pos	1.2000
freedwoman	pos	1.6000
freedwomen	pos	1.3000
freehanded	pos	1.4000
freehearted	pos	1.5000
freeing	pos	2.1000
freelance	pos	1.2000
freelancer	pos	1.1000
freeload	neg	-1.9000
freeloaded	neg	-1.6000
freeloading	neg	-1.3000
freeloads	neg	-1.3000
freely	pos	1.9000
freeman	pos	1.7000
freemen	pos	1.5000
freeness	pos	1.6000
freenesses	pos	1.7000
freer	pos	1.1000
frees	pos	1.2000
freest	pos	1.6000
freestanding	pos	1.1000
freethinking	pos	1.1000
freezingly	neg	-1.6000
frenzy	neg	-1.3000
fresh	pos	1.3000
friend	pos	2.2000
friended	pos	1.7000
friending	pos	1.8000
friendless	neg	-1.5000
friendlier	pos	2.0000
friendlies	pos	2.2000
friendliest	pos	2.6000
friendlily	pos	1.8000
friendliness	pos	2.0000
friendly	pos	2.2000
friends	pos	2.1000
friendship	pos	1.9000
friendships	pos	1.6000
fright	neg	-1.6000
frighted	neg	-1.4000
frighten	neg	-1.4000
frightened	neg	-1.9000
frightening	neg	-2.2000
frighteningly	neg	-2.1000
frightens	neg	-1.7000
frightful	neg	-2.3000
frightfully	neg	-1.7000
frightfulness	neg	-1.9000
frighting	neg	-1.5000
frights	neg	-1.1000
frowning	neg	-1.4000
frustrate	neg	-2.0000
frustrated	neg	-2.4000
frustrates	neg	-1.9000
frustrating	neg	-1.9000
frustratingly	neg	-2.0000
frustration	neg	-2.1000
frustrations	neg	-2.0000
fuck	neg	-2.5000
fucked	neg	-3.4000
fucker	neg	-3.3000
fuckers	neg	-2.9000
fuckface	neg	-3.2000
fuckhead	neg	-3.1000
fucks	neg	-2.1000
fucktard	neg	-3.1000
fud	neg	-1.1000
fuked	neg	-2.5000
fuking	neg	-3.2000
fulfill	pos	1.9000
fulfilled	pos	1.8000
fume	neg	-1.2000
fumed	neg	-1.8000
fuming	neg	-2.7000
fun	pos	2.3000
funeral	neg	-1.5000
funerals	neg	-1.6000
funned	pos	2.3000
funner	pos	2.2000
funnest	pos	2.9000
funnier	pos	1.7000
funnies	pos	1.3000
funniest	pos	2.6000
funnily	pos	1.9000
funniness	pos	1.8000
funninesses	pos	1.6000
funning	pos	1.8000
funny	pos	1.9000
funnyman	pos	1.4000
funnymen	pos	1.3000
furious	neg	-2.7000
furiously	neg	-1.9000
fury	neg	-2.7000
futile	neg	-1.9000
gag	neg	-1.4000
gagged	neg	-1.3000
gain	pos	2.4000
gained	pos	1.6000
gaining	pos	1.8000
gains	pos	1.4000
gallant	pos	1.7000
gallantly	pos	1.9000
gallantry	pos	2.6000
generosities	pos	2.6000
generosity	pos	2.3000
generous	pos	2.3000
generously	pos	1.8000
generousness	pos	2.4000
genial	pos	1.8000
gentle	pos	1.9000
gentler	pos	1.4000
gentlest	pos	1.8000
gently	pos	2.0000
ghost	neg	-1.3000
gift	pos	1.9000
giggle	pos	1.8000
giggled	pos	1.5000
gigglers	pos	1.4000
giggliest	pos	1.7000
giggling	pos	1.5000
gigglingly	pos	1.1000
giver	pos	1.4000
givers	pos	1.7000
giving	pos	1.4000
glad	pos	2.0000
gladly	pos	1.4000
glamor	pos	2.1000
glamorise	pos	1.3000
glamorised	pos	1.8000
glamorises	pos	2.1000
glamorising	pos	1.2000
glamorization	pos	1.6000
glamorize	pos	1.7000
glamorized	pos	2.1000
glamorizer	pos	2.4000
glamorizers	pos	1.6000
glamorizes	pos	2.4000
glamorizing	pos	1.8000
glamorous	pos	2.3000
glamorously	pos	2.1000
glamors	pos	1.4000
glamour	pos	2.4000
glamourless	neg	-1.6000
glamourous	pos	2.0000
glamours	pos	1.9000
glee	pos	3.2000
gleeful	pos	2.9000
gloom	neg	-2.6000
gloomed	neg	-1.9000
gloomful	neg	-2.1000
gloomier	neg	-1.5000
gloomiest	neg	-1.8000
gloominess	neg	-1.8000
glooming	neg	-1.8000
gloried	pos	2.4000
glories	pos	2.1000
glorification	pos	2.0000
glorified	pos	2.3000
glorifier	pos	2.3000
glorifiers	pos	1.6000
glorifies	pos	2.2000
glorify	pos	2.7000
glorifying	pos	2.4000
gloriole	pos	1.5000
glorioles	pos	1.2000
glorious	pos	3.2000
gloriously	pos	2.9000
gloriousness	pos	2.6000
glory	pos	2.5000
glum	neg	-2.1000
god	pos	1.1000
goddam	neg	-2.5000
goddammed	neg	-2.4000
goddamn	neg	-2.1000
goddamned	neg	-1.8000
goddamns	neg	-2.1000
goddams	neg	-1.9000
godsend	pos	2.8000
good	pos	1.9000
goodness	pos	2.0000
gorgeous	pos	3.0000
gorgeously	pos	2.3000
gorgeousness	pos	2.9000
gorgeousnesses	pos	2.1000
gossiped	neg	-1.1000
gossiper	neg	-1.1000
gossipers	neg	-1.1000
gossiping	neg	-1.6000
gossipmongers	neg	-1.4000
gossipped	neg	-1.3000
gossipping	neg	-1.8000
gossipry	neg	-1.2000
gossips	neg	-1.3000
gossipy	neg	-1.3000
grace	pos	1.8000
graceful	pos	2.0000
gracefuller	pos	2.2000
gracefullest	pos	2.8000
gracefully	pos	2.4000
gracefulness	pos	2.2000
graces	pos	1.6000
gracile	pos	1.7000
gracility	pos	1.2000
gracing	pos	1.3000
gracious	pos	2.6000
graciously	pos	2.3000
graciousness	pos	2.4000
grand	pos	2.0000
grandee	pos	1.1000
grandees	pos	1.2000
grander	pos	1.7000
grandest	pos	2.4000
grandeur	pos	2.4000
grandeurs	pos	2.1000
grant	pos	1.5000
granting	pos	1.3000
grateful	pos	2.0000
gratefuller	pos	1.8000
gratefully	pos	2.1000
gratefulness	pos	2.2000
gratification	pos	1.6000
gratifications	pos	1.8000
gratified	pos	1.6000
gratifies	pos	1.5000
gratify	pos	1.3000
gratifying	pos	2.3000
gratifyingly	pos	2.0000
gratitude	pos	2.3000
gratz	pos	2.0000
grave	neg	-1.6000
graveless	neg	-1.3000
gravely	neg	-1.5000
graveness	neg	-1.5000
graver	neg	-1.1000
gravers	neg	-1.2000
graves	neg	-1.2000
gravesides	neg	-1.6000
gravest	neg	-1.3000
graveyard	neg	-1.2000
graveyards	neg	-1.2000
great	pos	3.1000
greater	pos	1.5000
greatest	pos	3.2000
greed	neg	-1.7000
greedier	neg	-2.0000
greediest	neg	-2.8000
greedily	neg	-1.9000
greediness	neg	-1.7000
greedy	neg	-1.3000
greenwash	neg	-1.8000
greet	pos	1.3000
greeted	pos	1.1000
greeting	pos	1.6000
greetings	pos	1.8000
grief	neg	-2.2000
grievance	neg	-2.1000
grievances	neg	-1.5000
grievants	neg	-1.1000
grieve	neg	-1.6000
grieved	neg	-2.0000
griever	neg	-1.9000
grieves	neg	-2.1000
grieving	neg	-2.3000
grievous	neg	-2.0000
grievously	neg	-1.7000
grievousness	neg	-2.7000
grim	neg	-2.7000
grimaced	neg	-2.0000
grimaces	neg	-1.8000
grimacing	neg	-1.4000
grime	neg	-1.5000
grimed	neg	-1.2000
grimier	neg	-1.6000
griminess	neg	-1.6000
grimly	neg	-1.3000
grimmer	neg	-1.5000
grimy	neg	-1.8000
grin	pos	2.1000
grinned	pos	1.1000
grinner	pos	1.1000
grinners	pos	1.6000
grinning	pos	1.5000
gross	neg	-2.1000
grossest	neg	-2.1000
grossness	neg	-1.8000
grouch	neg	-2.2000
grouchier	neg	-2.0000
grouchiest	neg	-2.3000
grouchily	neg	-1.4000
grouchiness	neg	-2.0000
grouching	neg	-1.7000
grouchy	neg	-1.9000
growth	pos	1.6000
guilt	neg	-1.1000
guiltier	neg	-2.0000
guiltiest	neg	-1.7000
guiltily	neg	-1.1000
guiltiness	neg	-1.8000
guilts	neg	-1.4000
guilty	neg	-1.8000
gullibility	neg	-1.6000
gullible	neg	-1.5000
gun	neg	-1.4000
h8	neg	-2.7000
ha	pos	1.4000
hacked	neg	-1.7000
haha	pos	2.0000
hahaha	pos	2.6000
hahas	pos	1.8000
hallelujah	pos	3.0000
handsome	pos	2.2000
handsomely	pos	1.9000
handsomeness	pos	2.4000
handsomer	pos	2.0000
handsomest	pos	2.6000
hapless	neg	-1.4000
haplessness	neg	-1.4000
happier	pos	2.4000
happiest	pos	3.2000
happily	pos	2.6000
happiness	pos	2.6000
happing	pos	1.1000
happy	pos	2.7000
harass	neg	-2.2000
harassed	neg	-2.5000
harasser	neg	-2.4000
harassers	neg	-2.8000
harasses	neg	-2.5000
harassing	neg	-2.5000
harassment	neg	-2.5000
harassments	neg	-2.6000
hardship	neg	-1.3000
hardy	pos	1.7000
harm	neg	-2.5000
harmed	neg	-2.1000
harmfully	neg	-2.6000
harmfulness	neg	-2.6000
harming	neg	-2.6000
harmlessly	pos	1.4000
harmonic	pos	1.8000
harmonically	pos	2.1000
harmonics	pos	1.5000
harmonies	pos	1.3000
harmonious	pos	2.0000
harmoniously	pos	1.9000
harmoniousness	pos	1.8000
harmonise	pos	1.8000
harmonised	pos	1.3000
harmonising	pos	1.4000
harmonization	pos	1.9000
harmonize	pos	1.7000
harmonized	pos	1.6000
harmonizer	pos	1.6000
harmonizers	pos	1.6000
harmonizes	pos	1.5000
harmonizing	pos	1.4000
harmony	pos	1.7000
harms	neg	-2.2000
harried	neg	-1.4000
harsh	neg	-1.9000
harsher	neg	-2.2000
harshest	neg	-2.9000
hate	neg	-2.7000
hated	neg	-3.2000
hateful	neg	-2.2000
hatefully	neg	-2.3000
hatefulness	neg	-3.6000
hater	neg	-1.8000
haters	neg	-2.2000
hates	neg	-1.9000
hating	neg	-2.3000
hatred	neg	-3.2000
haunt	neg	-1.7000
haunted	neg	-2.1000
haunting	neg	-1.1000
havoc	neg	-2.9000
healthy	pos	1.7000
heartbreak	neg	-2.7000
heartbreaker	neg	-2.2000
heartbreakers	neg	-2.1000
heartbreaking	neg	-2.0000
heartbreakingly	neg	-1.8000
heartbreaks	neg	-1.8000
heartbroken	neg	-3.3000
heartfelt	pos	2.5000
heartless	neg	-2.2000
heartlessly	neg	-2.8000
heartlessness	neg	-2.8000
heartwarming	pos	2.1000
heaven	pos	2.3000
heavenlier	pos	3.0000
heavenliest	pos	2.7000
heavenliness	pos	2.7000
heavenlinesses	pos	2.3000
heavenly	pos	3.0000
heavens	pos	1.7000
heavenward	pos	1.4000
heavenwards	pos	1.2000
heavyhearted	neg	-2.1000
hell	neg	-3.6000
hellish	neg	-3.2000
help	pos	1.7000
helper	pos	1.4000
helpers	pos	1.1000
helpful	pos	1.8000
helpfully	pos	2.3000
helpfulness	pos	1.9000
helping	pos	1.2000
helpless	neg	-2.0000
helplessly	neg	-1.4000
helplessness	neg	-2.1000
helplessnesses	neg	-1.7000
helps	pos	1.6000
hero	pos	2.6000
heroes	pos	2.3000
heroic	pos	2.6000
heroical	pos	2.9000
heroically	pos	2.4000
heroicomical	pos	1.1000
heroics	pos	2.4000
heroin	neg	-2.2000
heroine	pos	2.7000
heroines	pos	1.8000
heroinism	neg	-2.0000
heroism	pos	2.8000
heroisms	pos	2.2000
heroize	pos	2.1000
heroized	pos	2.0000
heroizes	pos	2.2000
heroizing	pos	1.9000
heros	pos	1.3000
hesitantly	neg	-1.2000
hesitate	neg	-1.1000
hesitated	neg	-1.3000
hesitater	neg	-1.4000
hesitaters	neg	-1.4000
hesitates	neg	-1.4000
hesitating	neg	-1.4000
hesitatingly	neg	-1.5000
hesitation	neg	-1.1000
hesitations	neg	-1.1000
hiding	neg	-1.2000
highlight	pos	1.4000
hilarious	pos	1.7000
hindrance	neg	-1.7000
hoax	neg	-1.1000
holiday	pos	1.7000
holidays	pos	1.6000
homesickness	neg	-1.8000
homesicknesses	neg	-1.8000
honest	pos	2.3000
honester	pos	1.9000
honestest	pos	3.0000
honesties	pos	1.8000
honestly	pos	2.0000
honesty	pos	2.2000
honor	pos	2.2000
honorability	pos	2.2000
honorable	pos	2.5000
honorableness	pos	2.2000
honorably	pos	2.4000
honoraries	pos	1.5000
honorarily	pos	1.9000
honorary	pos	1.4000
honored	pos	2.8000
honoree	pos	2.1000
honorees	pos	2.3000
honorer	pos	1.7000
honorers	pos	1.3000
honorific	pos	1.4000
honorifically	pos	2.2000
honorifics	pos	1.7000
honoring	pos	2.3000
honors	pos	2.3000
honour	pos	2.7000
honourable	pos	2.1000
honoured	pos	2.2000
honourer	pos	1.8000
honourers	pos	1.6000
honouring	pos	2.1000
honours	pos	2.2000
hooligan	neg	-1.5000
hooliganism	neg	-2.1000
hooligans	neg	-1.1000
hooray	pos	2.3000
hope	pos	1.9000
hoped	pos	1.6000
hopeful	pos	2.3000
hopefully	pos	1.7000
hopefulness	pos	1.6000
hopeless	neg	-2.0000
hopelessly	neg	-2.2000
hopelessness	neg	-3.1000
hopes	pos	1.8000
hoping	pos	1.8000
horrendous	neg	-2.8000
horrendously	neg	-1.9000
horrible	neg	-2.5000
horribleness	neg	-2.4000
horribles	neg	-2.1000
horribly	neg	-2.4000
horrid	neg	-2.5000
horridly	neg	-1.4000
horridness	neg	-2.3000
horridnesses	neg	-3.0000
horrific	neg	-3.4000
horrifically	neg	-2.9000
horrified	neg	-2.5000
horrifies	neg	-2.9000
horrify	neg	-2.5000
horrifying	neg	-2.7000
horrifyingly	neg	-3.3000
horror	neg	-2.7000
horrors	neg	-2.7000
hostile	neg	-1.6000
hostilely	neg	-2.2000
hostiles	neg	-1.3000
hostilities	neg	-2.1000
hostility	neg	-2.5000
hug	pos	2.1000
huge	pos	1.3000
huggable	pos	1.6000
hugged	pos	1.7000
hugger	pos	1.6000
huggers	pos	1.8000
hugging	pos	1.8000
hugs	pos	2.2000
humerous	pos	1.4000
humiliate	neg	-2.5000
humiliated	neg	-1.4000
humiliating	neg	-1.2000
humiliatingly	neg	-2.6000
humiliation	neg	-2.7000
humiliations	neg	-2.4000
humor	pos	1.1000
humored	pos	1.2000
humoresque	pos	1.2000
humoring	pos	2.1000
humorist	pos	1.2000
humoristic	pos	1.5000
humorists	pos	1.3000
humorless	neg	-1.3000
humorlessness	neg	-1.4000
humorous	pos	1.6000
humorously	pos	2.3000
humorousness	pos	2.4000
humors	pos	1.6000
humour	pos	2.1000
humoured	pos	1.1000
humouring	pos	1.7000
humourous	pos	2.0000
hurrah	pos	2.6000
hurrahed	pos	1.9000
hurrahing	pos	2.4000
hurrahs	pos	2.1000
hurray	pos	2.7000
hurrayed	pos	1.8000
hurraying	pos	1.2000
hurrays	pos	2.4000
hurt	neg	-2.4000
hurter	neg	-2.3000
hurters	neg	-1.9000
hurtful	neg	-2.4000
hurtfully	neg	-2.6000
hurtfulness	neg	-1.9000
hurting	neg	-1.7000
hurtling	neg	-1.4000
hurts	neg	-2.1000
hypocritical	neg	-2.0000
hysteria	neg	-1.9000
hysterics	neg	-1.8000
ideal	pos	2.4000
idealess	neg	-1.9000
idealise	pos	1.4000
idealised	pos	2.1000
idealises	pos	2.0000
idealism	pos	1.7000
idealist	pos	1.6000
idealistic	pos	1.8000
idealistically	pos	1.7000
idealities	pos	1.5000
ideality	pos	1.9000
idealization	pos	1.8000
idealizations	pos	1.4000
idealize	pos	1.2000
idealized	pos	1.8000
idealizer	pos	1.3000
idealizers	pos	1.9000
idealizes	pos	2.0000
idealizing	pos	1.4000
idealless	neg	-1.7000
ideally	pos	1.8000
idiot	neg	-2.3000
idiotic	neg	-2.6000
ignorami	neg	-1.9000
ignoramus	neg	-1.9000
ignoramuses	neg	-2.3000
ignorance	neg	-1.5000
ignorances	neg	-1.2000
ignorant	neg	-1.1000
ignorantly	neg	-1.6000
ignorantness	neg	-1.1000
ignore	neg	-1.5000
ignored	neg	-1.3000
ignorer	neg	-1.3000
ignores	neg	-1.1000
ignoring	neg	-1.7000
ill	neg	-1.8000
illegal	neg	-2.6000
illiteracy	neg	-1.9000
illness	neg	-1.7000
illnesses	neg	-2.2000
imbecile	neg	-2.2000
immobilized	neg	-1.2000
immoral	neg	-2.0000
immoralism	neg	-1.6000
immoralist	neg	-2.1000
immoralists	neg	-1.7000
immoralities	neg	-1.1000
immorally	neg	-2.1000
immune	pos	1.2000
impatience	neg	-1.8000
impatient	neg	-1.2000
impatiently	neg	-1.7000
imperfect	neg	-1.3000
impersonal	neg	-1.3000
impolite	neg	-1.6000
impolitely	neg	-1.8000
impoliteness	neg	-1.8000
impolitenesses	neg	-2.3000
importance	pos	1.5000
importancy	pos	1.4000
importantly	pos	1.3000
impose	neg	-1.2000
impotent	neg	-1.1000
impress	pos	1.9000
impressed	pos	2.1000
impresses	pos	2.1000
impressibility	pos	1.2000
impressing	pos	2.5000
impressionistic	pos	1.5000
impressionistically	pos	1.6000
impressive	pos	2.3000
impressively	pos	2.0000
impressiveness	pos	1.7000
imprisoned	neg	-2.0000
improve	pos	1.9000
improved	pos	2.1000
improvement	pos	2.0000
improvements	pos	1.3000
improver	pos	1.8000
improvers	pos	1.3000
improves	pos	1.8000
improving	pos	1.8000
inability	neg	-1.7000
inadequacies	neg	-1.7000
inadequacy	neg	-1.7000
inadequate	neg	-1.7000
inadequateness	neg	-1.7000
inadequatenesses	neg	-1.6000
incapable	neg	-1.6000
incapacitated	neg	-1.9000
incensed	neg	-2.0000
incentive	pos	1.5000
incentives	pos	1.3000
incompetence	neg	-2.3000
incompetent	neg	-2.1000
inconsiderate	neg	-1.9000
inconvenience	neg	-1.5000
inconvenient	neg	-1.4000
increase	pos	1.3000
increased	pos	1.1000
indecisions	neg	-1.1000
indecisiveness	neg	-1.3000
indignant	neg	-1.8000
indignation	neg	-2.4000
indoctrinate	neg	-1.4000
ineffectively	neg	-1.3000
ineffectiveness	neg	-1.3000
ineffectual	neg	-1.2000
ineffectuality	neg	-1.6000
ineffectually	neg	-1.1000
ineffectualness	neg	-1.3000
infected	neg	-2.2000
inferior	neg	-1.7000
inferiorities	neg	-1.9000
inferiority	neg	-1.1000
inferiorly	neg	-2.0000
inflamed	neg	-1.4000
influential	pos	1.9000
infringement	neg	-2.1000
infuriate	neg	-2.2000
infuriated	neg	-3.0000
infuriates	neg	-2.6000
infuriating	neg	-2.4000
inhibit	neg	-1.6000
inhibition	neg	-1.5000
inhibitive	neg	-1.4000
injured	neg	-1.7000
injury	neg	-1.8000
injustice	neg	-2.7000
innocence	pos	1.6000
innocency	pos	1.9000
innocent	pos	1.4000
innocently	pos	1.4000
innocents	pos	1.1000
innovate	pos	2.2000
innovates	pos	2.0000
innovation	pos	1.6000
innovative	pos	1.9000
inquisition	neg	-1.2000
insane	neg	-1.7000
insanity	neg	-2.7000
insecure	neg	-1.8000
insecurely	neg	-1.4000
insecureness	neg	-1.8000
insecurities	neg	-1.8000
insecurity	neg	-1.8000
insensitivity	neg	-1.8000
insignificant	neg	-1.4000
insincere	neg	-1.8000
insincerely	neg	-1.9000
insincerity	neg	-1.4000
insipid	neg	-2.0000
inspiration	pos	2.4000
inspirational	pos	2.3000
inspirationally	pos	2.3000
inspirations	pos	2.1000
inspirator	pos	1.9000
inspirators	pos	1.2000
inspiratory	pos	1.5000
inspire	pos	2.7000
inspired	pos	2.2000
inspirer	pos	2.2000
inspirers	pos	2.0000
inspires	pos	1.9000
inspiring	pos	1.8000
inspiringly	pos	2.6000
inspirit	pos	1.9000
inspirited	pos	1.3000
inspiriting	pos	1.8000
inspiritingly	pos	2.1000
insult	neg	-2.3000
insulted	neg	-2.3000
insulter	neg	-2.0000
insulters	neg	-2.0000
insulting	neg	-2.2000
insultingly	neg	-2.3000
insults	neg	-1.8000
integrity	pos	1.6000
intellect	pos	2.0000
intellective	pos	1.7000
intellects	pos	1.8000
intellectual	pos	2.3000
intellectualism	pos	2.2000
intellectualist	pos	2.0000
intellectualistic	pos	1.3000
intellectualities	pos	1.7000
intellectuality	pos	1.7000
intellectualization	pos	1.5000
intellectualize	pos	1.5000
intellectualized	pos	1.2000
intellectualizes	pos	1.8000
intellectually	pos	1.4000
intellectualness	pos	1.5000
intellectuals	pos	1.6000
intelligence	pos	2.1000
intelligencer	pos	1.5000
intelligencers	pos	1.6000
intelligences	pos	1.6000
intelligent	pos	2.0000
intelligential	pos	1.9000
intelligently	pos	2.0000
intelligentsia	pos	1.5000
intelligibility	pos	1.5000
intelligible	pos	1.4000
intelligibleness	pos	1.5000
intelligibly	pos	1.2000
interest	pos	2.0000
interested	pos	1.7000
interestedly	pos	1.5000
interesting	pos	1.7000
interestingly	pos	1.7000
interestingness	pos	1.8000
interrogated	neg	-1.6000
interrupt	neg	-1.4000
interrupted	neg	-1.2000
interrupter	neg	-1.1000
interrupters	neg	-1.3000
interruptible	neg	-1.3000
interrupting	neg	-1.2000
interruption	neg	-1.5000
interruptions	neg	-1.7000
interruptive	neg	-1.4000
interruptor	neg	-1.3000
interrupts	neg	-1.3000
intimidated	neg	-1.9000
intimidates	neg	-1.3000
intimidating	neg	-1.9000
intimidatingly	neg	-1.1000
intimidation	neg	-1.8000
intimidations	neg	-1.4000
intimidator	neg	-1.6000
intimidators	neg	-1.6000
intimidatory	neg	-1.1000
invigorate	pos	1.9000
invigorates	pos	2.1000
invigorating	pos	2.1000
invigoratingly	pos	2.0000
invigoration	pos	1.5000
invigorations	pos	1.2000
invigorator	pos	1.1000
invigorators	pos	1.2000
invincible	pos	2.2000
inviting	pos	1.3000
invulnerable	pos	1.3000
irate	neg	-2.9000
irrational	neg	-1.4000
irrationalism	neg	-1.5000
irrationalist	neg	-2.1000
irrationalists	neg	-1.5000
irrationalities	neg	-1.5000
irrationality	neg	-1.7000
irrationally	neg	-1.6000
irrationals	neg	-1.1000
irresistible	pos	1.4000
irresolute	neg	-1.4000
irresponsible	neg	-1.9000
irritabilities	neg	-1.7000
irritability	neg	-1.4000
irritable	neg	-2.1000
irritableness	neg	-1.7000
irritably	neg	-1.8000
irritant	neg	-2.3000
irritants	neg	-2.1000
irritate	neg	-1.8000
irritated	neg	-2.0000
irritates	neg	-1.7000
irritating	neg	-2.0000
irritatingly	neg	-2.0000
irritation	neg	-2.3000
irritations	neg	-1.5000
irritative	neg	-2.0000
isolated	neg	-1.3000
isolates	neg	-1.3000
isolation	neg	-1.7000
itchy	neg	-1.1000
jackass	neg	-1.8000
jackasses	neg	-2.8000
jaded	neg	-1.6000
jailed	neg	-2.2000
jaunty	pos	1.2000
jealous	neg	-2.0000
jealousies	neg	-2.0000
jealously	neg	-2.0000
jealousness	neg	-1.7000
jealousy	neg	-1.3000
jeopardy	neg	-2.1000
jerk	neg	-1.4000
jerks	neg	-1.1000
jewel	pos	1.5000
jewels	pos	2.0000
jocular	pos	1.2000
join	pos	1.2000
joke	pos	1.2000
joked	pos	1.3000
jokester	pos	1.5000
jokey	pos	1.1000
jollied	pos	2.4000
jollier	pos	2.4000
jollies	pos	2.0000
jolliest	pos	2.9000
jollification	pos	2.2000
jollifications	pos	2.0000
jollify	pos	2.1000
jollily	pos	2.7000
jolliness	pos	2.5000
jollities	pos	1.7000
jollity	pos	1.8000
jolly	pos	2.3000
jollying	pos	2.3000
jovial	pos	1.9000
joy	pos	2.8000
joyance	pos	2.3000
joyed	pos	2.9000
joyful	pos	2.9000
joyfuller	pos	2.4000
joyfully	pos	2.5000
joyfulness	pos	2.7000
joying	pos	2.5000
joyless	neg	-2.5000
joylessly	neg	-1.7000
joylessness	neg	-2.7000
joyous	pos	3.1000
joyously	pos	2.9000
joyousness	pos	2.8000
joyride	pos	1.1000
joyriders	pos	1.3000
joys	pos	2.2000
jubilant	pos	3.0000
justice	pos	2.4000
justified	pos	1.7000
keen	pos	1.5000
keenest	pos	1.9000
keenness	pos	1.4000
kewl	pos	1.3000
kill	neg	-3.7000
killdeer	neg	-1.1000
killed	neg	-3.5000
killer	neg	-3.3000
killers	neg	-3.3000
killing	neg	-3.4000
killingly	neg	-2.6000
killings	neg	-3.5000
killjoy	neg	-2.1000
killjoys	neg	-1.7000
kills	neg	-2.5000
kind	pos	2.4000
kinder	pos	2.2000
kindly	pos	2.2000
kindness	pos	2.0000
kindnesses	pos	2.3000
kiss	pos	1.8000
kissable	pos	2.0000
kissably	pos	1.9000
kissed	pos	1.6000
kisser	pos	1.7000
kissers	pos	1.5000
kisses	pos	2.3000
kissing	pos	2.7000
kissy	pos	1.8000
kudos	pos	2.3000
lack	neg	-1.3000
lackadaisical	neg	-1.6000
lag	neg	-1.4000
lagged	neg	-1.2000
lagging	neg	-1.1000
lags	neg	-1.5000
lame	neg	-1.8000
lamebrain	neg	-1.6000
lamebrained	neg	-2.5000
lamebrains	neg	-1.2000
lamely	neg	-2.0000
lament	neg	-2.0000
lamentable	neg	-1.5000
lamentableness	neg	-1.3000
lamentably	neg	-1.5000
lamentation	neg	-1.4000
lamentations	neg	-1.9000
lamented	neg	-1.4000
lamenter	neg	-1.2000
lamenting	neg	-2.0000
laments	neg	-1.5000
lamer	neg	-1.4000
lames	neg	-1.2000
lamest	neg	-1.5000
laugh	pos	2.6000
laughableness	pos	1.2000
laughably	pos	1.2000
laughed	pos	2.0000
laugher	pos	1.7000
laughers	pos	1.7000
laughing	pos	2.2000
laughingly	pos	2.3000
laughings	pos	1.9000
laughingstocks	neg	-1.3000
laughs	pos	2.2000
laughter	pos	2.2000
laughters	pos	2.2000
lawl	pos	1.4000
lazier	neg	-2.3000
laziest	neg	-2.7000
lazy	neg	-1.5000
leak	neg	-1.4000
leaked	neg	-1.3000
leet	pos	1.3000
lenient	pos	1.1000
lethargic	neg	-1.2000
lethargy	neg	-1.4000
liar	neg	-2.3000
liars	neg	-2.4000
libelous	neg	-2.1000
liberties	pos	2.3000
libertinisms	pos	1.2000
liberty	pos	2.4000
lied	neg	-1.6000
lies	neg	-1.8000
lifesaver	pos	2.8000
lighthearted	pos	1.8000
like	pos	1.5000
likeable	pos	2.0000
liked	pos	1.8000
likes	pos	1.8000
liking	pos	1.7000
limitation	neg	-1.2000
livelier	pos	1.7000
liveliest	pos	2.1000
livelily	pos	1.8000
liveliness	pos	1.6000
livelong	pos	1.7000
lively	pos	1.9000
livid	neg	-2.5000
lmao	pos	2.9000
loathe	neg	-2.2000
loathed	neg	-2.1000
loathes	neg	-1.9000
loathing	neg	-2.7000
lol	pos	1.8000
lone	neg	-1.1000
lonelier	neg	-1.4000
loneliest	neg	-2.4000
loneliness	neg	-1.8000
lonelinesses	neg	-1.5000
lonely	neg	-1.5000
loneness	neg	-1.1000
loner	neg	-1.3000
lonesome	neg	-1.5000
lonesomely	neg	-1.3000
lonesomeness	neg	-1.8000
lonesomes	neg	-1.4000
loomed	neg	-1.1000
loose	neg	-1.3000
lose	neg	-1.7000
loser	neg	-2.4000
losers	neg	-2.4000
loses	neg	-1.3000
losing	neg	-1.6000
loss	neg	-1.3000
losses	neg	-1.7000
lossy	neg	-1.2000
lost	neg	-1.3000
louse	neg	-1.6000
louses	neg	-1.3000
lousier	neg	-2.2000
lousiest	neg	-2.6000
lousily	neg	-1.2000
lousiness	neg	-1.7000
lousing	neg	-1.1000
lousy	neg	-2.5000
lovable	pos	3.0000
love	pos	3.2000
loved	pos	2.9000
lovelies	pos	2.2000
lovely	pos	2.8000
lover	pos	2.8000
loverly	pos	2.8000
lovers	pos	2.4000
loves	pos	2.7000
loving	pos	2.9000
lovingly	pos	3.2000
lovingness	pos	2.7000
low	neg	-1.1000
lowballed	neg	-1.5000
lowballs	neg	-1.2000
lowbred	neg	-2.6000
lowbrow	neg	-1.9000
lower	neg	-1.2000
lowermost	neg	-1.4000
lowery	neg	-1.8000
lowest	neg	-1.6000
lowlier	neg	-1.7000
lowliest	neg	-1.8000
lowlife	neg	-1.5000
lowlifes	neg	-2.2000
lowlight	neg	-2.0000
lowliness	neg	-1.1000
lowlinesses	neg	-1.2000
lowlives	neg	-2.1000
lowness	neg	-1.3000
loyal	pos	2.1000
loyalist	pos	1.5000
loyalists	pos	1.1000
loyally	pos	2.1000
loyalties	pos	1.9000
loyalty	pos	2.5000
luck	pos	2.0000
lucked	pos	1.9000
luckie	pos	1.6000
luckier	pos	1.9000
luckiest	pos	2.9000
luckily	pos	2.3000
lucking	pos	1.2000
luckless	neg	-1.3000
lucks	pos	1.6000
lucky	pos	1.8000
ludicrous	neg	-1.5000
ludicrousness	neg	-1.9000
lugubrious	neg	-2.1000
lulz	pos	2.0000
lunatic	neg	-2.2000
lunatics	neg	-1.6000
lying	neg	-2.4000
mad	neg	-2.2000
maddening	neg	-2.2000
madder	neg	-1.2000
maddest	neg	-2.8000
madly	neg	-1.7000
madness	neg	-1.9000
magnific	pos	2.3000
magnifical	pos	2.4000
magnifically	pos	2.4000
magnifications	pos	1.2000
magnificence	pos	2.4000
magnificences	pos	2.3000
magnificent	pos	2.9000
magnificently	pos	3.4000
magnifico	pos	1.8000
magnificoes	pos	1.4000
maniac	neg	-2.1000
maniacally	neg	-1.7000
maniacs	neg	-1.2000
manipulated	neg	-1.6000
manipulating	neg	-1.5000
manipulation	neg	-1.2000
marvel	pos	1.8000
marvelous	pos	2.9000
marvels	pos	2.0000
masochism	neg	-1.6000
masochisms	neg	-1.1000
masochist	neg	-1.7000
masochistic	neg	-2.2000
masochistically	neg	-1.6000
masochists	neg	-1.2000
masterpiece	pos	3.1000
masterpieces	pos	2.5000
mature	pos	1.8000
meaningful	pos	1.3000
meaningless	neg	-1.9000
medal	pos	2.1000
meditative	pos	1.4000
melancholiac	neg	-2.0000
melancholias	neg	-1.6000
melancholies	neg	-1.1000
melancholy	neg	-1.9000
menace	neg	-2.2000
menaced	neg	-1.7000
mercy	pos	1.5000
merit	pos	1.8000
merited	pos	1.4000
meriting	pos	1.1000
meritocrats	pos	1.1000
meritorious	pos	2.1000
meritoriously	pos	1.3000
meritoriousness	pos	1.7000
merits	pos	1.7000
merrier	pos	1.7000
merriest	pos	2.7000
merrily	pos	2.4000
merriment	pos	2.4000
merriments	pos	2.0000
merriness	pos	2.2000
merry	pos	2.5000
merrymaker	pos	2.2000
merrymakers	pos	1.7000
merrymaking	pos	2.2000
merrymakings	pos	2.4000
merrythought	pos	1.1000
merrythoughts	pos	1.6000
mess	neg	-1.5000
messed	neg	-1.4000
messy	neg	-1.5000
mindless	neg	-1.9000
miracle	pos	2.8000
mirth	pos	2.6000
mirthful	pos	2.7000
mirthfully	pos	2.0000
misbehave	neg	-1.9000
misbehaved	neg	-1.6000
misbehaves	neg	-1.6000
misbehaving	neg	-1.7000
mischief	neg	-1.5000
miser	neg	-1.8000
miserable	neg	-2.2000
miserableness	neg	-2.8000
miserably	neg	-2.1000
miseries	neg	-2.7000
miserliness	neg	-2.6000
miserly	neg	-1.4000
misers	neg	-1.5000
misery	neg	-2.7000
misgiving	neg	-1.4000
misinformation	neg	-1.3000
misinformed	neg	-1.6000
misinterpreted	neg	-1.3000
misleading	neg	-1.7000
misread	neg	-1.1000
misreporting	neg	-1.5000
misrepresentation	neg	-2.0000
missed	neg	-1.2000
missing	neg	-1.2000
mistake	neg	-1.4000
mistaken	neg	-1.5000
mistakenly	neg	-1.2000
mistaker	neg	-1.6000
mistakers	neg	-1.6000
mistakes	neg	-1.5000
mistaking	neg	-1.1000
misunderstand	neg	-1.5000
misunderstanding	neg	-1.8000
misunderstands	neg	-1.3000
misunderstood	neg	-1.4000
mlm	neg	-1.4000
mock	neg	-1.8000
mocked	neg	-1.3000
mockeries	neg	-1.6000
mockers	neg	-1.3000
mockery	neg	-1.3000
mocking	neg	-1.7000
mocks	neg	-2.0000
molest	neg	-2.1000
molestation	neg	-1.9000
molestations	neg	-2.9000
molested	neg	-1.9000
molester	neg	-2.3000
molesters	neg	-2.2000
molesting	neg	-2.8000
molests	neg	-3.1000
monopolizes	neg	-1.1000
mooch	neg	-1.7000
mooched	neg	-1.4000
moocher	neg	-1.5000
moochers	neg	-1.9000
mooches	neg	-1.4000
mooching	neg	-1.7000
moodier	neg	-1.1000
moodiest	neg	-2.1000
moodily	neg	-1.3000
moodiness	neg	-1.4000
moodinesses	neg	-1.4000
moody	neg	-1.5000
mope	neg	-1.9000
moron	neg	-2.2000
moronic	neg	-2.7000
moronically	neg	-1.4000
moronity	neg	-1.1000
morons	neg	-1.3000
motherfucker	neg	-3.6000
motherfucking	neg	-2.8000
motivate	pos	1.6000
motivated	pos	2.0000
motivating	pos	2.2000
motivation	pos	1.4000
mourn	neg	-1.8000
mourned	neg	-1.3000
mourner	neg	-1.6000
mourners	neg	-1.8000
mournful	neg	-1.6000
mournfuller	neg	-1.9000
mournfully	neg	-1.7000
mournfulness	neg	-1.8000
mourning	neg	-1.9000
mourningly	neg	-2.3000
mourns	neg	-2.4000
muah	pos	2.3000
mumpish	neg	-1.4000
murder	neg	-3.7000
murdered	neg	-3.4000
murderee	neg	-3.2000
murderees	neg	-3.1000
murderer	neg	-3.6000
murderers	neg	-3.3000
murderess	neg	-2.2000
murderesses	neg	-2.6000
murdering	neg	-3.3000
murderous	neg	-3.2000
murderously	neg	-3.1000
murderousness	neg	-2.9000
murders	neg	-3.0000
n00b	neg	-1.6000
nag	neg	-1.5000
nagana	neg	-1.7000
nagged	neg	-1.7000
nagger	neg	-1.8000
naggers	neg	-1.5000
naggier	neg	-1.4000
naggiest	neg	-2.4000
nagging	neg	-1.7000
naggy	neg	-1.7000
nags	neg	-1.1000
naive	neg	-1.1000
nastier	neg	-2.3000
nasties	neg	-2.1000
nastiest	neg	-2.4000
nastily	neg	-1.9000
nastiness	neg	-1.1000
nastinesses	neg	-2.6000
nasty	neg	-2.6000
natural	pos	1.5000
neat	pos	2.0000
neaten	pos	1.2000
neatened	pos	2.0000
neatening	pos	1.3000
neatens	pos	1.1000
neatest	pos	1.7000
neatly	pos	1.4000
neatness	pos	1.3000
neats	pos	1.1000
needy	neg	-1.4000
negative	neg	-2.7000
negativity	neg	-2.3000
neglect	neg	-2.0000
neglected	neg	-2.4000
neglecter	neg	-1.7000
neglecters	neg	-1.5000
neglectful	neg	-2.0000
neglectfully	neg	-2.1000
neglectfulness	neg	-2.0000
neglecting	neg	-1.7000
neglects	neg	-2.2000
nerd	neg	-1.2000
nervous	neg	-1.1000
nervousness	neg	-1.2000
neurotic	neg	-1.4000
neurotically	neg	-1.8000
nice	pos	1.8000
nicely	pos	1.9000
niceness	pos	1.6000
nicenesses	pos	2.1000
nicer	pos	1.9000
nicest	pos	2.2000
niceties	pos	1.5000
nicety	pos	1.2000
nifty	pos	1.7000
niggas	neg	-1.4000
nigger	neg	-3.3000
no	neg	-1.2000
noble	pos	2.0000
nonsense	neg	-1.7000
notorious	neg	-1.9000
novel	pos	1.3000
numb	neg	-1.4000
numbing	neg	-1.1000
numbingly	neg	-1.3000
numbly	neg	-1.4000
numbness	neg	-1.1000
numbskull	neg	-2.3000
numbskulls	neg	-2.2000
nurtural	pos	1.5000
nurturance	pos	1.6000
nurturances	pos	1.3000
nurturant	pos	1.7000
nurture	pos	1.4000
nurtured	pos	1.9000
nurturer	pos	1.9000
nurtures	pos	1.9000
nurturing	pos	2.0000
nuts	neg	-1.3000
o/\o	pos	2.1000
obliterate	neg	-2.9000
obliterated	neg	-2.1000
obnoxious	neg	-2.0000
obnoxiously	neg	-2.3000
obnoxiousness	neg	-2.1000
obscene	neg	-2.8000
obsessing	neg	-1.4000
obsession	neg	-1.4000
obsessional	neg	-1.5000
obsessionally	neg	-1.3000
obsessiveness	neg	-1.2000
obsolete	neg	-1.2000
obstacle	neg	-1.5000
obstacles	neg	-1.6000
obstinate	neg	-1.2000
odd	neg	-1.3000
offence	neg	-1.2000
offences	neg	-1.4000
offend	neg	-1.2000
offender	neg	-1.5000
offenders	neg	-1.5000
offending	neg	-2.3000
offends	neg	-2.0000
offenses	neg	-1.5000
offensive	neg	-2.0000
offensively	neg	-2.8000
offensiveness	neg	-2.3000
ok	pos	1.2000
okays	pos	2.1000
ominous	neg	-1.4000
once-in-a-lifetime	pos	1.8000
openness	pos	1.4000
opportune	pos	1.7000
opportunely	pos	1.5000
opportuneness	pos	1.2000
opportunities	pos	1.6000
opportunity	pos	1.8000
oppressed	neg	-2.1000
oppressive	neg	-1.7000
optimal	pos	1.5000
optimality	pos	1.9000
optimally	pos	1.3000
optimisation	pos	1.6000
optimisations	pos	1.8000
optimise	pos	1.9000
optimised	pos	1.7000
optimises	pos	1.6000
optimising	pos	1.7000
optimism	pos	2.5000
optimisms	pos	2.0000
optimist	pos	2.4000
optimistic	pos	1.3000
optimistically	pos	2.1000
optimists	pos	1.6000
optimization	pos	1.6000
optimize	pos	2.2000
optimized	pos	2.0000
optimizer	pos	1.5000
optimizers	pos	2.1000
optimizes	pos	1.8000
optimizing	pos	2.0000
optionless	neg	-1.7000
original	pos	1.3000
outcry	neg	-2.3000
outgoing	pos	1.2000
outrage	neg	-2.3000
outraged	neg	-2.5000
outrageous	neg	-2.0000
outrageously	neg	-1.2000
outrageousness	neg	-1.2000
outrageousnesses	neg	-1.3000
outrages	neg	-2.3000
outraging	neg	-2.0000
outreach	pos	1.1000
outstanding	pos	3.0000
overjoyed	pos	2.7000
overload	neg	-1.5000
overreacted	neg	-1.7000
overreacts	neg	-2.2000
overstatement	neg	-1.1000
overweight	neg	-1.5000
pain	neg	-2.3000
pained	neg	-1.8000
painful	neg	-1.9000
painfuller	neg	-1.7000
painfully	neg	-2.4000
painfulness	neg	-2.7000
paining	neg	-1.7000
painless	pos	1.2000
painlessly	pos	1.1000
pains	neg	-1.8000
palatable	pos	1.6000
palatably	pos	1.1000
panic	neg	-2.3000
panicked	neg	-2.0000
panicking	neg	-1.9000
panicky	neg	-1.5000
panics	neg	-1.9000
paradise	pos	3.2000
paranoiac	neg	-1.3000
paranoias	neg	-1.5000
paranoids	neg	-1.6000
pardon	pos	1.3000
pardoning	pos	1.7000
pardons	pos	1.2000
partied	pos	1.4000
partier	pos	1.4000
parties	pos	1.7000
party	pos	1.7000
partyer	pos	1.2000
partyers	pos	1.1000
partying	pos	1.6000
passion	pos	2.0000
passional	pos	1.6000
passionate	pos	2.4000
passionately	pos	2.4000
passionateness	pos	2.3000
passionless	neg	-1.9000
passions	pos	2.2000
pathetic	neg	-2.7000
pathetical	neg	-1.2000
pathetically	neg	-1.8000
peace	pos	2.5000
peaceable	pos	1.7000
peaceableness	pos	1.8000
peaceably	pos	2.0000
peaceful	pos	2.2000
peacefuller	pos	1.9000
peacefullest	pos	3.1000
peacefully	pos	2.4000
peacefulness	pos	2.1000
peacekeeper	pos	1.6000
peacekeepers	pos	1.6000
peacekeeping	pos	2.0000
peacekeepings	pos	1.6000
peacemaker	pos	2.0000
peacemakers	pos	2.4000
peacemaking	pos	1.7000
peaces	pos	2.1000
peacetime	pos	2.2000
peacetimes	pos	2.1000
penalty	neg	-2.0000
perfect	pos	2.7000
perfecta	pos	1.4000
perfected	pos	2.7000
perfecter	pos	1.8000
perfecters	pos	1.4000
perfectest	pos	3.1000
perfectibilities	pos	2.1000
perfectibility	pos	1.8000
perfectible	pos	1.5000
perfecting	pos	2.3000
perfection	pos	2.7000
perfectionism	pos	1.3000
perfectionist	pos	1.5000
perfections	pos	2.5000
perfective	pos	1.2000
perfectively	pos	2.1000
perfectivity	pos	2.2000
perfectly	pos	3.2000
perfectness	pos	3.0000
perfecto	pos	1.3000
perfects	pos	1.6000
peril	neg	-1.7000
perjury	neg	-1.9000
perpetrator	neg	-2.2000
perplexed	neg	-1.3000
persecute	neg	-2.1000
persecuted	neg	-1.3000
persecutes	neg	-1.2000
persecuting	neg	-1.5000
perturbed	neg	-1.4000
perverse	neg	-1.8000
perversely	neg	-2.2000
perverseness	neg	-2.1000
perversion	neg	-1.3000
perversions	neg	-1.2000
perversities	neg	-1.1000
perversity	neg	-2.6000
perversive	neg	-2.1000
pervert	neg	-2.3000
perverted	neg	-2.5000
pervertedly	neg	-1.2000
pervertedness	neg	-1.2000
perverter	neg	-1.7000
perverts	neg	-2.8000
pesky	neg	-1.2000
pessimism	neg	-1.5000
pessimisms	neg	-2.0000
pessimist	neg	-1.5000
pessimistic	neg	-1.5000
pessimistically	neg	-2.0000
petrifaction	neg	-1.9000
petrified	neg	-2.5000
petrifies	neg	-2.3000
petrify	neg	-1.7000
petrifying	neg	-2.6000
pettiest	neg	-1.3000
phobia	neg	-1.6000
phobias	neg	-2.0000
phobic	neg	-1.2000
phobics	neg	-1.3000
picturesque	pos	1.6000
pileup	neg	-1.1000
pique	neg	-1.1000
piss	neg	-1.7000
pissant	neg	-1.5000
pissants	neg	-2.5000
pissed	neg	-3.2000
pisser	neg	-2.0000
pissers	neg	-1.4000
pisses	neg	-1.4000
pissing	neg	-1.7000
piteous	neg	-1.2000
pitiable	neg	-1.1000
pitiableness	neg	-1.1000
pitiably	neg	-1.1000
pitied	neg	-1.3000
pitier	neg	-1.2000
pitiers	neg	-1.3000
pities	neg	-1.2000
pitiful	neg	-2.2000
pitifuller	neg	-1.8000
pitifullest	neg	-1.1000
pitifully	neg	-1.2000
pitifulness	neg	-1.2000
pitiless	neg	-1.8000
pitilessly	neg	-2.1000
pity	neg	-1.2000
pitying	neg	-1.4000
play	pos	1.4000
played	pos	1.4000
playful	pos	1.9000
playfully	pos	1.6000
playfulness	pos	1.2000
pleasant	pos	2.3000
pleasanter	pos	1.5000
pleasantest	pos	2.6000
pleasantly	pos	2.1000
pleasantness	pos	2.3000
pleasantnesses	pos	2.3000
pleasantries	pos	1.3000
pleasantry	pos	2.0000
please	pos	1.3000
pleased	pos	1.9000
pleaser	pos	1.7000
pleases	pos	1.7000
pleasing	pos	2.4000
pleasurability	pos	1.9000
pleasurable	pos	2.4000
pleasurableness	pos	2.4000
pleasurably	pos	2.6000
pleasure	pos	2.7000
pleasured	pos	2.3000
pleasureless	neg	-1.6000
pleasures	pos	1.9000
pleasuring	pos	2.8000
poison	neg	-2.5000
poisoned	neg	-2.2000
poisoner	neg	-2.7000
poisoners	neg	-3.1000
poisoning	neg	-2.8000
poisonings	neg	-2.4000
poisonous	neg	-2.7000
poisonously	neg	-2.9000
poisons	neg	-2.7000
pollute	neg	-2.3000
polluted	neg	-2.0000
polluter	neg	-1.8000
polluters	neg	-2.0000
pollutes	neg	-2.2000
poor	neg	-2.1000
poorer	neg	-1.5000
poorest	neg	-2.5000
popular	pos	1.8000
popularise	pos	1.6000
popularised	pos	1.1000
popularising	pos	1.2000
popularities	pos	1.6000
popularity	pos	2.1000
popularization	pos	1.3000
popularize	pos	1.3000
popularized	pos	1.9000
popularizer	pos	1.8000
popularizes	pos	1.4000
popularizing	pos	1.5000
popularly	pos	1.8000
positive	pos	2.6000
positively	pos	2.4000
positiveness	pos	2.3000
positivenesses	pos	2.2000
positiver	pos	2.3000
positives	pos	2.4000
positivest	pos	2.9000
positivism	pos	1.6000
positivisms	pos	1.8000
positivist	pos	2.0000
positivistic	pos	1.9000
positivists	pos	1.7000
positivities	pos	2.6000
positivity	pos	2.3000
postpones	neg	-1.1000
poverty	neg	-2.3000
powerful	pos	1.8000
powerless	neg	-2.2000
praise	pos	2.6000
praised	pos	2.2000
praiser	pos	2.0000
praisers	pos	2.0000
praises	pos	2.4000
praiseworthily	pos	1.9000
praiseworthiness	pos	2.4000
praiseworthy	pos	2.6000
praising	pos	2.5000
pray	pos	1.3000
praying	pos	1.5000
prays	pos	1.4000
prblm	neg	-1.6000
prblms	neg	-2.3000
precious	pos	2.7000
preciously	pos	2.2000
preciousness	pos	1.9000
prejudice	neg	-2.3000
prejudiced	neg	-1.9000
prejudices	neg	-1.8000
prejudicial	neg	-2.6000
prejudicially	neg	-1.5000
prejudicialness	neg	-2.4000
prejudicing	neg	-1.8000
pressure	neg	-1.2000
pressures	neg	-1.3000
pressuring	neg	-1.4000
prettied	pos	1.6000
prettier	pos	2.1000
pretties	pos	1.7000
prettiest	pos	2.7000
pretty	pos	2.2000
prick	neg	-1.4000
pricklier	neg	-1.6000
prickliest	neg	-1.4000
pride	pos	1.4000
prison	neg	-2.3000
prisoner	neg	-2.5000
prisoners	neg	-2.3000
privilege	pos	1.5000
privileged	pos	1.9000
privileges	pos	1.6000
prize	pos	2.3000
prized	pos	2.4000
prizes	pos	2.0000
prizewinner	pos	2.3000
prizewinners	pos	2.4000
prizewinning	pos	3.0000
proactive	pos	1.8000
problem	neg	-1.7000
problematic	neg	-1.9000
problematical	neg	-1.8000
problematically	neg	-2.0000
problematics	neg	-1.3000
problems	neg	-1.7000
profit	pos	1.9000
profitabilities	pos	1.1000
profitability	pos	1.1000
profitable	pos	1.9000
profitableness	pos	2.4000
profitably	pos	1.6000
profited	pos	1.3000
profiting	pos	1.6000
profitless	neg	-1.5000
profits	pos	1.9000
progress	pos	1.8000
prominent	pos	1.3000
promiscuity	neg	-1.8000
promiscuously	neg	-1.5000
promise	pos	1.3000
promised	pos	1.5000
promisees	pos	1.1000
promiser	pos	1.3000
promisers	pos	1.6000
promises	pos	1.6000
promising	pos	1.7000
promisingly	pos	1.2000
promote	pos	1.6000
promoted	pos	1.8000
promotes	pos	1.4000
promoting	pos	1.5000
prosecute	neg	-1.7000
prosecuted	neg	-1.6000
prosecutes	neg	-1.8000
prosecution	neg	-2.2000
prospect	pos	1.2000
prospects	pos	1.2000
prosperous	pos	2.1000
protect	pos	1.6000
protected	pos	1.9000
protects	pos	1.3000
protesting	neg	-1.8000
proud	pos	2.1000
prouder	pos	2.2000
proudest	pos	2.6000
proudful	pos	1.9000
proudhearted	pos	1.4000
proudly	pos	2.6000
provoke	neg	-1.7000
provoked	neg	-1.1000
provokes	neg	-1.3000
pseudoscience	neg	-1.2000
puke	neg	-2.4000
puked	neg	-1.8000
pukes	neg	-1.9000
puking	neg	-1.8000
pukka	pos	2.8000
punish	neg	-2.4000
punishabilities	neg	-1.7000
punishability	neg	-1.6000
punishable	neg	-1.9000
punished	neg	-2.0000
punisher	neg	-1.9000
punishers	neg	-2.6000
punishes	neg	-2.1000
punishing	neg	-2.6000
punishment	neg	-2.2000
punishments	neg	-1.8000
punitive	neg	-2.3000
pushy	neg	-1.1000
quaking	neg	-1.5000
questionable	neg	-1.2000
racism	neg	-3.1000
racist	neg	-3.0000
racists	neg	-2.5000
radiance	pos	1.4000
radiances	pos	1.1000
radiancy	pos	1.4000
radiant	pos	2.1000
radiantly	pos	1.3000
radiants	pos	1.2000
rage	neg	-2.6000
raged	neg	-2.0000
rageful	neg	-2.8000
rages	neg	-2.1000
raging	neg	-2.4000
rancid	neg	-2.5000
rancidity	neg	-2.6000
rancidly	neg	-2.5000
rancidness	neg	-2.6000
rancidnesses	neg	-1.6000
rant	neg	-1.4000
ranter	neg	-1.2000
ranters	neg	-1.2000
rants	neg	-1.3000
rape	neg	-3.7000
raped	neg	-3.6000
raper	neg	-3.4000
rapers	neg	-3.6000
rapes	neg	-3.5000
raping	neg	-3.8000
rapist	neg	-3.9000
rapists	neg	-3.3000
rapturous	pos	1.7000
rash	neg	-1.7000
ready	pos	1.5000
reassurance	pos	1.5000
reassurances	pos	1.4000
reassure	pos	1.4000
reassured	pos	1.7000
reassures	pos	1.5000
reassuring	pos	1.7000
reassuringly	pos	1.8000
rebeldom	neg	-1.5000
rebelling	neg	-1.1000
rebellions	neg	-1.1000
rebellious	neg	-1.2000
rebelliously	neg	-1.8000
rebelliousness	neg	-1.2000
recession	neg	-1.8000
reckless	neg	-1.7000
recommend	pos	1.5000
redeemed	pos	1.3000
reek	neg	-2.4000
reeked	neg	-2.0000
reeker	neg	-1.7000
reekers	neg	-1.5000
reeking	neg	-2.0000
refuse	neg	-1.2000
refused	neg	-1.2000
refusing	neg	-1.7000
regret	neg	-1.8000
regretful	neg	-1.9000
regretfully	neg	-1.9000
regretfulness	neg	-1.6000
regrets	neg	-1.5000
regrettable	neg	-2.3000
regrettably	neg	-2.0000
regretted	neg	-1.6000
regretter	neg	-1.6000
regretters	neg	-2.0000
regretting	neg	-1.7000
reinvigorate	pos	2.3000
reinvigorated	pos	1.9000
reinvigorates	pos	1.8000
reinvigorating	pos	1.7000
reinvigoration	pos	2.2000
reject	neg	-1.7000
rejected	neg	-2.3000
rejectee	neg	-2.3000
rejectees	neg	-1.8000
rejecter	neg	-1.6000
rejecters	neg	-1.8000
rejecting	neg	-2.0000
rejectingly	neg	-1.7000
rejection	neg	-2.5000
rejections	neg	-2.1000
rejective	neg	-1.8000
rejector	neg	-1.8000
rejects	neg	-2.2000
rejoice	pos	1.9000
rejoiced	pos	2.0000
rejoices	pos	2.1000
rejoicing	pos	2.8000
relax	pos	1.9000
relaxation	pos	2.4000
relaxed	pos	2.2000
relaxedly	pos	1.5000
relaxedness	pos	2.0000
relaxer	pos	1.6000
relaxers	pos	1.4000
relaxes	pos	1.5000
relaxin	pos	1.7000
relaxing	pos	2.2000
relaxins	pos	1.2000
relief	pos	2.1000
reliefs	pos	1.3000
relievable	pos	1.1000
relieve	pos	1.5000
relieved	pos	1.6000
relievedly	pos	1.4000
reliever	pos	1.5000
relieves	pos	1.5000
relieving	pos	1.5000
relievo	pos	1.3000
relishing	pos	1.6000
reluctance	neg	-1.4000
reluctancy	neg	-1.6000
remarkable	pos	2.6000
remorse	neg	-1.1000
remorseless	neg	-2.3000
remorselessly	neg	-2.0000
remorselessness	neg	-2.8000
repress	neg	-1.4000
repressed	neg	-1.3000
represses	neg	-1.3000
repressible	neg	-1.5000
repressing	neg	-1.8000
repression	neg	-1.6000
repressions	neg	-1.7000
repressive	neg	-1.4000
repressively	neg	-1.7000
repressor	neg	-1.4000
repressors	neg	-2.2000
repulse	neg	-2.8000
repulsed	neg	-2.2000
rescue	pos	2.3000
rescued	pos	1.8000
rescues	pos	1.3000
resented	neg	-1.6000
resentful	neg	-2.1000
resentfully	neg	-1.4000
resentfulness	neg	-2.0000
resenting	neg	-1.2000
resentment	neg	-1.9000
resentments	neg	-1.9000
resents	neg	-1.2000
resign	neg	-1.4000
resignation	neg	-1.2000
resignations	neg	-1.2000
resigner	neg	-1.2000
resigns	neg	-1.3000
resolute	pos	1.1000
resolve	pos	1.6000
resolvers	pos	1.4000
resolving	pos	1.6000
respect	pos	2.1000
respectabilities	pos	1.8000
respectability	pos	2.4000
respectable	pos	1.9000
respectableness	pos	1.2000
respectably	pos	1.7000
respected	pos	2.1000
respecter	pos	2.1000
respecters	pos	1.6000
respectful	pos	2.0000
respectfully	pos	1.7000
respectfulness	pos	1.9000
respectfulnesses	pos	1.3000
respecting	pos	2.2000
respective	pos	1.8000
respectively	pos	1.4000
respectiveness	pos	1.1000
respects	pos	1.3000
responsible	pos	1.3000
responsive	pos	1.5000
restful	pos	1.5000
restless	neg	-1.1000
restlessly	neg	-1.4000
restlessness	neg	-1.2000
restore	pos	1.2000
restored	pos	1.4000
restores	pos	1.2000
restoring	pos	1.2000
restrict	neg	-1.6000
restricted	neg	-1.6000
restricting	neg	-1.6000
restriction	neg	-1.1000
restricts	neg	-1.3000
retard	neg	-2.4000
retarded	neg	-2.7000
revenge	neg	-2.4000
revengeful	neg	-2.4000
revengefully	neg	-1.4000
revengefulness	neg	-2.2000
revenger	neg	-2.1000
revengers	neg	-2.0000
revenges	neg	-1.9000
revered	pos	2.3000
revive	pos	1.4000
revives	pos	1.6000
reward	pos	2.7000
rewardable	pos	2.0000
rewarded	pos	2.2000
rewarder	pos	1.6000
rewarders	pos	1.9000
rewarding	pos	2.4000
rewardingly	pos	2.4000
rewards	pos	2.1000
rich	pos	2.6000
richened	pos	1.9000
richer	pos	2.4000
riches	pos	2.4000
richest	pos	2.4000
richly	pos	1.9000
richness	pos	2.2000
richnesses	pos	2.1000
ridicule	neg	-2.0000
ridiculed	neg	-1.5000
ridiculer	neg	-1.6000
ridiculers	neg	-1.6000
ridicules	neg	-1.8000
ridiculing	neg	-1.8000
ridiculous	neg	-1.5000
ridiculously	neg	-1.4000
ridiculousness	neg	-1.1000
ridiculousnesses	neg	-1.6000
rigged	neg	-1.5000
rigidification	neg	-1.1000
rigorous	neg	-1.1000
riot	neg	-2.6000
riots	neg	-2.3000
risk	neg	-1.1000
riskier	neg	-1.4000
riskiest	neg	-1.5000
riskiness	neg	-1.3000
riskinesses	neg	-1.6000
risking	neg	-1.3000
riskless	pos	1.3000
risks	neg	-1.1000
rob	neg	-2.6000
robber	neg	-2.6000
robing	neg	-1.5000
robs	neg	-2.0000
robust	pos	1.4000
roflcopter	pos	2.1000
romance	pos	2.6000
romanced	pos	2.2000
romancer	pos	1.3000
romancers	pos	1.7000
romances	pos	1.3000
romancing	pos	2.0000
romantic	pos	1.7000
romantically	pos	1.8000
romanticise	pos	1.7000
romanticised	pos	1.7000
romanticises	pos	1.3000
romanticising	pos	2.7000
romanticism	pos	2.2000
romanticisms	pos	2.1000
romanticist	pos	1.9000
romanticists	pos	1.3000
romanticization	pos	1.5000
romanticizations	pos	2.0000
romanticize	pos	1.8000
romanticizes	pos	1.8000
romanticizing	pos	1.2000
romantics	pos	1.9000
rotten	neg	-2.3000
rude	neg	-2.0000
rudely	neg	-2.2000
rudeness	neg	-1.5000
ruder	neg	-2.1000
rudesby	neg	-2.0000
rudest	neg	-2.5000
ruin	neg	-2.8000
ruinable	neg	-1.6000
ruinate	neg	-2.8000
ruinated	neg	-1.5000
ruinates	neg	-1.5000
ruinating	neg	-1.5000
ruination	neg	-2.7000
ruinations	neg	-1.6000
ruined	neg	-2.1000
ruiner	neg	-2.0000
ruing	neg	-1.6000
ruinous	neg	-2.7000
ruinously	neg	-2.6000
ruins	neg	-1.9000
sabotage	neg	-2.4000
sad	neg	-2.1000
sadden	neg	-2.6000
saddened	neg	-2.4000
saddening	neg	-2.2000
saddens	neg	-1.9000
sadder	neg	-2.4000
saddest	neg	-3.0000
sadly	neg	-1.8000
sadness	neg	-1.9000
safe	pos	1.9000
safeguard	pos	1.6000
safeguarded	pos	1.5000
safeguarding	pos	1.1000
safeguards	pos	1.4000
safekeeping	pos	1.4000
safelight	pos	1.1000
safely	pos	2.2000
safeness	pos	1.5000
safer	pos	1.8000
safest	pos	1.7000
safeties	pos	1.5000
safety	pos	1.8000
salient	pos	1.1000
sarcastically	neg	-1.1000
satisfaction	pos	1.9000
satisfactions	pos	2.1000
satisfactorily	pos	1.6000
satisfactoriness	pos	1.5000
satisfactory	pos	1.5000
satisfiable	pos	1.9000
satisfied	pos	1.8000
satisfies	pos	1.8000
satisfy	pos	2.0000
satisfying	pos	2.0000
satisfyingly	pos	1.9000
savage	neg	-2.0000
savaged	neg	-2.0000
savagely	neg	-2.2000
savageness	neg	-2.6000
savageries	neg	-1.9000
savagery	neg	-2.5000
savages	neg	-2.4000
save	pos	2.2000
saved	pos	1.8000
scam	neg	-2.7000
scams	neg	-2.8000
scandal	neg	-1.9000
scandalous	neg	-2.4000
scandals	neg	-2.2000
scapegoat	neg	-1.7000
scapegoats	neg	-1.4000
scare	neg	-2.2000
scared	neg	-1.9000
scaremonger	neg	-2.1000
scaremongers	neg	-2.0000
scarer	neg	-1.7000
scarers	neg	-1.3000
scares	neg	-1.4000
scarey	neg	-1.7000
scaring	neg	-1.9000
scary	neg	-2.2000
sceptical	neg	-1.2000
scold	neg	-1.7000
scorn	neg	-1.7000
scornful	neg	-1.8000
scream	neg	-1.7000
screamed	neg	-1.3000
screamers	neg	-1.5000
screaming	neg	-1.6000
screams	neg	-1.2000
screwed	neg	-2.2000
screwed up	neg	-1.5000
screwer	neg	-1.2000
screwiest	neg	-2.0000
screwup	neg	-1.7000
screwy	neg	-1.4000
scrumptious	pos	2.1000
scrumptiously	pos	1.5000
scumbag	neg	-3.2000
secure	pos	1.4000
secured	pos	1.7000
securely	pos	1.4000
securement	pos	1.1000
secureness	pos	1.4000
securer	pos	1.5000
secures	pos	1.3000
securest	pos	2.6000
securing	pos	1.3000
securities	pos	1.2000
securitized	pos	1.4000
securitizes	pos	1.6000
security	pos	1.4000
sedition	neg	-1.8000
seditious	neg	-1.7000
seduced	neg	-1.5000
self-confident	pos	2.5000
selfish	neg	-2.1000
selfishly	neg	-1.4000
selfishness	neg	-1.7000
selfishnesses	neg	-2.0000
sentimental	pos	1.3000
sentimentalise	pos	1.2000
sentimentality	pos	1.2000
sentimentalization	pos	1.2000
sentimentalized	pos	1.1000
sentimentalizes	pos	1.1000
sentimentally	pos	1.9000
serene	pos	2.0000
severe	neg	-1.6000
severed	neg	-1.5000
severely	neg	-2.0000
severer	neg	-1.6000
severest	neg	-1.5000
sexy	pos	2.4000
shakedown	neg	-1.2000
shakedowns	neg	-1.4000
shakeout	neg	-1.3000
shakiest	neg	-1.2000
shame	neg	-2.1000
shamed	neg	-2.6000
shamefaced	neg	-2.3000
shamefacedly	neg	-1.9000
shamefacedness	neg	-2.0000
shameful	neg	-2.2000
shamefully	neg	-1.9000
shamefulness	neg	-2.4000
shamefulnesses	neg	-2.3000
shameless	neg	-1.4000
shamelessly	neg	-1.4000
shamelessness	neg	-1.4000
shamelessnesses	neg	-2.0000
shames	neg	-1.7000
share	pos	1.2000
shared	pos	1.4000
shares	pos	1.2000
sharing	pos	1.8000
shattered	neg	-2.1000
shit	neg	-2.6000
shitakes	neg	-1.1000
shithead	neg	-3.1000
shitheads	neg	-2.6000
shits	neg	-2.1000
shitted	neg	-1.7000
shittier	neg	-2.1000
shittiest	neg	-3.4000
shitting	neg	-1.8000
shitty	neg	-2.6000
shock	neg	-1.6000
shocked	neg	-1.3000
shockers	neg	-1.1000
shocking	neg	-1.7000
shockproof	pos	1.3000
shocks	neg	-1.6000
shoot	neg	-1.4000
short-sighted	neg	-1.2000
short-sightedness	neg	-1.1000
shylock	neg	-2.1000
shylocking	neg	-1.5000
shylocks	neg	-1.4000
shyness	neg	-1.3000
shynesses	neg	-1.2000
shyster	neg	-1.6000
sick	neg	-2.3000
sicken	neg	-1.9000
sickened	neg	-2.5000
sickener	neg	-2.2000
sickeners	neg	-2.2000
sickening	neg	-2.4000
sickeningly	neg	-2.1000
sickens	neg	-2.0000
significance	pos	1.1000
sillinesses	neg	-1.2000
sin	neg	-2.6000
sincere	pos	1.7000
sincerely	pos	2.1000
sincereness	pos	1.8000
sincerer	pos	2.0000
sincerest	pos	2.0000
sincerities	pos	1.5000
sinful	neg	-2.6000
singleminded	pos	1.2000
sinister	neg	-2.9000
sins	neg	-2.0000
skeptical	neg	-1.3000
skeptically	neg	-1.2000
skepticisms	neg	-1.2000
slam	neg	-1.6000
slash	neg	-1.1000
slashing	neg	-1.1000
slavery	neg	-3.8000
sleeplessness	neg	-1.6000
sluggish	neg	-1.7000
slut	neg	-2.8000
sluts	neg	-2.7000
sluttier	neg	-2.7000
sluttiest	neg	-3.1000
sluttish	neg	-2.2000
sluttishly	neg	-2.1000
sluttishness	neg	-2.5000
sluttishnesses	neg	-2.0000
slutty	neg	-2.3000
smart	pos	1.7000
smartass	neg	-2.1000
smartasses	neg	-1.7000
smarten	pos	1.9000
smartened	pos	1.5000
smartening	pos	1.7000
smartens	pos	1.5000
smarter	pos	2.0000
smartest	pos	3.0000
smartie	pos	1.3000
smarties	pos	1.7000
smartly	pos	1.5000
smartness	pos	2.0000
smartnesses	pos	1.5000
smarts	pos	1.6000
smarty	pos	1.1000
smear	neg	-1.5000
smile	pos	1.5000
smiled	pos	2.5000
smileless	neg	-1.4000
smiler	pos	1.7000
smiles	pos	2.1000
smiley	pos	1.7000
smileys	pos	1.5000
smiling	pos	2.0000
smilingly	pos	2.3000
smog	neg	-1.2000
smother	neg	-1.8000
smothering	neg	-1.4000
smothers	neg	-1.9000
smothery	neg	-1.1000
smuggest	neg	-1.5000
smuggle	neg	-1.6000
smuggled	neg	-1.5000
smuggler	neg	-2.1000
smugglers	neg	-1.4000
smuggles	neg	-1.7000
smuggling	neg	-2.1000
smugness	neg	-1.4000
smugnesses	neg	-1.7000
snob	neg	-2.0000
snobbery	neg	-2.0000
snobbily	neg	-1.6000
snobbishly	neg	-1.2000
snobbishness	neg	-1.1000
snobbishnesses	neg	-1.7000
snobby	neg	-1.7000
snobs	neg	-1.4000
snub	neg	-1.8000
snubbed	neg	-2.0000
snubs	neg	-2.1000
sobbed	neg	-1.9000
sobbing	neg	-1.6000
sobs	neg	-2.5000
sociabilities	pos	1.2000
sociability	pos	1.1000
sociable	pos	1.9000
sociableness	pos	1.5000
sociably	pos	1.6000
sok	pos	1.3000
solemnity	neg	-1.1000
solidarity	pos	1.2000
solution	pos	1.3000
solved	pos	1.1000
solves	pos	1.1000
solving	pos	1.4000
somber	neg	-1.8000
son-of-a-bitch	neg	-2.7000
soothe	pos	1.5000
soothing	pos	1.3000
sophisticated	pos	2.6000
sore	neg	-1.5000
sorrow	neg	-2.4000
sorrowed	neg	-2.4000
sorrower	neg	-2.3000
sorrowful	neg	-2.2000
sorrowfully	neg	-2.3000
sorrowfulness	neg	-2.5000
sorrowing	neg	-1.7000
sorrows	neg	-1.6000
soulmate	pos	2.9000
spam	neg	-1.5000
spammer	neg	-2.2000
spammers	neg	-1.6000
spamming	neg	-2.1000
sparkle	pos	1.8000
sparkles	pos	1.3000
sparkling	pos	1.2000
special	pos	1.7000
spirited	pos	1.3000
spiritless	neg	-1.3000
spite	neg	-2.4000
spited	neg	-2.4000
spiteful	neg	-1.9000
spitefully	neg	-2.3000
spitefulness	neg	-1.5000
spitefulnesses	neg	-2.3000
spites	neg	-1.4000
splendent	pos	2.7000
splendid	pos	2.8000
splendidly	pos	2.1000
splendidness	pos	2.3000
splendiferous	pos	2.6000
splendiferously	pos	1.9000
splendiferousness	pos	1.7000
splendor	pos	3.0000
splendorous	pos	2.2000
splendors	pos	2.0000
splendour	pos	2.2000
splendours	pos	2.2000
splendrous	pos	2.2000
sprightly	pos	2.0000
stab	neg	-2.8000
stabbed	neg	-1.9000
stable	pos	1.2000
stabs	neg	-1.9000
stamina	pos	1.2000
stammerer	neg	-1.1000
stampede	neg	-1.8000
stank	neg	-1.9000
startle	neg	-1.3000
starve	neg	-1.9000
starved	neg	-2.6000
starves	neg	-2.3000
starving	neg	-1.8000
steal	neg	-2.2000
stealable	neg	-1.7000
stealer	neg	-1.7000
stealers	neg	-2.2000
stealing	neg	-2.7000
stealings	neg	-1.9000
steals	neg	-2.3000
stench	neg	-2.3000
stenches	neg	-1.5000
stenchful	neg	-2.4000
stenchy	neg	-2.3000
stereotype	neg	-1.3000
stereotyped	neg	-1.2000
stifled	neg	-1.4000
stimulating	pos	1.9000
stingy	neg	-1.6000
stink	neg	-1.7000
stinkard	neg	-2.3000
stinker	neg	-1.5000
stinkers	neg	-1.2000
stinkier	neg	-1.5000
stinkiest	neg	-2.1000
stinking	neg	-2.4000
stinkingly	neg	-1.3000
stinko	neg	-1.5000
stinkpot	neg	-2.5000
stinky	neg	-1.5000
stolen	neg	-2.2000
stop	neg	-1.2000
strained	neg	-1.7000
straining	neg	-1.3000
strains	neg	-1.2000
strangely	neg	-1.2000
strangled	neg	-2.5000
strength	pos	2.2000
strengthen	pos	1.3000
strengthened	pos	1.8000
strengthener	pos	1.8000
strengtheners	pos	1.4000
strengthening	pos	2.2000
strengthens	pos	2.0000
strengths	pos	1.7000
stress	neg	-1.8000
stressed	neg	-1.4000
stresses	neg	-2.0000
stressful	neg	-2.3000
stressfully	neg	-2.6000
stressing	neg	-1.5000
stressless	pos	1.6000
stresslessness	pos	1.6000
stressor	neg	-1.8000
stressors	neg	-2.1000
stricken	neg	-2.3000
strikes	neg	-1.5000
strong	pos	2.3000
stronger	pos	1.6000
strongest	pos	1.9000
strongish	pos	1.7000
strongly	pos	1.1000
struggle	neg	-1.3000
struggled	neg	-1.4000
struggler	neg	-1.1000
strugglers	neg	-1.4000
struggles	neg	-1.5000
struggling	neg	-1.8000
stubborn	neg	-1.7000
stubborner	neg	-1.5000
stubbornly	neg	-1.4000
stubbornness	neg	-1.1000
stubbornnesses	neg	-1.5000
stunk	neg	-1.6000
stunning	pos	1.6000
stupid	neg	-2.4000
stupider	neg	-2.5000
stupidest	neg	-2.4000
stupidities	neg	-2.0000
stupidity	neg	-1.9000
stupidly	neg	-2.0000
stupidness	neg	-1.7000
stupidnesses	neg	-2.6000
stupids	neg	-2.3000
stutterers	neg	-1.1000
stuttering	neg	-1.3000
suave	pos	2.0000
submissive	neg	-1.3000
succeed	pos	2.2000
succeeded	pos	1.8000
succeeder	pos	1.2000
succeeders	pos	1.3000
succeeding	pos	2.2000
succeeds	pos	2.2000
success	pos	2.7000
successes	pos	2.6000
successful	pos	2.8000
successfully	pos	2.2000
successfulness	pos	2.7000
successionally	pos	1.1000
successive	pos	1.1000
successors	pos	1.1000
suck	neg	-1.9000
sucked	neg	-2.0000
sucker	neg	-2.4000
suckered	neg	-2.0000
suckering	neg	-2.1000
suckers	neg	-2.3000
sucks	neg	-1.5000
sucky	neg	-1.9000
suffer	neg	-2.5000
suffered	neg	-2.2000
sufferer	neg	-2.0000
sufferers	neg	-2.4000
suffering	neg	-2.1000
suffers	neg	-2.1000
suicidal	neg	-3.5000
suicide	neg	-3.5000
suing	neg	-1.1000
sulking	neg	-1.5000
sullen	neg	-1.7000
sunnier	pos	2.3000
sunniest	pos	2.4000
sunny	pos	1.8000
sunshine	pos	2.2000
sunshiny	pos	1.9000
super	pos	2.9000
superb	pos	3.1000
superior	pos	2.5000
superiority	pos	1.4000
superiorly	pos	2.2000
support	pos	1.7000
supported	pos	1.3000
supporter	pos	1.1000
supporters	pos	1.9000
supporting	pos	1.9000
supportive	pos	1.2000
supportiveness	pos	1.5000
supports	pos	1.5000
supreme	pos	2.6000
supremely	pos	2.7000
supremeness	pos	2.3000
supremer	pos	2.3000
supremest	pos	2.2000
supremo	pos	1.9000
supremos	pos	1.3000
sure	pos	1.3000
surefooted	pos	1.9000
surefootedly	pos	1.6000
surefootedness	pos	1.5000
surely	pos	1.9000
sureness	pos	2.0000
surer	pos	1.2000
surest	pos	1.3000
sureties	pos	1.3000
surprisal	pos	1.5000
surprise	pos	1.1000
surprising	pos	1.1000
surprisingly	pos	1.2000
survived	pos	2.3000
surviving	pos	1.2000
survivor	pos	1.5000
suspect	neg	-1.2000
suspects	neg	-1.4000
suspend	neg	-1.3000
suspended	neg	-2.1000
suspicion	neg	-1.6000
suspicions	neg	-1.5000
suspicious	neg	-1.5000
suspiciously	neg	-1.7000
suspiciousness	neg	-1.2000
sux	neg	-1.5000
sweet	pos	2.0000
sweet<3	pos	3.0000
sweetheart	pos	3.3000
sweethearts	pos	2.8000
sweetie	pos	2.2000
sweeties	pos	2.1000
sweetly	pos	2.1000
sweetness	pos	2.2000
sweets	pos	2.2000
swiftly	pos	1.2000
swindle	neg	-2.4000
swindles	neg	-1.5000
swindling	neg	-2.0000
sympathetic	pos	2.3000
sympathy	pos	1.5000
talent	pos	1.8000
talented	pos	2.3000
talentless	neg	-1.6000
talents	pos	2.0000
tantrum	neg	-1.8000
tantrums	neg	-1.5000
tard	neg	-2.5000
tease	neg	-1.3000
teased	neg	-1.2000
teaselers	neg	-1.2000
teases	neg	-1.2000
temper	neg	-1.8000
tempers	neg	-1.3000
tenderers	pos	1.2000
tenderest	pos	1.4000
tenderhearted	pos	1.5000
tenderheartedly	pos	2.7000
tenderheartednesses	pos	2.8000
tenderly	pos	1.8000
tenderness	pos	1.8000
tense	neg	-1.4000
tensely	neg	-1.2000
tenseness	neg	-1.5000
tenser	neg	-1.5000
tensest	neg	-1.2000
tension	neg	-1.3000
tensioner	neg	-1.6000
tensioning	neg	-1.4000
tensions	neg	-1.7000
terrible	neg	-2.1000
terribleness	neg	-1.9000
terriblenesses	neg	-2.6000
terribly	neg	-2.6000
terrific	pos	2.1000
terrifically	pos	1.7000
terrified	neg	-3.0000
terrifies	neg	-2.6000
terrify	neg	-2.3000
terrifying	neg	-2.7000
terror	neg	-2.4000
terrorise	neg	-3.1000
terrorised	neg	-3.3000
terrorises	neg	-3.3000
terrorising	neg	-3.0000
terrorism	neg	-3.6000
terrorisms	neg	-3.2000
terrorist	neg	-3.7000
terroristic	neg	-3.3000
terrorists	neg	-3.1000
terrorization	neg	-2.7000
terrorize	neg	-3.3000
terrorized	neg	-3.1000
terrorizes	neg	-3.1000
terrorizing	neg	-3.0000
terrors	neg	-2.6000
thank	pos	1.5000
thanked	pos	1.9000
thankful	pos	2.7000
thankfuller	pos	1.9000
thankfullest	pos	2.0000
thankfully	pos	1.8000
thankfulness	pos	2.1000
thanks	pos	1.9000
thief	neg	-2.4000
thieve	neg	-2.2000
thieved	neg	-1.4000
thieveries	neg	-2.1000
thievery	neg	-2.0000
thieves	neg	-2.3000
thorny	neg	-1.1000
thoughtful	pos	1.6000
thoughtfully	pos	1.7000
thoughtfulness	pos	1.9000
thoughtless	neg	-2.0000
threat	neg	-2.4000
threaten	neg	-1.6000
threatened	neg	-2.0000
threatener	neg	-1.4000
threateners	neg	-1.8000
threatening	neg	-2.4000
threateningly	neg	-2.2000
threatens	neg	-1.6000
threating	neg	-2.0000
threats	neg	-1.8000
thrill	pos	1.5000
thrilled	pos	1.9000
thrilling	pos	2.1000
thrillingly	pos	2.0000
thrills	pos	1.5000
ticked	neg	-1.8000
timidity	neg	-1.3000
tired	neg	-1.9000
tolerance	pos	1.2000
tolerant	pos	1.1000
toothless	neg	-1.4000
tops	pos	2.3000
torture	neg	-2.9000
tortured	neg	-2.6000
torturer	neg	-2.3000
torturers	neg	-3.5000
tortures	neg	-2.5000
torturing	neg	-3.0000
torturous	neg	-2.7000
torturously	neg	-2.2000
totalitarian	neg	-2.1000
totalitarianism	neg	-2.7000
toughly	neg	-1.1000
tragediennes	neg	-1.4000
tragedies	neg	-1.9000
tragedy	neg	-3.4000
tragic	neg	-2.0000
tragical	neg	-2.4000
tragically	neg	-2.7000
tragics	neg	-2.2000
tranquiler	pos	1.9000
tranquilest	pos	1.6000
tranquilities	pos	1.5000
tranquility	pos	1.8000
tranquillity	pos	1.8000
tranquilly	pos	1.2000
tranquilness	pos	1.5000
trap	neg	-1.3000
trapped	neg	-2.4000
trauma	neg	-1.8000
traumas	neg	-2.2000
traumata	neg	-1.7000
traumatic	neg	-2.7000
traumatically	neg	-2.8000
traumatise	neg	-2.8000
traumatised	neg	-2.4000
traumatises	neg	-2.2000
traumatising	neg	-1.9000
traumatism	neg	-2.4000
traumatization	neg	-3.0000
traumatizations	neg	-2.2000
traumatize	neg	-2.4000
traumatized	neg	-1.7000
traumatizes	neg	-1.4000
traumatizing	neg	-2.3000
travesty	neg	-2.7000
treason	neg	-1.9000
treasonous	neg	-2.7000
treasurable	pos	2.5000
treasure	pos	1.2000
treasured	pos	2.6000
treasurerships	pos	1.2000
treasures	pos	1.8000
treasuring	pos	2.1000
treat	pos	1.7000
tremble	neg	-1.1000
trembled	neg	-1.1000
trembling	neg	-1.5000
trembly	neg	-1.2000
trickeries	neg	-1.2000
trickers	neg	-1.4000
trickery	neg	-1.1000
trickiest	neg	-1.2000
trickiness	neg	-1.2000
tricksters	neg	-1.3000
triumph	pos	2.1000
triumphal	pos	2.0000
triumphalisms	pos	1.9000
triumphant	pos	2.4000
triumphantly	pos	2.3000
triumphed	pos	2.2000
triumphing	pos	2.3000
triumphs	pos	2.0000
trivialises	neg	-1.1000
trivialising	neg	-1.4000
trivialize	neg	-1.1000
trouble	neg	-1.7000
troubled	neg	-2.0000
troublemaker	neg	-2.0000
troublemakers	neg	-2.2000
troublemaking	neg	-1.8000
troubler	neg	-1.4000
troublers	neg	-1.9000
troubles	neg	-2.0000
troublesome	neg	-2.3000
troublesomely	neg	-1.8000
troublesomeness	neg	-1.9000
troubling	neg	-2.5000
troublous	neg	-2.1000
troublously	neg	-2.1000
trueness	pos	2.1000
truer	pos	1.5000
truest	pos	1.9000
truly	pos	1.9000
trust	pos	2.3000
trustability	pos	2.1000
trustable	pos	2.3000
trusted	pos	2.1000
truster	pos	1.9000
trustful	pos	2.1000
trustfully	pos	1.5000
trustfulness	pos	2.1000
trustier	pos	1.3000
trustiest	pos	2.2000
trustily	pos	1.6000
trustiness	pos	1.6000
trusting	pos	1.7000
trustingly	pos	1.6000
trustingness	pos	1.6000
trustless	neg	-2.3000
trustors	pos	1.2000
trusts	pos	2.1000
trustworthily	pos	2.3000
trustworthiness	pos	1.8000
trustworthy	pos	2.6000
trusty	pos	2.2000
truth	pos	1.3000
truthful	pos	2.0000
truthfully	pos	1.9000
truthfulness	pos	1.7000
truths	pos	1.8000
tumor	neg	-1.6000
turmoil	neg	-1.5000
twat	neg	-3.4000
ugh	neg	-1.8000
uglier	neg	-2.2000
uglies	neg	-2.0000
ugliest	neg	-2.8000
uglification	neg	-2.2000
uglified	neg	-1.5000
uglifies	neg	-1.8000
uglify	neg	-2.1000
uglifying	neg	-2.2000
uglily	neg	-2.1000
ugliness	neg	-2.7000
uglinesses	neg	-2.5000
ugly	neg	-2.3000
unacceptable	neg	-2.0000
unappreciated	neg	-1.7000
unapproved	neg	-1.4000
unattractive	neg	-1.9000
uncertain	neg	-1.2000
uncertainly	neg	-1.4000
uncertainness	neg	-1.3000
uncertainties	neg	-1.4000
uncertainty	neg	-1.4000
uncomfortable	neg	-1.6000
uncomfortably	neg	-1.7000
uncontrollability	neg	-1.7000
uncontrollable	neg	-1.5000
uncontrollably	neg	-1.5000
unconvinced	neg	-1.6000
underestimate	neg	-1.2000
underestimated	neg	-1.1000
underestimates	neg	-1.1000
undermine	neg	-1.2000
undermined	neg	-1.5000
undermines	neg	-1.4000
undermining	neg	-1.5000
undeserving	neg	-1.9000
undesirable	neg	-1.9000
unease	neg	-1.7000
uneasier	neg	-1.4000
uneasiest	neg	-2.1000
uneasily	neg	-1.4000
uneasiness	neg	-1.6000
uneasinesses	neg	-1.8000
uneasy	neg	-1.6000
unemployment	neg	-1.9000
unequal	neg	-1.4000
unethical	neg	-2.3000
unfair	neg	-2.1000
unfocused	neg	-1.7000
unfortunate	neg	-2.0000
unfortunately	neg	-1.4000
unfortunates	neg	-1.9000
unfriendly	neg	-1.5000
unfulfilled	neg	-1.8000
ungrateful	neg	-2.0000
ungratefully	neg	-1.8000
ungratefulness	neg	-1.6000
unhappier	neg	-2.4000
unhappiest	neg	-2.5000
unhappily	neg	-1.9000
unhappiness	neg	-2.4000
unhappinesses	neg	-2.2000
unhappy	neg	-1.8000
unhealthy	neg	-2.4000
unified	pos	1.6000
unimportant	neg	-1.3000
unimpressed	neg	-1.4000
unimpressive	neg	-1.4000
unintelligent	neg	-2.0000
uninvolved	neg	-2.2000
uninvolving	neg	-2.0000
united	pos	1.8000
unjust	neg	-2.3000
unkind	neg	-1.6000
unlovable	neg	-2.7000
unloved	neg	-1.9000
unlovelier	neg	-1.9000
unloveliest	neg	-1.9000
unloveliness	neg	-2.0000
unlovely	neg	-2.1000
unloving	neg	-2.3000
unmotivated	neg	-1.4000
unpleasant	neg	-2.1000
unprofessional	neg	-2.3000
unprotected	neg	-1.5000
unresearched	neg	-1.1000
unsatisfied	neg	-1.7000
unsavory	neg	-1.9000
unsecured	neg	-1.6000
unsettled	neg	-1.3000
unsophisticated	neg	-1.2000
unstable	neg	-1.5000
unsuccessful	neg	-1.5000
unsuccessfully	neg	-1.7000
unsupported	neg	-1.7000
unsurely	neg	-1.3000
untarnished	pos	1.6000
unwelcome	neg	-1.7000
unworthy	neg	-2.0000
upset	neg	-1.6000
upsets	neg	-1.5000
upsetter	neg	-1.9000
upsetters	neg	-2.0000
upsetting	neg	-2.1000
uptight	neg	-1.6000
uptightness	neg	-1.2000
useful	pos	1.9000
usefully	pos	1.8000
usefulness	pos	1.2000
useless	neg	-1.8000
uselessly	neg	-1.5000
uselessness	neg	-1.6000
v.v	neg	-2.9000
vain	neg	-1.8000
validate	pos	1.5000
validates	pos	1.4000
validating	pos	1.4000
valuable	pos	2.1000
valuableness	pos	1.7000
valuables	pos	2.1000
valuably	pos	2.3000
value	pos	1.4000
valued	pos	1.9000
values	pos	1.7000
valuing	pos	1.4000
vexation	neg	-1.9000
vexing	neg	-2.0000
vibrant	pos	2.4000
vicious	neg	-1.5000
viciously	neg	-1.3000
viciousness	neg	-2.4000
victim	neg	-1.1000
victimhood	neg	-2.0000
victimise	neg	-1.1000
victimised	neg	-1.5000
victimises	neg	-1.2000
victimising	neg	-2.5000
victimization	neg	-2.3000
victimizations	neg	-1.5000
victimize	neg	-2.5000
victimized	neg	-1.8000
victimizer	neg	-1.8000
victimizers	neg	-1.6000
victimizes	neg	-1.5000
victimizing	neg	-2.6000
victims	neg	-1.3000
vigor	pos	1.1000
vigoroso	pos	1.5000
vile	neg	-3.1000
villain	neg	-2.6000
villainess	neg	-2.9000
villainesses	neg	-2.0000
villainies	neg	-2.3000
villainous	neg	-2.0000
villainously	neg	-2.9000
villainousness	neg	-2.7000
villains	neg	-3.4000
villainy	neg	-2.6000
vindicated	pos	1.8000
vindicates	pos	1.6000
vindicating	neg	-1.1000
violate	neg	-2.2000
violated	neg	-2.4000
violater	neg	-2.6000
violaters	neg	-2.4000
violates	neg	-2.3000
violating	neg	-2.5000
violation	neg	-2.2000
violations	neg	-2.4000
violative	neg	-2.4000
violator	neg	-2.4000
violators	neg	-1.9000
violence	neg	-3.1000
violent	neg	-2.9000
violently	neg	-2.8000
virtue	pos	1.8000
virtueless	neg	-1.4000
virtues	pos	1.5000
virtuosa	pos	1.7000
virtuosas	pos	1.8000
virtuosic	pos	2.2000
virtuosity	pos	2.1000
virtuoso	pos	2.0000
virtuosos	pos	1.8000
virtuous	pos	2.4000
virtuously	pos	1.8000
virtuousness	pos	2.0000
virulent	neg	-2.7000
visionary	pos	2.4000
visioning	pos	1.1000
vital	pos	1.2000
vitalise	pos	1.1000
vitalises	pos	1.1000
vitalising	pos	2.1000
vitalities	pos	1.2000
vitality	pos	1.3000
vitalization	pos	1.6000
vitalize	pos	1.6000
vitalized	pos	1.5000
vitalizes	pos	1.4000
vitalizing	pos	1.3000
vitally	pos	1.1000
vitals	pos	1.1000
vitamin	pos	1.2000
vitriolic	neg	-2.1000
vivacious	pos	1.8000
vulnerableness	neg	-1.1000
vulnerably	neg	-1.2000
vulture	neg	-2.0000
vultures	neg	-1.3000
w00t	pos	2.2000
walkout	neg	-1.3000
wanker	neg	-2.5000
war	neg	-2.9000
warfare	neg	-1.2000
warfares	neg	-1.8000
warmed	pos	1.1000
warmer	pos	1.2000
warmest	pos	1.7000
warmhearted	pos	1.8000
warmheartedness	pos	2.7000
warmish	pos	1.4000
warmly	pos	1.7000
warmness	pos	1.5000
warmonger	neg	-2.9000
warmongering	neg	-2.5000
warmongers	neg	-2.8000
warms	pos	1.1000
warmth	pos	2.0000
warned	neg	-1.1000
warning	neg	-1.4000
warnings	neg	-1.2000
warred	neg	-2.4000
warring	neg	-1.9000
wars	neg	-2.6000
waste	neg	-1.8000
wasted	neg	-2.2000
wasting	neg	-1.7000
weak	neg	-1.9000
weaken	neg	-1.8000
weakened	neg	-1.3000
weakener	neg	-1.6000
weakeners	neg	-1.3000
weakening	neg	-1.3000
weakens	neg	-1.3000
weaker	neg	-1.9000
weakest	neg	-2.3000
weakhearted	neg	-1.6000
weakish	neg	-1.2000
weaklier	neg	-1.5000
weakliest	neg	-2.1000
weakling	neg	-1.3000
weaklings	neg	-1.4000
weakly	neg	-1.8000
weakness	neg	-1.8000
weaknesses	neg	-1.5000
weakside	neg	-1.1000
wealth	pos	2.2000
wealthier	pos	2.2000
wealthiest	pos	2.2000
wealthily	pos	2.0000
wealthiness	pos	2.4000
wealthy	pos	1.5000
weapon	neg	-1.2000
weaponed	neg	-1.4000
weapons	neg	-1.9000
weary	neg	-1.1000
weep	neg	-2.7000
weeper	neg	-1.9000
weepers	neg	-1.1000
weepier	neg	-1.8000
weepies	neg	-1.6000
weepiest	neg	-2.4000
weeping	neg	-1.9000
weepings	neg	-1.9000
weeps	neg	-1.4000
weepy	neg	-1.3000
weirdie	neg	-1.3000
weirdly	neg	-1.2000
weirdo	neg	-1.8000
weirdoes	neg	-1.3000
weirdos	neg	-1.1000
welcome	pos	2.0000
welcomed	pos	1.4000
welcomely	pos	1.9000
welcomeness	pos	2.0000
welcomer	pos	1.4000
welcomers	pos	1.9000
welcomes	pos	1.7000
welcoming	pos	1.9000
well	pos	1.1000
wellborn	pos	1.8000
welldoer	pos	2.5000
welldoers	pos	1.6000
welling	pos	1.6000
wellness	pos	1.9000
wellspring	pos	1.5000
wellsprings	pos	1.4000
wept	neg	-2.0000
whine	neg	-1.5000
whiner	neg	-1.2000
whines	neg	-1.8000
whiney	neg	-1.3000
whore	neg	-3.3000
whored	neg	-2.8000
whoredom	neg	-2.1000
whoredoms	neg	-2.4000
whorehouse	neg	-1.1000
whorehouses	neg	-1.9000
whoremaster	neg	-1.9000
whoremasters	neg	-1.5000
whoremonger	neg	-2.6000
whoremongers	neg	-2.0000
whores	neg	-3.0000
whoreson	neg	-2.2000
whoresons	neg	-2.5000
wicked	neg	-2.4000
wickeder	neg	-2.2000
wickedest	neg	-2.9000
wickedly	neg	-2.1000
wickedness	neg	-2.1000
wickednesses	neg	-2.2000
widowed	neg	-2.1000
willingness	pos	1.1000
wimp	neg	-1.4000
wimpiness	neg	-1.2000
wimpish	neg	-1.6000
win	pos	2.8000
winnable	pos	1.8000
winned	pos	1.8000
winner	pos	2.8000
winners	pos	2.1000
winning	pos	2.4000
winningly	pos	2.3000
winnings	pos	2.5000
wins	pos	2.7000
wisdom	pos	2.4000
wise	pos	2.1000
wiseacre	neg	-1.2000
wiseass	neg	-1.8000
wiseasses	neg	-1.5000
wised	pos	1.5000
wiseliest	pos	1.6000
wisely	pos	1.8000
wiseness	pos	1.9000
wisenheimers	neg	-1.4000
wiser	pos	1.2000
wises	pos	1.3000
wisest	pos	2.1000
wisewomen	pos	1.3000
wish	pos	1.7000
witch	neg	-1.5000
woe	neg	-1.8000
woebegone	neg	-2.6000
woebegoneness	neg	-1.1000
woeful	neg	-1.9000
woefully	neg	-1.7000
woefulness	neg	-2.1000
woes	neg	-1.9000
woesome	neg	-1.2000
won	pos	2.7000
wonderful	pos	2.7000
wonderfully	pos	2.9000
wonderfulness	pos	2.9000
woo	pos	2.1000
woohoo	pos	2.3000
woot	pos	1.8000
worn	neg	-1.2000
worried	neg	-1.2000
worriedly	neg	-2.0000
worrier	neg	-1.8000
worriers	neg	-1.7000
worries	neg	-1.8000
worriment	neg	-1.5000
worriments	neg	-1.9000
worrisome	neg	-1.7000
worrisomely	neg	-2.0000
worrisomeness	neg	-1.9000
worrit	neg	-2.1000
worrits	neg	-1.2000
worry	neg	-1.9000
worrying	neg	-1.4000
worrywart	neg	-1.8000
worrywarts	neg	-1.5000
worse	neg	-2.1000
worsen	neg	-2.3000
worsened	neg	-1.9000
worsening	neg	-2.0000
worsens	neg	-2.1000
worser	neg	-2.0000
worship	pos	1.2000
worshiped	pos	2.4000
worshipfully	pos	1.1000
worshipfulness	pos	1.6000
worshipped	pos	2.7000
worshipping	pos	1.6000
worships	pos	1.4000
worst	neg	-3.1000
worthless	neg	-1.9000
worthwhile	pos	1.4000
worthy	pos	1.9000
wow	pos	2.8000
wowed	pos	2.6000
wowing	pos	2.5000
wows	pos	2.0000
wowser	neg	-1.1000
wrathful	neg	-2.7000
wreck	neg	-1.9000
wrong	neg	-2.1000
wronged	neg	-1.9000
x-d	pos	2.6000
x-p	pos	1.7000
xd	pos	2.8000
xp	pos	1.6000
yay	pos	2.4000
yeah	pos	1.2000
yeees	pos	1.7000
yep	pos	1.2000
yes	pos	1.7000
youthful	pos	1.3000
yucky	neg	-1.8000
yummy	pos	2.4000
zealot	neg	-1.9000
{:	pos	1.8000
|-0	neg	-1.2000
|-:>	neg	-1.6000
|-o	neg	-1.2000
|;-)	pos	2.2000
|^:	neg	-1.1000
||-:	neg	-2.3000
}:	neg	-2.1000
}:(	neg	-2.0000
}:-(	neg	-2.1000

In [ ]:
%%writefile /kaggle/working/ga/data/unqover.tsv
word	polarity	score
attractive	pos	1.0000
beautiful	pos	1.0000
bright	pos	1.0000
clean	pos	1.0000
courteous	pos	1.0000
decent	pos	1.0000
easy	pos	1.0000
easygoing	pos	1.0000
fit	pos	1.0000
generous	pos	1.0000
gentle	pos	1.0000
gently	pos	1.0000
great	pos	1.0000
healthy	pos	1.0000
helped	pos	1.0000
humble	pos	1.0000
legal	pos	1.0000
legally	pos	1.0000
loved	pos	1.0000
peace	pos	1.0000
peaceful	pos	1.0000
polite	pos	1.0000
quiet	pos	1.0000
respect	pos	1.0000
respected	pos	1.0000
respectful	pos	1.0000
rich	pos	1.0000
safe	pos	1.0000
sane	pos	1.0000
save	pos	1.0000
trustworthy	pos	1.0000
welcoming	pos	1.0000
wisely	pos	1.0000
aggressive	neg	-1.0000
angry	neg	-1.0000
bad	neg	-1.0000
criminal	neg	-1.0000
criminals	neg	-1.0000
dangerous	neg	-1.0000
destruction	neg	-1.0000
drunk	neg	-1.0000
foolishly	neg	-1.0000
harsh	neg	-1.0000
hateful	neg	-1.0000
hostile	neg	-1.0000
illegal	neg	-1.0000
lazy	neg	-1.0000
poor	neg	-1.0000
refused	neg	-1.0000
rude	neg	-1.0000
scam	neg	-1.0000
snobbish	neg	-1.0000
strange	neg	-1.0000
terrible	neg	-1.0000
terrorist	neg	-1.0000
violence	neg	-1.0000
violent	neg	-1.0000
yelled	neg	-1.0000

## 1. Smoke test: first 2,000 documents of one shard

In [ ]:
%%bash
cd /kaggle/working/ga
python - <<'EOF'
import gzip
from huggingface_hub import hf_hub_download
p = hf_hub_download("allenai/c4", "en.noblocklist/c4-train.00000-of-01024.json.gz", repo_type="dataset")
with gzip.open(p, "rt", encoding="utf-8") as f, gzip.open("smoke_c4-train.00000-of-01024.json.gz", "wt", encoding="utf-8") as g:
    for i, line in enumerate(f):
        if i >= 2000: break
        g.write(line)
EOF
python scripts/scan_corpus.py --files smoke_c4-train.00000-of-01024.json.gz --out_dir results/smoke_corpus
python scripts/analyze_corpus.py --scan results/smoke_corpus --out_dir results/smoke_corpus/analysis --B 20 --no_figures | tail -n 3

## 2. Full scan: 12 shards, 4 in parallel

In [ ]:
%%bash
cd /kaggle/working/ga
python scripts/scan_corpus.py --n 12 --seed 0 --workers 4 --out_dir results/C1_noblocklist_12_shards 2>&1 | tail -n 15

## 3. Sentiment classifier on the sampled paragraphs

In [ ]:
%%bash
cd /kaggle/working/ga
if nvidia-smi > /dev/null 2>&1; then CAP=0; else CAP=150; fi
python scripts/classify_paragraphs.py --scan results/C1_noblocklist_12_shards --max_per_term $CAP --batch_size 32 2>&1 | tail -n 3

## 4. Analysis

In [ ]:
%%bash
cd /kaggle/working/ga
python scripts/analyze_corpus.py --scan results/C1_noblocklist_12_shards --out_dir results/C1_noblocklist_12_shards/analysis --B 200

In [ ]:
import os
from IPython.display import Image, Markdown, display
os.chdir("/kaggle/working/ga/results/C1_noblocklist_12_shards/analysis")
display(Markdown(open("summary.md").read()))
for f in ("fig_exclusion_pmi.png", "fig_exclusion_gaps.png", "fig_sentiment_gaps.png", "fig_sentiment_terms.png"):
    if os.path.exists(f):
        display(Image(f, width=900))

## 5. Download (Output tab). Raw shards are deleted after scanning.

In [ ]:
%%bash
cd /kaggle/working/ga
zip -qr /kaggle/working/ga_corpus.zip results -x 'results/smoke*'
ls -la /kaggle/working/ga_corpus.zip